# GF-HashBA — Generative Flow-guided Hash Bundle Adjustment

Research prototype: **Hash-SDF** + **learnable cameras** + **MV-SAM3D Stage-2 prior** for unobserved regions.

**Just Run all** on a Colab A100 (Python 3.12 runtime). If §1.0b pins torch/numpy it restarts the runtime — then Run all again. Use `RUN_PRESET="smoke"` in §2 for a 10-minute end-to-end check first.

## What changed after the 8/18 blob run

1. **Prior sign bug (main cause of the blob).** The old flow distillation noised the density proxy with data at t=0 but the priors returned velocities for data at t=1, so the target was `(1+t)·z_t − t·target`: it pushed density *away* from the teacher and added noise with std ≈ t(1+t) — the 0.07 ↔ 3.8 `flow=` swings in the 8/18 log. Replaced by a direct, visibility-gated occupancy distillation (BCE on σ(−f/τ)).
2. **Real Stage-2 geometry.** The MV-SAM3D Stage-2 teacher SLAT (multi-view MultiDiffusion, optional LoRA) is decoded with the SLAT mesh decoder, mapped with the same Sim3 as Stage-1, solidified, and distilled on blind spots. The old per-iteration `reverse_fn` call re-noised the teacher's own SLAT and never saw the Hash-SDF, so caching its output is exact (and frees the GPU).
3. **Occlusion-aware V.** z-buffers rendered from the current Hash-SDF (refreshed every `VIS_REFRESH_EVERY` iters). Before, V was mask-only on Colab (no depth maps), so back faces and the interior counted as "seen" and never got the prior.
4. **Space carving.** The mask BCE uses the *unrouted* foreground opacity on every ray. With hard routing it was ≈0.0003 and carved nothing.
5. **Stage-1 only in the prefit** (solidified shell), never in main training.
6. **Prefit → main-train continuity.** The prefit ended with all 16 hash levels on, then main training restarted the anneal at γ=1 and zeroed most of the fitted features at iteration 0. The prefit now uses `PREFIT_GAMMA=6` levels and the main anneal starts there.
7. **Floating debris.** Marching cubes ran on the whole [-1,1]³ cube, but the field is only constrained inside the unit sphere; the corners became junk components. Extraction is now clipped to the sphere.
8. **Sharpness floor.** β (SDF→density) is floored by a ramp 8 → 40, so the surface stops being a fog the photo loss cannot resolve.
9. **Smaller fixes:** half-voxel offset in the Stage-1 grid lookup; marching-cubes axis order (exports were x↔z mirrored); eikonal only inside the sphere; images cached on the GPU (the old sampler re-uploaded every image every step); vectorized hash grid; dataset cell no longer trains on a synthetic stand-in when the Drive paths are wrong.

## What persists on Drive (`SatelliteFuse/GFhashBA/`)

| What | Where | Reuse |
|---|---|---|
| Stage-2 prior (occupancy, decoded mesh, raw teacher SLAT) | `cache/va_split/stage2_prior_<key>.npz` | next run skips the whole SAM3D install/load/sample/decode |
| Stage-1 prefit | `cache/va_split/prefit_<key>.pth` | next run skips the prefit |
| Resumable checkpoint (model, poses, optimizers, history) every `CKPT_EVERY` | `<RUN_TS>/va_split/gfhashba_ckpt.pth` | `RESUME_FROM="latest"` continues; a finished checkpoint jumps straight to §6.2/§7 for debugging |
| Training log, config, debug snapshots, every figure and mesh | `<RUN_TS>/va_split/` | uploaded as soon as each is written |

Cache keys hash the settings each artifact depends on, so changing e.g. `SAM_BANEU_RUN`, the LoRA, or `PRIOR_GRID_RES` rebuilds automatically. `REFRESH_CACHE=True` forces a rebuild.

## The four components

1. **Generative prior on blind spots** — MV-SAM3D Stage-2 decoded occupancy, distilled where `V≈0`
2. **Visibility-gated fusion** — photo/mask on rays; prior weighted by `1 − sg(V)` with occlusion-aware `V`
3. **Dual-field + mask routing** — Hash-SDF FG + contracted bg NeRF; SAM masks route colour, mask BCE carves
4. **Coarse-to-fine hash annealing** — unlock high-res hash levels after poses lock

## Workflow

```
§1 env → §2 config → §3 load views → §3.5 Stage-1 teacher (NeuS frame)
      → §4 modules → §5 smoke → §6.0a Stage-2 prior → §6.0 build + prefit → §6.1 train
      → §6.2 curves → §7 mesh + comparison → §7.1 renders vs inputs
```

**Needs from `sam_baneus`:** `cameras_sphere.npz`, `baneus_scene_meta.npz`, `stage1/stage1_occupancy.npz`.
**Pinned export:** `SatelliteFuse/SamBAneus/20260811_182029/va_split/` (edit `SAM_BANEU_RUN` in §2).
**RGB/masks:** `SatelliteFuse/datasets/va_split/satellite/`.
**Outputs:** `SatelliteFuse/GFhashBA/<RUN_TS>/va_split/` — report figures are `mesh_comparison.png`, `render_vs_input.png`, `train_curves.png`, `progress_it*.png`, plus `run_summary.json`.

**Debug:** after §3.5b / §6.0a / §7, paste the printed `=== DEBUG SNAPSHOT ===` JSON blocks back into chat (also saved as `debug_snapshot_*.json`).

## 0 — Method overview (why each piece exists)

### The paradox this notebook solves

- **MLP NeuS** (BA-NeuS): spectral bias → smooth early geometry → good for pose BA, slow for detail.
- **Hash grids** (Instant-NGP): learn detail immediately → with wrong poses they memorize floaters and never recover.
- **Generative priors** (MV-SAM3D): plausible geometry where no camera looks; photometric loss wants *this* building.
- **Unbounded scenes**: sky/trees steal hash capacity from the building.

GF-HashBA's answer:

```
prefit:       Hash-SDF ← solidified Stage-1 occupancy (scaffold, same Sim3 frame)
early iters:  only low hash levels ON  →  poses BA like an MLP
mid iters:    unlock finer levels      →  texture / edges
every iter:   rays → photo + mask BCE (carves) + eikonal
              random points → Stage-2 occupancy BCE × (1 − sg(V))   (prior owns blind spots)
every 500:    z-buffer the current SDF in 12 views → occlusion-aware V(x) grid
rays:         SAM mask → fg Hash-SDF colour vs bg NeRF colour (contracted)
```

## 1 — Environment (Colab A100 recommended)

Pin torch first if needed (same restart gotcha as the other notebooks).
`tinycudann` is optional — we ship a pure-PyTorch multi-res hash fallback so the
notebook runs without CUDA extensions.


In [ ]:
# 1.0 — Drive + GPU check (does not import torch, so §1.0b can still pin it)
import subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab — skipping Drive mount")

try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip())
except Exception:
    print("⚠ no NVIDIA GPU visible — Runtime → Change runtime type → A100")

### 1.0b — Pin torch 2.5.1 + numpy 1.26.4 (Stage-2 compiled stack)

`colab_compiled_pkgs.tar.gz` / kaolin are built for **torch 2.5.1+cu121** and **numpy 1.26** (same as `mvsam3d_stage2LoRA`).

**Python must be 3.11 or 3.12** — torch 2.5.1 / torchvision 0.20.1 have **no Python 3.13 wheels**.
If you see `py=3.13.x`, go to **Runtime → Change runtime type → Runtime version** and pin
e.g. **2026.07** or **2026.04** (Python 3.12), then reconnect and Run all.

If this cell installs anything it **restarts the runtime automatically**; when Colab reconnects, **Run all** again — the cell then passes.
Wrong torch → `kaolin/_C.so: undefined symbol ...`.

In [ ]:
# 1.0b — Pin torch 2.5.1+cu121 and numpy 1.26.4 (Stage-2 compiled stack ABI)
# Runs BEFORE torch/numpy are imported by later cells. If anything is pinned, the
# Colab runtime is restarted automatically — then just "Run all" again.
import os, sys, subprocess
import importlib.metadata as _md

TARGET_TORCH = "2.5.1"
TARGET_NUMPY = "1.26.4"
CU_INDEX = "https://download.pytorch.org/whl/cu121"


def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None


def _pip(*args):
    print("  pip install", " ".join(args))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


print(f"py={sys.version.split()[0]}  torch={_ver('torch')}  numpy={_ver('numpy')}")
if sys.version_info >= (3, 13):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} cannot install torch==2.5.1 / torchvision==0.20.1 "
        "(no cp313 wheels; the Stage-2 compiled stack needs 3.11/3.12).\n"
        "Fix: Runtime → Change runtime type → Runtime version → pin 2026.07 or 2026.04 "
        "(Python 3.12), Connect, then Run all."
    )

need_torch = not (_ver("torch") or "").startswith(TARGET_TORCH)
need_numpy = (_ver("numpy") or "") != TARGET_NUMPY
if need_torch:
    print(f"▶ torch {_ver('torch')} → {TARGET_TORCH}+cu121")
    # One package per call so torchvision cannot fall back to bare PyPI.
    _pip("torch==2.5.1", "--index-url", CU_INDEX)
    _pip("torchvision==0.20.1", "--index-url", CU_INDEX)
    _pip("torchaudio==2.5.1", "--index-url", CU_INDEX)
    try:
        _pip("xformers==0.0.28.post3", "--index-url", CU_INDEX)
    except Exception as e:
        print("⚠ xformers pin skipped:", e)
if need_torch or need_numpy:
    _pip(f"numpy=={TARGET_NUMPY}", "scipy==1.13.1")
    print("✓ pins installed. Restarting the runtime — when it reconnects, Run all again (this cell will pass).")
    if "google.colab" in sys.modules:
        os.kill(os.getpid(), 9)
    raise SystemExit("Restart the kernel, then re-run from the top.")

import torch
import numpy
print(f"✓ torch {torch.__version__} (CUDA {torch.version.cuda}) | numpy {numpy.__version__} — Stage-2 ABI OK")

In [ ]:
# 1.1 — Lightweight deps (the pure-PyTorch hash grid needs nothing exotic)
import os, sys, subprocess
from pathlib import Path

_pin_dir = Path("/content") if Path("/content").exists() else Path.cwd()
_pin = _pin_dir / "_numpy_pin.txt"
_pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_pin)   # keeps numpy/scipy pinned for every later pip call
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "trimesh", "matplotlib", "Pillow", "scikit-image", "opencv-python-headless",
])

# Optional: tinycudann for Instant-NGP-speed hash (skip if the build fails — we fall back)
# !pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
print(f"✓ core imports | numpy {np.__version__} | torch {torch.__version__}")

## 2 — Configuration

Edit the `<--` lines. Two data modes:

| Mode | When |
|------|------|
| `DATA_MODE="neus_bundle"` | `cameras_sphere.npz` (BA-NeuS / sam_baneus export) + **RGB/masks from SatelliteFuse** |
| `DATA_MODE="raw_views"` | Load RGB+masks from `views/` or `satellite/` (poses from DA3 npz or identity stub) |

**Persistence:** `/content` is scratch. Everything worth keeping goes to `SatelliteFuse/GFhashBA/` on Drive (see the table at the top). Debug a later step without redoing earlier ones with `RESUME_FROM="latest"` (cached prior + prefit are picked up automatically).

In [ ]:
# 2.0 — Paths and hyper-parameters  (EDIT the <-- marked lines)
from pathlib import Path
from datetime import datetime, timezone
import os

CASE_NAME   = "va_split"                                          # <--
DATA_MODE   = "neus_bundle"                                       # <-- "neus_bundle" | "raw_views"

# ---- va_split on Drive (source RGB + masks; same as sam_baneus / mvsam3d) ----
DATA_ROOT   = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")  # <--
AERIAL_DIR  = DATA_ROOT / "satellite"
IMG_SUBDIR  = "images"
MASK_SUBDIR = "masks"
if not DATA_ROOT.exists():
    _local = Path("/Users/taiganishida/Documents/GSV-MSV/datasets/va_split")
    if _local.exists():
        DATA_ROOT, AERIAL_DIR = _local, _local / "satellite"

VIEW_DIR   = Path("/content/gfhashba_out") / "views"
DA3_NPZ    = Path("/content/gfhashba_out") / "da3_views" / "da3_output.npz"

# BA-NeuS cameras (normalized sphere frame). Images stay on SatelliteFuse.
NEUS_CASE_DIR   = Path("/content/bundle-adjusting-neus/public_data") / CASE_NAME

def _first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

# Prefer the validated Stage-1 + Sim3 export. Set None to auto-pick latest SamBAneus run.
SAM_BANEU_RUN = "20260811_182029"  # <-- good alignment (edge-refined); None = latest
_SAM_ROOT = Path("/content/drive/MyDrive/SatelliteFuse/SamBAneus")

def _latest_case_dir(root, case):
    if not Path(root).exists():
        return None
    cands = sorted([p for p in Path(root).iterdir() if p.is_dir() and (p / case).exists()], reverse=True)
    return (cands[0] / case) if cands else None

if SAM_BANEU_RUN:
    BANEUS_META_DIR = _SAM_ROOT / SAM_BANEU_RUN / CASE_NAME
    if not BANEUS_META_DIR.exists():
        print(f"⚠ pinned SamBAneus run missing: {BANEUS_META_DIR} — falling back to latest")
        BANEUS_META_DIR = _latest_case_dir(_SAM_ROOT, CASE_NAME)
else:
    BANEUS_META_DIR = _latest_case_dir(_SAM_ROOT, CASE_NAME)

if BANEUS_META_DIR is None:
    BANEUS_META_DIR = _first_existing(
        Path("/content/drive/MyDrive/baneus_ultrra_results") / CASE_NAME,
        Path("/content/drive/MyDrive/SatelliteFuse/baneus_ultrra_results") / CASE_NAME,
    )

BANEUS_LOCAL = _first_existing(
    Path("/content/sam_baneus_ultrra_out"),
    Path("/content/baneus_ultrra_out"),
) or Path("/content/sam_baneus_ultrra_out")

# Drive export first (pinned/good), then local Colab out, then NeuS public_data.
CAMERAS_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "cameras_sphere.npz"] if BANEUS_META_DIR else [] ),
    NEUS_CASE_DIR / "cameras_sphere.npz",
    BANEUS_LOCAL / "cameras_sphere.npz",
)
SCENE_META_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "baneus_scene_meta.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "baneus_scene_meta.npz",
    Path("/content/gfhashba_out") / "baneus_scene_meta.npz",
)
NEUS_DEPTH_DIR = NEUS_CASE_DIR / "depth" if (NEUS_CASE_DIR / "depth").is_dir() else None

STAGE1_DIR = _first_existing(
    *( [BANEUS_META_DIR / "stage1"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1",
    Path("/content/gfhashba_out") / "stage1",
)
STAGE1_OCC_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "stage1" / "stage1_occupancy.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1" / "stage1_occupancy.npz",
    Path("/content/gfhashba_out") / "stage1" / "stage1_occupancy.npz",
)
STAGE1_FRAME_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "stage1" / "stage1_frame.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1" / "stage1_frame.npz",
    Path("/content/gfhashba_out") / "stage1" / "stage1_frame.npz",
)

# Training resolution — must match cameras_sphere export (auto from baneus_scene_meta.npz)
TRAIN_W = TRAIN_H = None   # None → read meta train_w/train_h, else 768×576

# ---- where things live: /content is scratch (wiped with the runtime); Drive persists ----
OUTPUT_DIR = Path("/content/gfhashba_out"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TS     = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
GFHASHBA_DRIVE = Path("/content/drive/MyDrive/SatelliteFuse/GFhashBA")
HAVE_DRIVE = Path("/content/drive/MyDrive").exists()
DRIVE_OUT  = GFHASHBA_DRIVE / RUN_TS / CASE_NAME          # this run: figures, meshes, checkpoints, logs
CACHE_DIR  = (GFHASHBA_DRIVE if HAVE_DRIVE else OUTPUT_DIR) / "cache" / CASE_NAME   # reused across runs
USE_CACHE     = True    # <-- reuse the cached Stage-2 prior / Stage-1 prefit when their settings match
REFRESH_CACHE = False   # <-- True = recompute both and overwrite the caches
RESUME_FROM   = None    # <-- None | "latest" | path to a gfhashba_ckpt.pth on Drive (finished ckpt → straight to §6.2/§7)

# ---- run length ----
RUN_PRESET = "report"   # <-- "report" (20k iters) | "full" (30k) | "smoke" (600 iters, pipeline check only)
END_ITER     = {"smoke": 600, "report": 20000, "full": 30000}[RUN_PRESET]
ANNEAL_ITERS = {"smoke": 300, "report": 6000, "full": 8000}[RUN_PRESET]   # γ: 1 → L hash levels
SNAPSHOT_EVERY = max(500, END_ITER // 4)   # progress mesh preview → Drive
CKPT_EVERY     = 2000                      # resumable checkpoint (model + poses + optimizers + history) → Drive
MESH_RES       = 256                       # final marching-cubes grid
PREVIEW_RES    = 128                       # prefit / progress previews

# ---- model ----
HASH_N_LEVELS     = 16
HASH_N_FEATURES   = 2
HASH_LOG2_SIZE    = 19
HASH_BASE_RES     = 16
HASH_MAX_RES      = 2048
SDF_HIDDEN        = 64
SDF_N_LAYERS      = 2
BG_HIDDEN         = 64

# ---- training ----
BATCH_RAYS        = 1024
N_SAMPLES         = 64
N_BG_SAMPLES      = 32
LR_SDF            = 1e-2
LR_COLOR          = 1e-2
LR_POSE           = 1e-3
WARMUP_END        = 1000
POSE_START_ITER   = max(200, WARMUP_END // 5)

# ---- loss weights ----
W_PHOTO           = 1.0
W_EIKONAL         = 0.1
# SDF→density sharpness β: learnable, floored by a ramp (soft early, sharp later).
BETA_FLOOR_START  = 8.0
BETA_FLOOR_END    = 40.0
BETA_FLOOR_FRAC   = 0.6     # reach BETA_FLOOR_END at 60 % of training
W_MASK            = 0.5     # BCE on the *unrouted* FG opacity, all rays → also carves outside the masks
# Stage-2 prior (blind spots): hold W_FLOW_START for FLOW_HOLD_FRAC, then cosine → W_FLOW_END.
# Applied every iteration on PRIOR_N_PTS random points, weighted by (1 − V).
W_FLOW_START      = 0.5
W_FLOW_END        = 0.1     # floor > 0: unseen regions keep their prior late in training
FLOW_HOLD_FRAC    = 0.35
FLOW_ANNEAL_ITERS = END_ITER
PRIOR_N_PTS       = 8192
PRIOR_TAU         = 0.02    # occupancy proxy o = σ(−sdf/τ)
PRIOR_GRID_RES    = 128     # dense teacher occupancy grid on NeuS [-1,1]^3

# ---- Stage-1 prefit (Stage-1 only; no Stage-2 here) ----
USE_STAGE1_PREFIT = True
SDF_PREFIT_ITERS  = 2000
SDF_PREFIT_BATCH  = 8192
SDF_PREFIT_LR     = 1e-2
PREFIT_SOLIDIFY   = True    # Stage-1 occupancy is a surface shell; fill enclosed interior before fitting
PREFIT_GAMMA      = 6.0     # hash levels used by the prefit (res ≤ ~80 ≈ Stage-1 detail)
# Main training continues the anneal from the prefit's level count, so iteration 0 sees the fitted field.
ANNEAL_START_LVL  = PREFIT_GAMMA if USE_STAGE1_PREFIT else 1.0

# ---- generative prior ----
# "stage2" (MV-SAM3D Stage-2 decoded geometry) | "occupancy" (Stage-1 grid) | "stub" | "off"
USE_FLOW_PRIOR    = True          # <--
FLOW_PRIOR_MODE   = "stage2"      # <--
STAGE2_LORA_DIR   = Path("/content/drive/MyDrive/SatelliteFuse/Stage2LoRA/20260813_181113/va_split/lora_adapter")  # <-- or None
STAGE2_TEACHER_STEPS = 25
STAGE2_MICRO_N   = 8
USE_EXPORTED_STAGE1_COORDS = True  # <-- keep True (same voxels as the prefit)
STAGE2_FALLBACK_TO_STAGE1 = True   # if the Stage-2 stack fails, still train with the Stage-1 prior (logged loudly)
# False = prior only on blind spots (1−V) — default.  True = also on visible regions (A/B ablation).
FLOW_PRIOR_ON_VISIBLE = False  # <--

# ---- occlusion-aware visibility V(x) ----
VIS_DEPTH_SOURCE  = "model"   # "model" = z-buffers rendered from the current Hash-SDF | "none" = mask-only (x-ray)
VIS_MAX_VIEWS     = 12
VIS_ZBUF_DS       = 6         # z-buffer downsample vs train resolution
VIS_ZBUF_SAMPLES  = 128
VIS_REFRESH_EVERY = 500
VIS_GRID_RES      = 48
VIS_EPS_ABS       = 0.05      # NeuS units: behind the surface by more than this ⇒ occluded
VIS_EPS_REL       = 0.01

# ---- dual-field mask routing ----
HARD_MASK_ROUTE   = True      # masked rays → FG colour; unmasked → BG colour. Mask loss is never routed.

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CASE={CASE_NAME} | mode={DATA_MODE} | device={DEVICE} | preset={RUN_PRESET} ({END_ITER} iters)")
print(f"DATA_ROOT={DATA_ROOT} | aerial images={AERIAL_DIR / IMG_SUBDIR}")
print(f"SAM_BANEU_RUN={SAM_BANEU_RUN} | BANEUS_META_DIR={BANEUS_META_DIR}")
print(f"CAMERAS_NPZ={CAMERAS_NPZ} | meta={SCENE_META_NPZ}")
print(f"STAGE1_OCC={STAGE1_OCC_NPZ}")
print(f"prior={FLOW_PRIOR_MODE if USE_FLOW_PRIOR else 'off'} | on_visible={FLOW_PRIOR_ON_VISIBLE} | "
      f"W hold {W_FLOW_START} for {FLOW_HOLD_FRAC:.0%} → {W_FLOW_END} | LoRA={STAGE2_LORA_DIR}")
print(f"visibility={VIS_DEPTH_SOURCE} z-buffers ({VIS_MAX_VIEWS} views, refresh {VIS_REFRESH_EVERY}) | "
      f"prefit={USE_STAGE1_PREFIT}x{SDF_PREFIT_ITERS} solid={PREFIT_SOLIDIFY}")
print(f"OUTPUT_DIR={OUTPUT_DIR} (scratch) | DRIVE_OUT={DRIVE_OUT}")
print(f"CACHE_DIR={CACHE_DIR} | USE_CACHE={USE_CACHE} REFRESH_CACHE={REFRESH_CACHE} | RESUME_FROM={RESUME_FROM}")
if not HAVE_DRIVE:
    print("⚠ Drive is not mounted — nothing will persist after this runtime (run §1.0 first)")

import hashlib, json, shutil


def to_drive(*paths):
    """Best-effort copy of finished artifacts to DRIVE_OUT (a Drive hiccup never kills a run)."""
    if not HAVE_DRIVE:
        return
    try:
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print("⚠ Drive mkdir failed:", e)
        return
    for p in paths:
        if p is None or not Path(p).exists():
            continue
        try:
            shutil.copy2(p, DRIVE_OUT / Path(p).name)
        except Exception as e:
            print(f"⚠ Drive copy failed for {Path(p).name}: {e}")


def cache_key(*parts):
    """Short hash of everything a cached artifact depends on."""
    return hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:12]


CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG = {k: (str(v) if isinstance(v, Path) else v) for k, v in dict(globals()).items()
              if k.isupper() and isinstance(v, (int, float, str, bool, type(None), Path))}
(OUTPUT_DIR / "config.json").write_text(json.dumps(RUN_CONFIG, indent=2))
to_drive(OUTPUT_DIR / "config.json")

### 2.1 — Write `gfhashba_align.py` (Colab self-contained)

No local tests required. Dumps Sim3 / densify / mesh-health helpers to disk so
§3.5b / §6.0a / §7 can import them. Re-run after uploading a new notebook.

In [ ]:
# 2.1 — Ensure gfhashba_align.py is on disk (Colab)
from pathlib import Path
import sys, base64, importlib

_ALIGN_B64 = """IiIiR0YtSGFzaEJBIGZyYW1lIG1hdGgg4oCUIHNoYXJlZCBieSBub3RlYm9vayArIGxvY2FsIHVuaXQgdGVzdHMuCgpDYW5vbmljYWwgKFN0YWdlLTEgLyBTTEFUKSDihpIgREEzIHdvcmxkIChTaW0zKSDihpIgTmV1UyB1bml0IHNwaGVyZSBbLTEsMV1eMwoKTmV2ZXIgQUFCQi1zdHJldGNoIFNMQVQgaW5kaWNlcyBpbnRvIHRoZSBOZXVTIGN1YmUuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBNYXBwaW5nLCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCgp0cnk6CiAgICBpbXBvcnQgdG9yY2gKICAgIEhBU19UT1JDSCA9IFRydWUKZXhjZXB0IEltcG9ydEVycm9yOiAgIyBwcmFnbWE6IG5vIGNvdmVyCiAgICBIQVNfVE9SQ0ggPSBGYWxzZQogICAgdG9yY2ggPSBOb25lICAjIHR5cGU6IGlnbm9yZQoKCkBkYXRhY2xhc3MKY2xhc3MgQWxpZ25TaW0zOgogICAgTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkOiBucC5uZGFycmF5CiAgICB3b3JsZF9hbGlnbl9zOiBmbG9hdAogICAgd29ybGRfYWxpZ25fUjogbnAubmRhcnJheQogICAgd29ybGRfYWxpZ25fdDogbnAubmRhcnJheQogICAgY2VudGVyOiBucC5uZGFycmF5CiAgICByYWRpdXM6IGZsb2F0CiAgICB2b3hlbF9ncmlkOiBpbnQgPSA2NAoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGlkZW50aXR5KGNscywgdm94ZWxfZ3JpZDogaW50ID0gNjQpIC0+ICJBbGlnblNpbTMiOgogICAgICAgIHJldHVybiBjbHMoCiAgICAgICAgICAgIE1fY2Fub25pY2FsX3RvX2RhM193b3JsZD1ucC5leWUoNCwgZHR5cGU9bnAuZmxvYXQ2NCksCiAgICAgICAgICAgIHdvcmxkX2FsaWduX3M9MS4wLAogICAgICAgICAgICB3b3JsZF9hbGlnbl9SPW5wLmV5ZSgzLCBkdHlwZT1ucC5mbG9hdDY0KSwKICAgICAgICAgICAgd29ybGRfYWxpZ25fdD1ucC56ZXJvcygzLCBkdHlwZT1ucC5mbG9hdDY0KSwKICAgICAgICAgICAgY2VudGVyPW5wLnplcm9zKDMsIGR0eXBlPW5wLmZsb2F0NjQpLAogICAgICAgICAgICByYWRpdXM9MS4wLAogICAgICAgICAgICB2b3hlbF9ncmlkPWludCh2b3hlbF9ncmlkKSwKICAgICAgICApCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV9tYXBwaW5nKGNscywgZDogTWFwcGluZ1tzdHIsIEFueV0pIC0+ICJBbGlnblNpbTMiOgogICAgICAgIGRlZiBfbnAoa2V5LCBkZWZhdWx0KToKICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBkOgogICAgICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkoZGVmYXVsdCwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICAgICAgdiA9IGRba2V5XQogICAgICAgICAgICBpZiBIQVNfVE9SQ0ggYW5kIGlzaW5zdGFuY2UodiwgdG9yY2guVGVuc29yKToKICAgICAgICAgICAgICAgIHYgPSB2LmRldGFjaCgpLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkodiwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAgTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkPV9ucCgiTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkIiwgbnAuZXllKDQpKSwKICAgICAgICAgICAgd29ybGRfYWxpZ25fcz1mbG9hdChkLmdldCgid29ybGRfYWxpZ25fcyIsIDEuMCkpLAogICAgICAgICAgICB3b3JsZF9hbGlnbl9SPV9ucCgid29ybGRfYWxpZ25fUiIsIG5wLmV5ZSgzKSksCiAgICAgICAgICAgIHdvcmxkX2FsaWduX3Q9X25wKCJ3b3JsZF9hbGlnbl90IiwgbnAuemVyb3MoMykpLAogICAgICAgICAgICBjZW50ZXI9X25wKCJjZW50ZXIiLCBucC56ZXJvcygzKSksCiAgICAgICAgICAgIHJhZGl1cz1mbG9hdChkLmdldCgicmFkaXVzIiwgMS4wKSksCiAgICAgICAgICAgIHZveGVsX2dyaWQ9aW50KGQuZ2V0KCJ2b3hlbF9ncmlkIiwgNjQpKSwKICAgICAgICApCgogICAgZGVmIGFzX3RvcmNoX2RpY3Qoc2VsZik6CiAgICAgICAgaWYgbm90IEhBU19UT1JDSDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaCByZXF1aXJlZCIpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCI6IHRvcmNoLnRlbnNvcihzZWxmLk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCwgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICAgICAgICAgICJ3b3JsZF9hbGlnbl9zIjogZmxvYXQoc2VsZi53b3JsZF9hbGlnbl9zKSwKICAgICAgICAgICAgIndvcmxkX2FsaWduX1IiOiB0b3JjaC50ZW5zb3Ioc2VsZi53b3JsZF9hbGlnbl9SLCBkdHlwZT10b3JjaC5mbG9hdDMyKSwKICAgICAgICAgICAgIndvcmxkX2FsaWduX3QiOiB0b3JjaC50ZW5zb3Ioc2VsZi53b3JsZF9hbGlnbl90LCBkdHlwZT10b3JjaC5mbG9hdDMyKSwKICAgICAgICAgICAgImNlbnRlciI6IHRvcmNoLnRlbnNvcihzZWxmLmNlbnRlciwgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICAgICAgICAgICJyYWRpdXMiOiBmbG9hdChzZWxmLnJhZGl1cyksCiAgICAgICAgICAgICJ2b3hlbF9ncmlkIjogaW50KHNlbGYudm94ZWxfZ3JpZCksCiAgICAgICAgfQoKCmRlZiB2b3hlbF9pZHhfdG9fY2Fub25pY2FsKGNvb3Jkc194eXo6IG5wLm5kYXJyYXksIHZveGVsX2dyaWQ6IGludCA9IDY0KSAtPiBucC5uZGFycmF5OgogICAgYyA9IG5wLmFzYXJyYXkoY29vcmRzX3h5eiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGMubmRpbSAhPSAyIG9yIGMuc2hhcGVbMV0gbm90IGluICgzLCA0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY29vcmRzIGV4cGVjdGVkIChOLDN8NCksIGdvdCB7Yy5zaGFwZX0iKQogICAgaWYgYy5zaGFwZVsxXSA9PSA0OgogICAgICAgIGMgPSBjWzosIDE6NF0KICAgIHJldHVybiBjIC8gZmxvYXQodm94ZWxfZ3JpZCkgLSAwLjUKCgpkZWYgY2Fub25pY2FsX3RvX25ldXNfbm9ybSh4X2NhbjogbnAubmRhcnJheSwgYWxpZ246IEFsaWduU2ltMykgLT4gbnAubmRhcnJheToKICAgIHggPSBucC5hc2FycmF5KHhfY2FuLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgb25lcyA9IG5wLm9uZXMoKHguc2hhcGVbMF0sIDEpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgeF91bmFsID0gKG5wLmNvbmNhdGVuYXRlKFt4LCBvbmVzXSwgYXhpcz0tMSkgQCBhbGlnbi5NX2Nhbm9uaWNhbF90b19kYTNfd29ybGQpWzosIDozXQogICAgeF93b3JsZCA9IGZsb2F0KGFsaWduLndvcmxkX2FsaWduX3MpICogKHhfdW5hbCBAIGFsaWduLndvcmxkX2FsaWduX1IpICsgYWxpZ24ud29ybGRfYWxpZ25fdAogICAgcmV0dXJuICh4X3dvcmxkIC0gYWxpZ24uY2VudGVyKSAvIGZsb2F0KGFsaWduLnJhZGl1cykKCgpkZWYgbmV1c19ub3JtX3RvX2Nhbm9uaWNhbCh4X25vcm06IG5wLm5kYXJyYXksIGFsaWduOiBBbGlnblNpbTMpIC0+IG5wLm5kYXJyYXk6CiAgICB4X25vcm0gPSBucC5hc2FycmF5KHhfbm9ybSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHhfd29ybGQgPSB4X25vcm0gKiBmbG9hdChhbGlnbi5yYWRpdXMpICsgYWxpZ24uY2VudGVyCiAgICB4X3VuYWwgPSAoeF93b3JsZCAtIGFsaWduLndvcmxkX2FsaWduX3QpIEAgYWxpZ24ud29ybGRfYWxpZ25fUi5UIC8gZmxvYXQoYWxpZ24ud29ybGRfYWxpZ25fcykKICAgIG9uZXMgPSBucC5vbmVzKCh4X3VuYWwuc2hhcGVbMF0sIDEpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgTV9pbnYgPSBucC5saW5hbGcuaW52KGFsaWduLk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCkKICAgIHJldHVybiAobnAuY29uY2F0ZW5hdGUoW3hfdW5hbCwgb25lc10sIGF4aXM9LTEpIEAgTV9pbnYpWzosIDozXQoKCmRlZiB2b3hlbF9pZHhfdG9fbmV1cyhjb29yZHNfeHl6OiBucC5uZGFycmF5LCBhbGlnbjogQWxpZ25TaW0zKSAtPiBucC5uZGFycmF5OgogICAgcmV0dXJuIGNhbm9uaWNhbF90b19uZXVzX25vcm0odm94ZWxfaWR4X3RvX2Nhbm9uaWNhbChjb29yZHNfeHl6LCBhbGlnbi52b3hlbF9ncmlkKSwgYWxpZ24pCgoKZGVmIGFhYmJfc3RyZXRjaF90b19ncmlkKGNvb3Jkc194eXo6IG5wLm5kYXJyYXksIHJlczogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiQlVHR1kgb2xkIGRlbnNpZnkuIFJldHVybnMgKE4sMykgaW50ZWdlciB4LHksei4iIiIKICAgIGMgPSBucC5hc2FycmF5KGNvb3Jkc194eXosIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpZiBjLnNoYXBlWzFdID09IDQ6CiAgICAgICAgYyA9IGNbOiwgMTo0XQogICAgY21pbiwgY21heCA9IGMubWluKDApLCBjLm1heCgwKQogICAgc3BhbiA9IG5wLm1heGltdW0oY21heCAtIGNtaW4sIDEuMCkKICAgIGMwMSA9IChjIC0gY21pbikgLyBzcGFuCiAgICByZXR1cm4gbnAucm91bmQoYzAxICogKHJlcyAtIDEpKS5hc3R5cGUobnAuaW50NjQpLmNsaXAoMCwgcmVzIC0gMSkKCgpkZWYgc2ltM19zb2Z0X3NwbGF0X3RvX2dyaWQoCiAgICBjb29yZHNfeHl6OiBucC5uZGFycmF5LAogICAgd2VpZ2h0czogT3B0aW9uYWxbbnAubmRhcnJheV0sCiAgICBhbGlnbjogQWxpZ25TaW0zLAogICAgcmVzOiBpbnQgPSA2NCwKKSAtPiBucC5uZGFycmF5OgogICAgeF9ub3JtID0gdm94ZWxfaWR4X3RvX25ldXMoY29vcmRzX3h5eiwgYWxpZ24pCiAgICBnID0gKHhfbm9ybSArIDEuMCkgKiAwLjUgKiAocmVzIC0gMSkKICAgIGluc2lkZSA9IG5wLmFsbChucC5hYnMoeF9ub3JtKSA8PSAxLjA1LCBheGlzPTEpCiAgICBnID0gZ1tpbnNpZGVdCiAgICBpZiB3ZWlnaHRzIGlzIE5vbmU6CiAgICAgICAgdyA9IG5wLm9uZXMoZy5zaGFwZVswXSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGVsc2U6CiAgICAgICAgdyA9IG5wLmFzYXJyYXkod2VpZ2h0cywgZHR5cGU9bnAuZmxvYXQ2NClbaW5zaWRlXQogICAgICAgIHcgPSB3IC8gbWF4KGZsb2F0KHcubWVhbigpKSwgMWUtOCkKICAgIHZvbCA9IG5wLnplcm9zKChyZXMsIHJlcywgcmVzKSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGcuc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiB2b2wKICAgIHgsIHksIHogPSBnWzosIDBdLCBnWzosIDFdLCBnWzosIDJdCiAgICB4MCA9IG5wLmZsb29yKHgpLmFzdHlwZShucC5pbnQ2NCkKICAgIHkwID0gbnAuZmxvb3IoeSkuYXN0eXBlKG5wLmludDY0KQogICAgejAgPSBucC5mbG9vcih6KS5hc3R5cGUobnAuaW50NjQpCiAgICBmb3IgZHosIHd6IGluICgoMCwgMS4wIC0gKHogLSB6MCkpLCAoMSwgKHogLSB6MCkpKToKICAgICAgICBmb3IgZHksIHd5IGluICgoMCwgMS4wIC0gKHkgLSB5MCkpLCAoMSwgKHkgLSB5MCkpKToKICAgICAgICAgICAgZm9yIGR4LCB3eCBpbiAoKDAsIDEuMCAtICh4IC0geDApKSwgKDEsICh4IC0geDApKSk6CiAgICAgICAgICAgICAgICBpeCA9IG5wLmNsaXAoeDAgKyBkeCwgMCwgcmVzIC0gMSkKICAgICAgICAgICAgICAgIGl5ID0gbnAuY2xpcCh5MCArIGR5LCAwLCByZXMgLSAxKQogICAgICAgICAgICAgICAgaXogPSBucC5jbGlwKHowICsgZHosIDAsIHJlcyAtIDEpCiAgICAgICAgICAgICAgICBucC5hZGQuYXQodm9sLCAoaXosIGl5LCBpeCksIG5wLmFzYXJyYXkod3ogKiB3eSAqIHd4ICogdywgZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICBtID0gZmxvYXQodm9sLm1heCgpKQogICAgaWYgbSA+IDA6CiAgICAgICAgdm9sIC89IG0KICAgIHJldHVybiB2b2wKCgpkZWYgZ3JpZF9pb3UoYTogbnAubmRhcnJheSwgYjogbnAubmRhcnJheSwgdGhyX2E6IGZsb2F0ID0gMC4xLCB0aHJfYjogZmxvYXQgPSAwLjUpIC0+IGZsb2F0OgogICAgYWEsIGJiID0gYSA+IHRocl9hLCBiID4gdGhyX2IKICAgIGludGVyID0gbnAubG9naWNhbF9hbmQoYWEsIGJiKS5zdW0oKQogICAgdW5pb24gPSBucC5sb2dpY2FsX29yKGFhLCBiYikuc3VtKCkKICAgIHJldHVybiBmbG9hdChpbnRlciAvIG1heCh1bmlvbiwgMSkpCgoKZGVmIGdyaWRfY29ycihhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIGFhID0gYS5yYXZlbCgpLmFzdHlwZShucC5mbG9hdDY0KQogICAgYmIgPSBiLnJhdmVsKCkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBhYSA9IChhYSAtIGFhLm1lYW4oKSkgLyAoYWEuc3RkKCkgKyAxZS04KQogICAgYmIgPSAoYmIgLSBiYi5tZWFuKCkpIC8gKGJiLnN0ZCgpICsgMWUtOCkKICAgIHJldHVybiBmbG9hdCgoYWEgKiBiYikubWVhbigpKQoKCmRlZiBzZGZfdm9sdW1lX2hlYWx0aCh2b2w6IG5wLm5kYXJyYXksIGxldmVsOiBmbG9hdCA9IDAuMCkgLT4gZGljdDoKICAgIHZvbCA9IG5wLmFzYXJyYXkodm9sLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgaW5zaWRlID0gdm9sIDwgbGV2ZWwKICAgIGZyYWNfaW5zaWRlID0gZmxvYXQoaW5zaWRlLm1lYW4oKSkKICAgIHZtaW4sIHZtYXggPSBmbG9hdCh2b2wubWluKCkpLCBmbG9hdCh2b2wubWF4KCkpCiAgICBjcm9zc2VzID0gYm9vbCh2bWluIDwgbGV2ZWwgPCB2bWF4KQogICAgcmV0dXJuIHsKICAgICAgICAidm1pbiI6IHZtaW4sCiAgICAgICAgInZtYXgiOiB2bWF4LAogICAgICAgICJsZXZlbCI6IGZsb2F0KGxldmVsKSwKICAgICAgICAiY3Jvc3Nlc196ZXJvIjogY3Jvc3NlcywKICAgICAgICAiZnJhY19pbnNpZGUiOiBmcmFjX2luc2lkZSwKICAgICAgICAiZnJhY19vdXRzaWRlIjogMS4wIC0gZnJhY19pbnNpZGUsCiAgICAgICAgImR5bmFtaWNfcmFuZ2UiOiB2bWF4IC0gdm1pbiwKICAgICAgICAib2tfc2lnbl9iYWxhbmNlIjogY3Jvc3NlcyBhbmQgKDAuMDEgPD0gZnJhY19pbnNpZGUgPD0gMC44NSksCiAgICAgICAgIm9rX3JhbmdlIjogKHZtYXggLSB2bWluKSA+IDFlLTMsCiAgICB9CgoKZGVmIG1lc2hfaGVhbHRoKHZlcnRpY2VzOiBucC5uZGFycmF5LCBmYWNlczogbnAubmRhcnJheSwgb2NjX2Jib3g6IE9wdGlvbmFsW25wLm5kYXJyYXldID0gTm9uZSkgLT4gZGljdDoKICAgIHYgPSBucC5hc2FycmF5KHZlcnRpY2VzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgZiA9IG5wLmFzYXJyYXkoZmFjZXMsIGR0eXBlPW5wLmludDY0KQogICAgbl92LCBuX2YgPSBsZW4odiksIGxlbihmKQogICAgaWYgbl92ID09IDAgb3Igbl9mID09IDA6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm5fdmVydHMiOiBuX3YsICJuX2ZhY2VzIjogbl9mLCAibl9jb21wb25lbnRzIjogMCwgImxhcmdlc3RfZnJhYyI6IDAuMCwKICAgICAgICAgICAgImJib3hfbWluIjogTm9uZSwgImJib3hfbWF4IjogTm9uZSwgIm9rIjogRmFsc2UsICJyZWFzb24iOiAiZW1wdHkgbWVzaCIsCiAgICAgICAgfQogICAgcGFyZW50ID0gbnAuYXJhbmdlKG5fdikKCiAgICBkZWYgZmluZChpOiBpbnQpIC0+IGludDoKICAgICAgICB3aGlsZSBwYXJlbnRbaV0gIT0gaToKICAgICAgICAgICAgcGFyZW50W2ldID0gcGFyZW50W3BhcmVudFtpXV0KICAgICAgICAgICAgaSA9IGludChwYXJlbnRbaV0pCiAgICAgICAgcmV0dXJuIGkKCiAgICBkZWYgdW5pb24oYTogaW50LCBiOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgcmEsIHJiID0gZmluZChhKSwgZmluZChiKQogICAgICAgIGlmIHJhICE9IHJiOgogICAgICAgICAgICBwYXJlbnRbcmJdID0gcmEKCiAgICBmb3IgdHJpIGluIGY6CiAgICAgICAgdW5pb24oaW50KHRyaVswXSksIGludCh0cmlbMV0pKQogICAgICAgIHVuaW9uKGludCh0cmlbMV0pLCBpbnQodHJpWzJdKSkKCiAgICB1c2VkID0gbnAudW5pcXVlKGYucmF2ZWwoKSkKICAgIHJvb3RzID0gW2ZpbmQoaW50KGkpKSBmb3IgaSBpbiB1c2VkXQogICAgY250ID0gQ291bnRlcihyb290cykKICAgIG5fY29tcCA9IGxlbihjbnQpCiAgICBsYXJnZXN0ID0gbWF4KGNudC52YWx1ZXMoKSkgaWYgY250IGVsc2UgMAogICAgbGFyZ2VzdF9mcmFjID0gZmxvYXQobGFyZ2VzdCAvIG1heChsZW4odXNlZCksIDEpKQogICAgYmJfbWluLCBiYl9tYXggPSB2Lm1pbigwKSwgdi5tYXgoMCkKICAgIGJiX2V4dCA9IGJiX21heCAtIGJiX21pbgogICAgb2sgPSBuX2NvbXAgPD0gMjAgYW5kIGxhcmdlc3RfZnJhYyA+PSAwLjUgYW5kIG5fZiA+PSA1MDAKICAgIHJlYXNvbiA9ICJvayIKICAgIGlmIG5fY29tcCA+IDIwOgogICAgICAgIHJlYXNvbiA9IGYidG9vIG1hbnkgY29tcG9uZW50cyAoe25fY29tcH0pIOKAlCBmbG9hdGluZyBkZWJyaXMiCiAgICBlbGlmIGxhcmdlc3RfZnJhYyA8IDAuNToKICAgICAgICByZWFzb24gPSBmImxhcmdlc3QgY29tcG9uZW50IG9ubHkge2xhcmdlc3RfZnJhYzouMiV9IG9mIHZlcnRzIgogICAgZWxpZiBuX2YgPCA1MDA6CiAgICAgICAgcmVhc29uID0gInRvbyBmZXcgZmFjZXMiCiAgICBvdXQgPSB7CiAgICAgICAgIm5fdmVydHMiOiBuX3YsICJuX2ZhY2VzIjogbl9mLCAibl9jb21wb25lbnRzIjogbl9jb21wLCAibGFyZ2VzdF9mcmFjIjogbGFyZ2VzdF9mcmFjLAogICAgICAgICJiYm94X21pbiI6IGJiX21pbiwgImJib3hfbWF4IjogYmJfbWF4LCAiYmJveF9leHRlbnQiOiBiYl9leHQsICJvayI6IG9rLCAicmVhc29uIjogcmVhc29uLAogICAgfQogICAgaWYgb2NjX2Jib3ggaXMgbm90IE5vbmU6CiAgICAgICAgb2NjX21pbiwgb2NjX21heCA9IG9jY19iYm94WzBdLCBvY2NfYmJveFsxXQogICAgICAgIG1uID0gbnAubWF4aW11bShiYl9taW4sIG9jY19taW4pCiAgICAgICAgbXggPSBucC5taW5pbXVtKGJiX21heCwgb2NjX21heCkKICAgICAgICBpbnRlciA9IGZsb2F0KG5wLm1heGltdW0oMC4wLCBteCAtIG1uKS5wcm9kKCkpCiAgICAgICAgdW5pb25fdiA9IGZsb2F0KGJiX2V4dC5wcm9kKCkgKyBucC5tYXhpbXVtKDAuMCwgb2NjX21heCAtIG9jY19taW4pLnByb2QoKSAtIGludGVyKQogICAgICAgIG91dFsiYmJveF9pb3VfdnNfb2NjIl0gPSBpbnRlciAvIG1heCh1bmlvbl92LCAxZS0xMikKICAgICAgICBvdXRbIm9rX2Jib3hfb3ZlcmxhcCJdID0gb3V0WyJiYm94X2lvdV92c19vY2MiXSA+PSAwLjE1CiAgICAgICAgaWYgb3V0WyJvayJdIGFuZCBub3Qgb3V0WyJva19iYm94X292ZXJsYXAiXToKICAgICAgICAgICAgb3V0WyJvayJdID0gRmFsc2UKICAgICAgICAgICAgb3V0WyJyZWFzb24iXSA9IGYibWVzaCBiYm94IElvVSB2cyBPQ0M9e291dFsnYmJveF9pb3VfdnNfb2NjJ106LjNmfSB0b28gbG93IgogICAgcmV0dXJuIG91dAoKCmRlZiBvY2NfYmJveF9mcm9tX2dyaWQob2NjOiBucC5uZGFycmF5LCB0aHI6IGZsb2F0ID0gMC41KSAtPiBPcHRpb25hbFtucC5uZGFycmF5XToKICAgIG9jYyA9IG5wLmFzYXJyYXkob2NjKQogICAgaWR4ID0gbnAuYXJnd2hlcmUob2NjID4gdGhyKQogICAgaWYgbGVuKGlkeCkgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgcmVzID0gb2NjLnNoYXBlWzBdCiAgICB6eXggPSBpZHguYXN0eXBlKG5wLmZsb2F0NjQpCiAgICB4eXogPSBucC5zdGFjayhbCiAgICAgICAgenl4WzosIDJdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICAgICAgenl4WzosIDFdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICAgICAgenl4WzosIDBdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICBdLCBheGlzPS0xKQogICAgcmV0dXJuIG5wLnN0YWNrKFt4eXoubWluKDApLCB4eXoubWF4KDApXSwgYXhpcz0wKQoKCmRlZiB0b3JjaF9jYW5vbmljYWxfdG9fbmV1cyh4X2NhbiwgYWxpZ25fZGljdCk6CiAgICBpZiBub3QgSEFTX1RPUkNIOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigidG9yY2ggcmVxdWlyZWQiKQogICAgYSA9IEFsaWduU2ltMy5mcm9tX21hcHBpbmcoYWxpZ25fZGljdCkKICAgIHRkID0gYS5hc190b3JjaF9kaWN0KCkKICAgIGRldiwgZHR5cGUgPSB4X2Nhbi5kZXZpY2UsIHhfY2FuLmR0eXBlCiAgICBvbmVzID0gdG9yY2gub25lcyh4X2Nhbi5zaGFwZVswXSwgMSwgZGV2aWNlPWRldiwgZHR5cGU9ZHR5cGUpCiAgICB4X3VuYWwgPSAodG9yY2guY2F0KFt4X2Nhbiwgb25lc10sIGRpbT0tMSkgQCB0ZFsiTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkIl0udG8oZGV2LCBkdHlwZSkpWzosIDozXQogICAgeF93b3JsZCA9IGZsb2F0KHRkWyJ3b3JsZF9hbGlnbl9zIl0pICogKHhfdW5hbCBAIHRkWyJ3b3JsZF9hbGlnbl9SIl0udG8oZGV2LCBkdHlwZSkpICsgdGRbIndvcmxkX2FsaWduX3QiXS50byhkZXYsIGR0eXBlKQogICAgcmV0dXJuICh4X3dvcmxkIC0gdGRbImNlbnRlciJdLnRvKGRldiwgZHR5cGUpKSAvIGZsb2F0KHRkWyJyYWRpdXMiXSkKCgpkZWYgdG9yY2hfbmV1c190b19jYW5vbmljYWwoeF9ub3JtLCBhbGlnbl9kaWN0KToKICAgIGlmIG5vdCBIQVNfVE9SQ0g6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaCByZXF1aXJlZCIpCiAgICBhID0gQWxpZ25TaW0zLmZyb21fbWFwcGluZyhhbGlnbl9kaWN0KQogICAgdGQgPSBhLmFzX3RvcmNoX2RpY3QoKQogICAgTV9pbnYgPSB0b3JjaC5saW5hbGcuaW52KHRkWyJNX2Nhbm9uaWNhbF90b19kYTNfd29ybGQiXSkKICAgIGRldiwgZHR5cGUgPSB4X25vcm0uZGV2aWNlLCB4X25vcm0uZHR5cGUKICAgIHhfd29ybGQgPSB4X25vcm0gKiBmbG9hdCh0ZFsicmFkaXVzIl0pICsgdGRbImNlbnRlciJdLnRvKGRldiwgZHR5cGUpCiAgICB4X3VuYWwgPSAoeF93b3JsZCAtIHRkWyJ3b3JsZF9hbGlnbl90Il0udG8oZGV2LCBkdHlwZSkpIEAgdGRbIndvcmxkX2FsaWduX1IiXS50byhkZXYsIGR0eXBlKS5UIC8gZmxvYXQodGRbIndvcmxkX2FsaWduX3MiXSkKICAgIG9uZXMgPSB0b3JjaC5vbmVzKHhfdW5hbC5zaGFwZVswXSwgMSwgZGV2aWNlPWRldiwgZHR5cGU9ZHR5cGUpCiAgICByZXR1cm4gKHRvcmNoLmNhdChbeF91bmFsLCBvbmVzXSwgZGltPS0xKSBAIE1faW52LnRvKGRldiwgZHR5cGUpKVs6LCA6M10K"""
_ALIGN_BYTES = base64.b64decode(_ALIGN_B64)
_ALIGN_DESTS = [Path("/content/gfhashba_align.py"), Path.cwd() / "gfhashba_align.py"]

_written = None
for _d in _ALIGN_DESTS:
    try:
        _d.parent.mkdir(parents=True, exist_ok=True)
        if (not _d.exists()) or (_d.stat().st_size != len(_ALIGN_BYTES)):
            _d.write_bytes(_ALIGN_BYTES)
        _written = _d
        break
    except Exception as _e:
        print("skip", _d, ":", _e)

if _written is None:
    raise RuntimeError("Could not write gfhashba_align.py")

_parent = str(_written.parent.resolve())
if _parent not in sys.path:
    sys.path.insert(0, _parent)

import gfhashba_align
importlib.reload(gfhashba_align)
from gfhashba_align import AlignSim3, sdf_volume_health, mesh_health  # noqa: F401
print("✓ gfhashba_align ready at", _written, f"({len(_ALIGN_BYTES)} bytes)")

## 3 — Load posed RGB + masks (+ optional depth)

Loads a BA-NeuS `cameras_sphere.npz` bundle when available so GF-HashBA starts
in the **same normalized unit-sphere frame** as `sam_baneus_ultrra.ipynb`.


In [ ]:
# 3.0 — Dataset helpers (NeuS cameras_sphere.npz or raw stub)
import json
import numpy as np
import torch
from pathlib import Path
from PIL import Image

def load_K_Rt_from_P(P):
    """IDR / BA-NeuS convention: P = K @ [R|t], 3x4."""
    import cv2
    P = np.asarray(P, np.float64)
    if P.shape == (4, 4):
        P = P[:3]
    P = P[:3, :4]
    K, R, t, _, _, _, _ = cv2.decomposeProjectionMatrix(P)
    K = K / K[2, 2]
    pose = np.eye(4, dtype=np.float64)
    pose[:3, :3] = R.T
    pose[:3, 3] = (t[:3] / t[3])[:, 0]   # homogeneous t from OpenCV
    return K[:3, :3].astype(np.float32), pose.astype(np.float32)

def _resolve_train_size(meta_npz=None, train_w=None, train_h=None):
    if train_w and train_h:
        return int(train_w), int(train_h)
    if meta_npz is not None and Path(meta_npz).exists():
        meta = np.load(meta_npz, allow_pickle=True)
        if "train_w" in meta.files and "train_h" in meta.files:
            return int(meta["train_w"]), int(meta["train_h"])
    return 768, 576   # sam_baneus va_split default

def _resize_to_train(img, mask, depth, K, train_w, train_h, scale_k=True):
    """Resize RGB/mask/depth to export resolution; optionally scale intrinsics."""
    import cv2
    h0, w0 = img.shape[:2]
    if (w0, h0) == (train_w, train_h):
        return img, mask, depth, K.astype(np.float32)
    img_u8 = (np.clip(img, 0.0, 1.0) * 255.0).astype(np.uint8)
    img_r = np.asarray(
        Image.fromarray(img_u8).resize((train_w, train_h), Image.LANCZOS), np.float32
    ) / 255.0
    mask_r = cv2.resize(mask.astype(np.float32), (train_w, train_h), interpolation=cv2.INTER_NEAREST)
    depth_r = None
    if depth is not None:
        depth_r = cv2.resize(depth.astype(np.float32), (train_w, train_h), interpolation=cv2.INTER_NEAREST)
    K_r = np.asarray(K, np.float32).copy()
    if scale_k:
        sx, sy = train_w / w0, train_h / h0
        K_r[0, 0] *= sx; K_r[1, 1] *= sy; K_r[0, 2] *= sx; K_r[1, 2] *= sy
    return img_r, mask_r, depth_r, K_r

def _glob_stem(directory, stem):
    directory = Path(directory)
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
        p = directory / f"{stem}{ext}"
        if p.exists():
            return p
    matches = sorted(directory.glob(f"{stem}.*"))
    return matches[0] if matches else None

def _read_view_names(cameras_npz, meta_npz=None, images_dir=None, da3_npz=None, n_views=None):
    if meta_npz is not None and Path(meta_npz).exists():
        meta = np.load(meta_npz, allow_pickle=True)
        if "view_names" in meta.files:
            return [str(x) for x in meta["view_names"]]
    txt = Path(cameras_npz).parent / "view_names.txt"
    if txt.exists():
        return [ln.strip() for ln in txt.read_text().splitlines() if ln.strip()]
    if da3_npz is not None and Path(da3_npz).exists() and n_views is not None:
        da3 = np.load(da3_npz, allow_pickle=True)
        stems = [Path(str(p)).stem for p in da3["image_files"]]
        if len(stems) == n_views:
            return stems
    if images_dir is not None and Path(images_dir).exists():
        files = sorted(list(Path(images_dir).glob("*.png")) + list(Path(images_dir).glob("*.jpg")))
        if n_views is None or len(files) == n_views:
            return [f.stem for f in files]
    return None

def load_neus_bundle(cameras_npz, images_dir, masks_dir=None, depth_dir=None,
                     meta_npz=None, view_names=None, da3_npz=None,
                     train_w=None, train_h=None):
    """Cameras from BA-NeuS npz; RGB/masks from SatelliteFuse (or views/)."""
    cameras_npz = Path(cameras_npz)
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir) if masks_dir else images_dir.parent / "masks"
    depth_dir = Path(depth_dir) if depth_dir else None
    train_w, train_h = _resolve_train_size(meta_npz, train_w, train_h)

    cam = np.load(cameras_npz)
    n = len([k for k in cam.files if k.startswith("world_mat_")])
    if view_names is None:
        view_names = _read_view_names(cameras_npz, meta_npz, images_dir, da3_npz, n)
    if view_names is None or len(view_names) != n:
        raise FileNotFoundError(
            f"Need {n} view_names matching cameras_sphere.npz; got "
            f"{len(view_names) if view_names else 0}. "
            f"Restore baneus_scene_meta.npz from sam_baneus or run §12 export."
        )

    images, masks, depths, Ks, c2ws = [], [], [], [], []
    for i, name in enumerate(view_names):
        P = cam[f"world_mat_{i}"] @ cam[f"scale_mat_{i}"]
        K, c2w = load_K_Rt_from_P(P)
        Ks.append(K); c2ws.append(c2w)

        ip = _glob_stem(images_dir, name)
        if ip is None:
            raise FileNotFoundError(
                f"Missing source image '{name}' under {images_dir} "
                f"(SatelliteFuse satellite/images, not baneus_ultrra_results)"
            )
        img = np.asarray(Image.open(ip).convert("RGB"), np.float32) / 255.0

        mp = _glob_stem(masks_dir, name)
        if mp is not None:
            mimg = Image.open(mp)
            if mimg.mode == "RGBA":
                m = (np.asarray(mimg)[..., 3] > 127).astype(np.float32)
            else:
                m = (np.asarray(mimg.convert("L")) > 127).astype(np.float32)
        else:
            m = np.ones(img.shape[:2], np.float32)
        if m.shape != img.shape[:2]:
            import cv2
            m = cv2.resize(m.astype(np.float32), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        dep = None
        dp = depth_dir / f"{i:03d}.npy" if depth_dir is not None else None
        if dp is not None and dp.exists():
            dep = np.load(dp).astype(np.float32)

        img, m, dep, K = _resize_to_train(img, m, dep, K, train_w, train_h, scale_k=False)
        images.append(img); masks.append(m); depths.append(dep); Ks[-1] = K

    if da3_npz is not None and Path(da3_npz).exists():
        da3 = np.load(da3_npz, allow_pickle=True)
        d_stems = [Path(str(p)).stem for p in da3["image_files"]]
        for i, s in enumerate(view_names):
            if depths[i] is not None or s not in d_stems:
                continue
            j = d_stems.index(s)
            dep = da3["depth"][j].astype(np.float32)
            _, _, dep, _ = _resize_to_train(images[i], masks[i], dep, Ks[i], train_w, train_h, scale_k=False)
            depths[i] = dep
        n_dep = sum(d is not None for d in depths)
        if n_dep:
            print(f"Depth: {n_dep}/{n} views (bundle + DA3 fallback)")

    print(f"Train resolution: {train_w}x{train_h} | views: {n}")
    return {
        "images": images, "masks": masks, "depths": depths,
        "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
        "H": images[0].shape[0], "W": images[0].shape[1], "n_views": n,
        "view_names": list(view_names),
        "scale_mat": cam["scale_mat_0"] if "scale_mat_0" in cam.files else np.eye(4),
    }

def load_raw_views(images_dir, masks_dir, da3_npz=None, max_views=None,
                   train_w=None, train_h=None):
    """Fallback: RGB+masks; poses from DA3 if present else identity stubs."""
    import cv2
    images_dir, masks_dir = Path(images_dir), Path(masks_dir)
    train_w, train_h = _resolve_train_size(None, train_w, train_h)
    files = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")))
    if max_views:
        files = files[:max_views]
    da3 = np.load(da3_npz, allow_pickle=True) if da3_npz is not None and Path(da3_npz).exists() else None
    d_stems = [Path(str(p)).stem for p in da3["image_files"]] if da3 is not None else []

    images, masks, depths, Ks, c2ws, stems = [], [], [], [], [], []
    for f in files:
        img = np.asarray(Image.open(f).convert("RGB"), np.float32) / 255.0
        H, W = img.shape[:2]
        mp = masks_dir / f"{f.stem}.png"
        if mp.exists():
            mimg = Image.open(mp)
            if mimg.mode == "RGBA":
                m = (np.asarray(mimg)[..., 3] > 127).astype(np.float32)
            else:
                m = (np.asarray(mimg.convert("L")) > 127).astype(np.float32)
        else:
            m = np.ones((H, W), np.float32)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.float32), (W, H), interpolation=cv2.INTER_NEAREST)
        K = np.array([[W, 0, W / 2], [0, W, H / 2], [0, 0, 1]], np.float32)
        c2w = np.eye(4, dtype=np.float32)
        dep = None
        if da3 is not None and f.stem in d_stems:
            j = d_stems.index(f.stem)
            E = da3["extrinsics"][j]
            w2c = np.eye(4, dtype=np.float64); w2c[:3] = E[:3]
            c2w = np.linalg.inv(w2c).astype(np.float32)
            K = da3["intrinsics"][j].astype(np.float32)
            dep = da3["depth"][j].astype(np.float32)
        img, m, dep, K = _resize_to_train(img, m, dep, K, train_w, train_h, scale_k=True)
        images.append(img); masks.append(m); depths.append(dep)
        Ks.append(K); c2ws.append(c2w); stems.append(f.stem)

    if da3 is not None:
        print(f"Loaded DA3 poses/depth for {sum(d is not None for d in depths)}/{len(files)} views")
    print(f"Train resolution: {train_w}x{train_h} | views: {len(images)}")
    return {"images": images, "masks": masks, "depths": depths,
            "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
            "H": train_h, "W": train_w, "n_views": len(images),
            "view_names": stems,
            "scale_mat": np.eye(4, dtype=np.float32)}

img_dir = VIEW_DIR / "images" if (VIEW_DIR / "images").exists() else AERIAL_DIR / IMG_SUBDIR
msk_dir = VIEW_DIR / "masks"  if (VIEW_DIR / "masks").exists()  else AERIAL_DIR / MASK_SUBDIR
_da3 = DA3_NPZ if Path(DA3_NPZ).exists() else None

if DATA_MODE == "neus_bundle" and CAMERAS_NPZ is None and not globals().get("ALLOW_SYNTHETIC_SCENE", False):
    raise FileNotFoundError(
        "neus_bundle mode: cameras_sphere.npz not found — check the Drive mount and SAM_BANEU_RUN "
        f"(§2 printed BANEUS_META_DIR={BANEUS_META_DIR}). Refusing to train on a synthetic stand-in scene."
    )
if DATA_MODE == "neus_bundle" and CAMERAS_NPZ is not None:
    scene = load_neus_bundle(
        CAMERAS_NPZ, img_dir, msk_dir,
        depth_dir=NEUS_DEPTH_DIR,
        meta_npz=SCENE_META_NPZ,
        da3_npz=_da3,
        train_w=TRAIN_W, train_h=TRAIN_H,
    )
    print(f"✓ NeuS cameras + source views: {scene['n_views']} @ {scene['W']}x{scene['H']}")
elif DATA_MODE == "raw_views" or CAMERAS_NPZ is None:
    if not img_dir.exists():
        print(f"⚠ No images at {img_dir} — creating synthetic smoke-test scene")
        # Tiny synthetic sphere scene for offline unit tests
        H = W = 128; n = 4
        images, masks, depths, Ks, c2ws = [], [], [], [], []
        for i in range(n):
            ang = 2 * np.pi * i / n
            c2w = np.eye(4, dtype=np.float32)
            c2w[:3, 3] = [2.5 * np.cos(ang), 0.3, 2.5 * np.sin(ang)]
            # look-at origin
            forward = -c2w[:3, 3]; forward /= np.linalg.norm(forward)
            right = np.cross(np.array([0.,1.,0.]), forward); right /= (np.linalg.norm(right)+1e-8)
            up = np.cross(forward, right)
            c2w[:3, 0] = right; c2w[:3, 1] = up; c2w[:3, 2] = forward
            img = np.full((H, W, 3), 0.7, np.float32)
            yy, xx = np.mgrid[0:H, 0:W]
            m = (((xx - W/2)**2 + (yy - H/2)**2) < (W*0.25)**2).astype(np.float32)
            img[m > 0] = [0.85, 0.55, 0.35]
            images.append(img); masks.append(m); depths.append(None)
            Ks.append(np.array([[W,0,W/2],[0,W,H/2],[0,0,1]], np.float32))
            c2ws.append(c2w)
        scene = {"images": images, "masks": masks, "depths": depths,
                 "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
                 "H": H, "W": W, "n_views": n, "scale_mat": np.eye(4, dtype=np.float32)}
        print(f"✓ Synthetic smoke scene: {n} views @ {W}x{H}")
    else:
        scene = load_raw_views(img_dir, msk_dir, _da3, train_w=TRAIN_W, train_h=TRAIN_H)
        if _da3 is None:
            print("⚠ raw_views without DA3 poses → identity cameras; geometry will be meaningless")
        print(f"✓ Raw views: {scene['n_views']} @ {scene['W']}x{scene['H']}")
else:
    raise FileNotFoundError(
        f"neus_bundle mode needs cameras_sphere.npz (run sam_baneus §12) "
        f"and images under {img_dir}"
    )

In [ ]:
# 3.1 — Preview a few views + masks
import matplotlib.pyplot as plt

n_show = min(4, scene["n_views"])
fig, axes = plt.subplots(2, n_show, figsize=(3.2 * n_show, 6))
if n_show == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for i in range(n_show):
    axes[0, i].imshow(scene["images"][i]); axes[0, i].set_title(f"view {i}"); axes[0, i].axis("off")
    axes[1, i].imshow(scene["masks"][i], cmap="gray"); axes[1, i].set_title("mask"); axes[1, i].axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "input_views.png", dpi=120, bbox_inches="tight")
plt.show()
to_drive(OUTPUT_DIR / "input_views.png")
print("c2w centers:", scene["c2ws"][:, :3, 3].round(3))

## 3.5 — Stage-1 occupancy teacher (NeuS-norm frame)

Loads `stage1_occupancy.npz` + `baneus_scene_meta.npz` from a prior `sam_baneus` run,
maps the 64³ canonical occupancy into the NeuS unit-sphere frame, and exposes:

- `sample_prob(x_norm)` / `pseudo_sdf_from_prob(p)` — continuous SDF teacher
- `OCC_GRID_N11` — dense occupancy on $[-1,1]^3$ for flow / prefit
- `STAGE1_COORDS` / `STAGE1_ALIGN` — same sparse coords + Sim3 used by Stage-2 densify


In [ ]:
# 3.5 — Load Stage-1 occupancy into NeuS-normalized frame
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
import shutil


def n11_grid_points(res, device="cpu"):
    """Grid points of a res^3 lattice on [-1,1]^3; reshaping values to (res,res,res) gives layout [z,y,x]."""
    lin = torch.linspace(-1, 1, res, device=device)
    zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
    return torch.stack([xx, yy, zz], dim=-1).reshape(-1, 3)


def grid_sample_n11(grid, pts):
    """Trilinear lookup of a [z,y,x] grid over NeuS [-1,1]^3 (align_corners=True; outside → 0)."""
    g = pts.reshape(1, -1, 1, 1, 3).to(device=grid.device, dtype=grid.dtype)
    v = F.grid_sample(grid[None, None], g, mode="bilinear", align_corners=True, padding_mode="zeros")
    return v.reshape(-1)


def solidify_occupancy(occ, thr=0.5, close_iters=2, smooth=True):
    """Binary closing + fill enclosed cavities. Returns (solid grid tensor, shell_frac, solid_frac)."""
    from scipy import ndimage
    a = occ.detach().float().cpu().numpy() if torch.is_tensor(occ) else np.asarray(occ, np.float32)
    b = a > thr
    if close_iters > 0:
        b = ndimage.binary_closing(b, iterations=int(close_iters))
    solid = ndimage.binary_fill_holes(b)
    out = solid.astype(np.float32)
    if smooth:
        out = ndimage.uniform_filter(out, size=3)
    return torch.from_numpy(out), float((a > thr).mean()), float(solid.mean())


# Restore stage1 from Drive if needed
if STAGE1_OCC_NPZ is None and BANEUS_META_DIR is not None and Path(BANEUS_META_DIR).exists():
    _src = BANEUS_META_DIR / "stage1"
    _dst = OUTPUT_DIR / "stage1"
    if _src.is_dir():
        _dst.mkdir(parents=True, exist_ok=True)
        for f in _src.iterdir():
            if f.is_file() and not (_dst / f.name).exists():
                shutil.copy2(f, _dst / f.name)
                print("restored", f.name)
    STAGE1_OCC_NPZ = _first_existing(_dst / "stage1_occupancy.npz", _src / "stage1_occupancy.npz")
    STAGE1_FRAME_NPZ = _first_existing(_dst / "stage1_frame.npz", _src / "stage1_frame.npz")

HAS_STAGE1 = STAGE1_OCC_NPZ is not None and Path(STAGE1_OCC_NPZ).exists()
OCC_GRID_N11 = None    # Stage-1 shell occupancy, 64^3 on NeuS [-1,1]^3 (diagnostics)
OCC_SOLID_N11 = None   # solidified Stage-1 occupancy, PRIOR_GRID_RES^3 (prefit target / fallback prior)
STAGE1_COORDS = None   # exported sparse coords (N,4) in the 64^3 lattice; used by Stage-2
STAGE1_ALIGN = None    # Sim3 + NeuS sphere params shared with Stage-2

if not HAS_STAGE1:
    print("⚠ No stage1_occupancy.npz — prefit/occupancy prior disabled. Run sam_baneus §9 first.")

    def sample_prob(x_norm):
        return torch.zeros(x_norm.shape[0], device=x_norm.device)

    def pseudo_sdf_from_prob(p):
        return p * 0.0

    def canonical_to_neus_norm(x_can):
        raise RuntimeError("no Stage-1 alignment — run §3.5 with stage1_occupancy + baneus_scene_meta")
else:
    occ = np.load(STAGE1_OCC_NPZ)
    prob_can = torch.from_numpy(np.asarray(occ["prob"], np.float32))  # (G,G,G) canonical, prob[i,j,k]
    _G = int(prob_can.shape[0])
    if "coords" in occ.files:
        STAGE1_COORDS = torch.from_numpy(np.asarray(occ["coords"], np.int32))  # spconv needs int32
        print(f"✓ exported Stage-1 coords {tuple(STAGE1_COORDS.shape)} (reuse for Stage-2)")
    else:
        print("⚠ stage1_occupancy.npz has no 'coords' — Stage-2 will re-run sparse structure")
    if SCENE_META_NPZ is None:
        raise FileNotFoundError("baneus_scene_meta.npz required to map Stage-1 → NeuS norm")
    meta = np.load(SCENE_META_NPZ, allow_pickle=True)
    center = torch.tensor(np.asarray(meta["center"], np.float32))
    radius = float(meta["radius"])
    _M = np.asarray(
        meta["M_canonical_to_da3_world"] if "M_canonical_to_da3_world" in meta.files else np.eye(4),
        np.float32,
    )
    _M_inv = torch.tensor(np.linalg.inv(_M), dtype=torch.float32)
    _M_fwd = torch.tensor(_M, dtype=torch.float32)
    _was = float(meta["world_align_s"] if "world_align_s" in meta.files else 1.0)
    _waR = torch.tensor(
        np.asarray(meta["world_align_R"] if "world_align_R" in meta.files else np.eye(3), np.float32)
    )
    _wat = torch.tensor(
        np.asarray(meta["world_align_t"] if "world_align_t" in meta.files else np.zeros(3), np.float32)
    )
    _prob = prob_can
    STAGE1_ALIGN = {
        "M_canonical_to_da3_world": _M_fwd,
        "world_align_s": _was,
        "world_align_R": _waR,
        "world_align_t": _wat,
        "center": center,
        "radius": radius,
        "voxel_grid": _G,
    }

    def neus_norm_to_canonical(x_norm):
        dev, dtype = x_norm.device, x_norm.dtype
        x_world = x_norm * radius + center.to(dev, dtype)
        x_world = (x_world - _wat.to(dev, dtype)) @ _waR.to(dev, dtype).T / _was
        ones = torch.ones(x_world.shape[0], 1, device=dev, dtype=dtype)
        return (torch.cat([x_world, ones], dim=-1) @ _M_inv.to(dev, dtype))[:, :3]

    def canonical_to_neus_norm(x_can):
        """Stage-1 canonical [-0.5,0.5]^3 → NeuS unit-sphere (inverse of neus_norm_to_canonical)."""
        dev, dtype = x_can.device, x_can.dtype
        ones = torch.ones(x_can.shape[0], 1, device=dev, dtype=dtype)
        x_unal = (torch.cat([x_can, ones], dim=-1) @ _M_fwd.to(device=dev, dtype=dtype))[:, :3]
        x_world = float(_was) * (x_unal @ _waR.to(device=dev, dtype=dtype)) + _wat.to(device=dev, dtype=dtype)
        return (x_world - center.to(device=dev, dtype=dtype)) / float(radius)

    @torch.no_grad()
    def sample_prob(x_norm):
        # canonical c = idx/G − 0.5 (same convention as coords/64 − 0.5 and the Stage-1 mesh),
        # so the align_corners=True sample coordinate is 2·idx/(G−1) − 1.
        idx = (neus_norm_to_canonical(x_norm) + 0.5) * _G
        g = 2.0 * idx / (_G - 1) - 1.0
        g = g[:, [2, 1, 0]].reshape(1, -1, 1, 1, 3)        # prob[i,j,k] ↔ (D,H,W) ↔ grid (x=k, y=j, z=i)
        v = F.grid_sample(
            _prob.to(x_norm.device)[None, None], g.to(_prob.dtype), align_corners=True, padding_mode="zeros"
        )
        return v.reshape(-1)

    def pseudo_sdf_from_prob(p):
        return (0.5 - p) * 0.2

    def _eval_on_grid(fn, res, bs=65536):
        _dev = torch.device(DEVICE)
        pts = n11_grid_points(res, _dev)
        vals = [fn(pts[i:i + bs]) for i in range(0, pts.shape[0], bs)]
        return torch.cat(vals).reshape(res, res, res).float().cpu()

    OCC_GRID_N11 = _eval_on_grid(sample_prob, 64)
    print(
        f"✓ Stage-1 occupancy → NeuS grid 64³ | occ mean={float(OCC_GRID_N11.mean()):.3f} "
        f"| frac>0.5={float((OCC_GRID_N11 > 0.5).float().mean()):.3f}"
    )
    print(f"  center={center.numpy().round(3)} radius={radius:.3f}")

    _occ_hi = _eval_on_grid(sample_prob, PRIOR_GRID_RES)
    OCC_SOLID_N11, _shell_frac, _solid_frac = solidify_occupancy(_occ_hi, thr=0.5, close_iters=2)
    print(f"  solidified Stage-1 @ {PRIOR_GRID_RES}³: shell frac={_shell_frac:.4f} → solid frac={_solid_frac:.4f} "
          f"(×{_solid_frac / max(_shell_frac, 1e-8):.1f})")
    if _solid_frac < 1.3 * _shell_frac:
        print("  ⚠ fill barely grew the volume — the Stage-1 shell is open; the target stays shell-like")

    # Round-trip sanity: voxel centers → NeuS → back should land near export
    if STAGE1_COORDS is not None:
        _c = STAGE1_COORDS
        _xyz = _c[:, 1:4].float() if _c.shape[-1] == 4 else _c[:, :3].float()
        _xcan = _xyz / float(_G) - 0.5
        _xn = canonical_to_neus_norm(_xcan)
        _back = neus_norm_to_canonical(_xn)
        _err = (_back - _xcan).norm(dim=-1).mean().item()
        print(f"  Sim3 round-trip mean err={_err:.2e} (should be ~1e-6)")
        print(f"  exported voxels in NeuS: mean={_xn.mean(0).numpy().round(3)} "
              f"std={_xn.std(0).numpy().round(3)} "
              f"|in|<1 frac={float((_xn.abs() < 1).all(-1).float().mean()):.3f}")


def sample_prefit_occ(x_norm):
    """Prefit teacher: solidified Stage-1 occupancy (or the raw shell if PREFIT_SOLIDIFY=False)."""
    if PREFIT_SOLIDIFY and OCC_SOLID_N11 is not None:
        return grid_sample_n11(OCC_SOLID_N11.to(x_norm.device), x_norm)
    return sample_prob(x_norm)

### 3.5b — Alignment diagnostics (visual + quantitative)

**Run after §3.5.** Local unit tests live in `tests/test_gfhashba_alignment.py`
(`python tests/test_gfhashba_alignment.py`).

This cell checks:
1. Sim3 round-trip on exported Stage-1 coords
2. Voxel → NeuS soft-splat vs `OCC_GRID_N11` (IoU / corr) — must **not** look like AABB fill
3. Mid-slice overlays (save `align_diag_s1.png`)

In [ ]:
# 3.5b — Alignment diagnostics (quant + visual)
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# Prefer shared module (upload gfhashba_align.py next to the notebook / on PYTHONPATH)
_repo_candidates = [
    Path("/content/GSV-MSV"),
    Path("/content"),
    Path.cwd(),
    Path("/Users/taiganishida/Documents/GSV-MSV"),
]
for _r in _repo_candidates:
    if (_r / "gfhashba_align.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))

try:
    from gfhashba_align import (
        AlignSim3,
        aabb_stretch_to_grid,
        grid_corr,
        grid_iou,
        occ_bbox_from_grid,
        sim3_soft_splat_to_grid,
        voxel_idx_to_neus,
    )
    _HAS_ALIGN_LIB = True
except Exception as _e:
    _HAS_ALIGN_LIB = False
    # Colab: upload gfhashba_align.py or place it on Drive
    import shutil
    for _cand in (
        Path("/content/drive/MyDrive/SatelliteFuse/GFhashBA/gfhashba_align.py"),
        Path("/content/drive/MyDrive/SatelliteFuse/gfhashba_align.py"),
        Path("/content/GSV-MSV/gfhashba_align.py"),
    ):
        if _cand.exists():
            shutil.copy2(_cand, Path("/content/gfhashba_align.py"))
            if "/content" not in sys.path:
                sys.path.insert(0, "/content")
            try:
                from gfhashba_align import (
                    AlignSim3, aabb_stretch_to_grid, grid_corr, grid_iou,
                    occ_bbox_from_grid, sim3_soft_splat_to_grid, voxel_idx_to_neus,
                )
                _HAS_ALIGN_LIB = True
                print("✓ loaded gfhashba_align from", _cand)
                break
            except Exception as _e2:
                print("still failing after copy:", _e2)
    if not _HAS_ALIGN_LIB:
        print("⚠ gfhashba_align not importable — using notebook-local fallbacks:", _e)
        print("  Upload gfhashba_align.py to /content/ for full AABB-negative tests.")

assert HAS_STAGE1 and STAGE1_ALIGN is not None and OCC_GRID_N11 is not None, "Run §3.5 first"
assert STAGE1_COORDS is not None, "stage1_occupancy.npz needs 'coords' for Stage-2 align tests"

# ---- 1) Sim3 round-trip ----
_c = STAGE1_COORDS
_xyz = _c[:, 1:4].float() if _c.shape[-1] == 4 else _c[:, :3].float()
_xcan = _xyz / float(STAGE1_ALIGN["voxel_grid"]) - 0.5
_xn = canonical_to_neus_norm(_xcan)
_back = neus_norm_to_canonical(_xn)
_rt_err = float((_back - _xcan).norm(dim=-1).mean())
_in_frac = float((_xn.abs() < 1.0).all(dim=-1).float().mean())
print(f"[1] Sim3 round-trip mean err={_rt_err:.2e}  | voxels |x|<1 frac={_in_frac:.3f}")
assert _rt_err < 1e-5, f"Sim3 round-trip broken: {_rt_err}"
assert _in_frac > 0.5, f"Too many Stage-1 voxels outside NeuS ball ({_in_frac:.3f})"

# ---- 2) densify vs OCC (Sim3) + negative AABB check ----
_occ = OCC_GRID_N11.detach().float().cpu().numpy()
_coords_np = _c.detach().cpu().numpy()

if _HAS_ALIGN_LIB:
    _align = AlignSim3.from_mapping(STAGE1_ALIGN)
    _dens = sim3_soft_splat_to_grid(_coords_np, None, _align, res=_occ.shape[0])
    # AABB stretch of the same coords (should score worse vs OCC)
    _aabb_idx = aabb_stretch_to_grid(_coords_np, res=_occ.shape[0])
    _aabb = np.zeros_like(_occ)
    _aabb[_aabb_idx[:, 2], _aabb_idx[:, 1], _aabb_idx[:, 0]] = 1.0
else:
    # fallback: splat via notebook canonical_to_neus_norm
    _res = _occ.shape[0]
    _dens = np.zeros((_res, _res, _res), np.float64)
    _g = ((_xn.cpu().numpy() + 1.0) * 0.5 * (_res - 1))
    for p in _g:
        ix, iy, iz = [int(np.clip(round(v), 0, _res - 1)) for v in p]
        _dens[iz, iy, ix] += 1.0
    if _dens.max() > 0:
        _dens /= _dens.max()
    _aabb = None

_iou = float(((_dens > 0.1) & (_occ > 0.5)).sum() / max(((_dens > 0.1) | (_occ > 0.5)).sum(), 1))
# pearson
_dn = (_dens - _dens.mean()) / (_dens.std() + 1e-8)
_on = (_occ - _occ.mean()) / (_occ.std() + 1e-8)
_corr = float((_dn * _on).mean())
print(f"[2] Sim3 densify vs OCC_GRID_N11: IoU={_iou:.3f}  corr={_corr:.3f}")
if _iou < 0.05:
    raise RuntimeError(
        f"Align FAIL: IoU={_iou:.3f} — Stage-1 coords do not land on OCC in NeuS. "
        "Check baneus_scene_meta / pinned SamBAneus run."
    )
if _aabb is not None:
    _iou_aabb = float(((_aabb > 0.5) & (_occ > 0.5)).sum() / max(((_aabb > 0.5) | (_occ > 0.5)).sum(), 1))
    print(f"    AABB-stretch vs OCC (should be worse): IoU={_iou_aabb:.3f}")
    if _iou_aabb > _iou + 0.05:
        print("⚠ AABB scored higher than Sim3 — suspicious; inspect overlays")

# ---- 3) visual mid-slices ----
_mid = _occ.shape[0] // 2
fig, axs = plt.subplots(2, 3, figsize=(11, 7))
for ax, img, title in (
    (axs[0, 0], _dens[_mid], "Sim3 dens (z mid)"),
    (axs[0, 1], _occ[_mid], "OCC NeuS (z mid)"),
    (axs[0, 2], _dens[_mid] * _occ[_mid], "product"),
    (axs[1, 0], _dens[:, _mid, :], "Sim3 dens (y mid)"),
    (axs[1, 1], _occ[:, _mid, :], "OCC NeuS (y mid)"),
    (axs[1, 2], (_aabb[_mid] if _aabb is not None else np.zeros_like(_occ[_mid])), "AABB dens (z mid) BAD"),
):
    ax.imshow(img, origin="lower", cmap="magma")
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"Align diag | IoU={_iou:.3f} corr={_corr:.3f} | rt_err={_rt_err:.1e}")
fig.tight_layout()
_png = OUTPUT_DIR / "align_diag_s1.png"
fig.savefig(_png, dpi=130, bbox_inches="tight")
plt.show()
print("✓ saved", _png)
print("PASS §3.5b alignment gates" if _iou >= 0.05 and _rt_err < 1e-5 else "FAIL")

# ---- paste-friendly summary for chat debug ----
_snap = {
    "section": "3.5b",
    "sim3_roundtrip_err": _rt_err,
    "voxels_in_neus_frac": _in_frac,
    "densify_vs_occ_iou": _iou,
    "densify_vs_occ_corr": _corr,
    "n_stage1_coords": int(_coords_np.shape[0]),
    "occ_mean": float(_occ.mean()),
    "occ_frac_gt_0.5": float((_occ > 0.5).mean()),
    "neus_center": [float(x) for x in (STAGE1_ALIGN["center"].detach().cpu().numpy().ravel() if hasattr(STAGE1_ALIGN["center"], "detach") else np.asarray(STAGE1_ALIGN["center"]).ravel())],
    "neus_radius": float(STAGE1_ALIGN["radius"]),
    "FLOW_PRIOR_ON_VISIBLE": bool(globals().get("FLOW_PRIOR_ON_VISIBLE", False)),
    "W_FLOW_START": float(globals().get("W_FLOW_START", -1)),
    "FLOW_HOLD_FRAC": float(globals().get("FLOW_HOLD_FRAC", -1)),
    "SAM_BANEU_RUN": str(globals().get("SAM_BANEU_RUN")),
    "RUN_TS": str(globals().get("RUN_TS")),
}
import json as _json
_snap_path = OUTPUT_DIR / "debug_snapshot_3_5b.json"
_snap_path.write_text(_json.dumps(_snap, indent=2))
print("=== DEBUG SNAPSHOT §3.5b (paste this block) ===")
print(_json.dumps(_snap, indent=2))
print("=== END SNAPSHOT ===")
print("also wrote", _snap_path)
to_drive(_png, _snap_path)

## 4 — Core modules (the four components)

Implemented as pure PyTorch so the notebook is self-contained. If `tinycudann`
is installed, `HashGridEncoding` will prefer it; otherwise we use a compatible
multi-resolution hash fallback.


### 4.1 — Multi-resolution Hash encoding + Coarse-to-Fine Annealing (component 4)

Hash grids learn high frequency immediately — fatal for early pose BA.
We introduce a continuous capacity $\gamma(k)\in[1,L]$ and weight level $l$:

$$
W_l(k)=\tfrac12\Bigl(1-\cos\bigl(\pi\cdot\mathrm{clamp}(\gamma(k)-l,\,0,\,1)\bigr)\Bigr)
$$

$\gamma$ ramps $1\to L$ over the first `ANNEAL_ITERS` steps, so the Hash-SDF
behaves like a low-frequency MLP until cameras lock.


In [ ]:
# 4.1 — Hash grid encoding + annealing
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import tinycudann as tcnn
    HAS_TCNN = True
except Exception:
    HAS_TCNN = False
print(f"tinycudann available: {HAS_TCNN}")


def hash_anneal_weights(n_levels: int, gamma: float, device=None, dtype=torch.float32):
    """W_l = 0.5 * (1 - cos(pi * clamp(gamma - l, 0, 1))) for l = 0..L-1."""
    levels = torch.arange(n_levels, device=device, dtype=dtype)
    t = (gamma - levels).clamp(0.0, 1.0)
    return 0.5 * (1.0 - torch.cos(math.pi * t))


def anneal_gamma(iteration: int, n_levels: int, anneal_iters: int, start: float = 1.0) -> float:
    if anneal_iters <= 0:
        return float(n_levels)
    t = min(max(iteration / float(anneal_iters), 0.0), 1.0)
    return float(start + t * (n_levels - start))


class HashGridEncoding(nn.Module):
    """Multi-res hash grid. TCNN if available, else a vectorized pure-PyTorch fallback.

    The fallback keeps all levels in one table (L·T, F) and gathers the 8 trilinear corners
    of every level in a single indexing op.
    """

    _P1 = 2654435761
    _P2 = 805459861

    def __init__(
        self,
        n_levels=16,
        n_features_per_level=2,
        log2_hashmap_size=19,
        base_resolution=16,
        max_resolution=2048,
        dtype=torch.float32,
    ):
        super().__init__()
        self.n_levels = n_levels
        self.n_features = n_features_per_level
        self.out_dim = n_levels * n_features_per_level
        self.base_resolution = base_resolution
        self.max_resolution = max_resolution
        self.log2_hashmap_size = log2_hashmap_size
        b = math.exp((math.log(max_resolution) - math.log(base_resolution)) / max(n_levels - 1, 1))
        self.per_level_scale = b
        self.register_buffer(
            "level_res",
            torch.tensor([base_resolution * (b ** l) for l in range(n_levels)], dtype=torch.float32),
            persistent=False,
        )

        if HAS_TCNN:
            cfg = {
                "otype": "HashGrid",
                "n_levels": n_levels,
                "n_features_per_level": n_features_per_level,
                "log2_hashmap_size": log2_hashmap_size,
                "base_resolution": base_resolution,
                "per_level_scale": b,
            }
            self.enc = tcnn.Encoding(3, cfg)
            self.backend = "tcnn"
        else:
            T = 1 << log2_hashmap_size
            self.hashmap_size = T
            self.table = nn.Parameter(torch.empty(n_levels * T, n_features_per_level, dtype=dtype))
            nn.init.uniform_(self.table, -1e-4, 1e-4)
            self.register_buffer(
                "level_offset", torch.arange(n_levels, dtype=torch.int64) * T, persistent=False
            )
            self.backend = "torch"

    def forward(self, x, gamma: float | None = None):
        """
        x: (N, 3) in roughly [-1, 1] (mapped to [0, 1] for hashing).
        gamma: if set, apply coarse-to-fine level weights.
        """
        if self.backend == "tcnn":
            x01 = (x.clamp(-1, 1) + 1.0) * 0.5
            feat = self.enc(x01.float()).to(dtype=x.dtype)
            feat = feat.view(-1, self.n_levels, self.n_features)
        else:
            x01 = ((x.clamp(-1, 1) + 1.0) * 0.5).float()
            xs = x01[:, None, :] * self.level_res[None, :, None]      # (N, L, 3)
            x0f = xs.floor()
            w1 = xs - x0f                                              # trilinear weights (grad → x)
            w0 = 1.0 - w1
            x0 = x0f.long()
            hx = (x0[..., 0], x0[..., 0] + 1)
            hy = (x0[..., 1] * self._P1, x0[..., 1] * self._P1 + self._P1)
            hz = (x0[..., 2] * self._P2, x0[..., 2] * self._P2 + self._P2)
            idx, wts = [], []
            for dx in (0, 1):
                wx = w1[..., 0] if dx else w0[..., 0]
                for dy in (0, 1):
                    wxy = wx * (w1[..., 1] if dy else w0[..., 1])
                    hxy = torch.bitwise_xor(hx[dx], hy[dy])
                    for dz in (0, 1):
                        idx.append(torch.bitwise_xor(hxy, hz[dz]))
                        wts.append(wxy * (w1[..., 2] if dz else w0[..., 2]))
            idx = torch.stack(idx, dim=-1) % self.hashmap_size + self.level_offset[None, :, None]  # (N, L, 8)
            wts = torch.stack(wts, dim=-1)                                                          # (N, L, 8)
            feat = (self.table[idx] * wts[..., None]).sum(dim=2)                                    # (N, L, F)

        if gamma is not None:
            W = hash_anneal_weights(self.n_levels, gamma, device=feat.device, dtype=feat.dtype)
            feat = feat * W.view(1, -1, 1)
        return feat.reshape(feat.shape[0], -1)


# quick anneal curve check
_g = [anneal_gamma(i, 16, 5000) for i in (0, 1000, 2500, 5000, 10000)]
print("gamma@iters", _g)
print("W@γ=3.5:", hash_anneal_weights(8, 3.5).numpy().round(3))


def flow_prior_schedule(
    iteration: int,
    anneal_iters: int | None = None,
    w_start: float | None = None,
    w_end: float | None = None,
    hold_frac: float | None = None,
):
    """Stage-2 prior weight over main train (after the Stage-1 prefit).

      1) HOLD W_FLOW_START for FLOW_HOLD_FRAC — the prior leads while photo is ill-behaved.
      2) COSINE decay to W_FLOW_END — photo owns fidelity once near a good basin.
      3) Floor W_FLOW_END > 0 — unseen regions keep a prior late (they have no photo signal).
    Spatial gating (1 − V) happens inside the loss, so the returned gate is always 1.
    """
    A = int(anneal_iters if anneal_iters is not None else FLOW_ANNEAL_ITERS)
    ws = float(w_start if w_start is not None else W_FLOW_START)
    we = float(w_end if w_end is not None else W_FLOW_END)
    hf = float(hold_frac if hold_frac is not None else FLOW_HOLD_FRAC)
    hf = min(max(hf, 0.0), 0.95)
    hold_end = int(hf * max(1, A))
    if iteration <= hold_end:
        w_flow = ws
    else:
        span = max(1, A - hold_end)
        t = min(1.0, float(iteration - hold_end) / span)
        w_flow = we + 0.5 * (ws - we) * (1.0 + math.cos(math.pi * t))
    return float(w_flow), 1.0


def beta_floor_schedule(iteration: int) -> float:
    """Linear ramp BETA_FLOOR_START → BETA_FLOOR_END over BETA_FLOOR_FRAC of training, then hold."""
    t = min(1.0, float(iteration) / max(1.0, BETA_FLOOR_FRAC * END_ITER))
    return float(BETA_FLOOR_START + t * (BETA_FLOOR_END - BETA_FLOOR_START))


def plot_flow_prior_schedule(save_path=None, end_iter=None, anneal_iters=None):
    """Graph hold-then-cosine W_flow vs iteration."""
    import matplotlib.pyplot as plt
    import numpy as np
    E = int(end_iter if end_iter is not None else END_ITER)
    A = int(anneal_iters if anneal_iters is not None else FLOW_ANNEAL_ITERS)
    xs = np.arange(0, E, max(1, E // 2000))
    ws = [flow_prior_schedule(int(it), anneal_iters=A)[0] for it in xs]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(xs, ws, lw=2, label=r"$W_{\mathrm{flow}}(t)$ (hold → cosine)")
    hold_it = int(float(FLOW_HOLD_FRAC) * A)
    ax.axvline(hold_it, color="C2", ls=":", lw=1.5, label=f"hold end ({FLOW_HOLD_FRAC:.0%})")
    ax.set_xlabel("iteration")
    ax.set_ylabel("prior weight")
    ax.set_title("Stage-2 prior on blind spots: strong early hold → cosine handoff to photo")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if save_path is None:
        save_path = OUTPUT_DIR / "flow_prior_schedule.png"
    fig.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Saved", save_path)
    return save_path

### 4.2 — Hash-SDF + Background NeRF + Scene Contraction (component 3)

Foreground building lives in normalized $[-1,1]^3$. Outside is warped:

$$
M_{\mathrm{contract}}(\mathbf{x})=\begin{cases}
\mathbf{x} & \|\mathbf{x}\|\le 1\\
\bigl(2-\frac{1}{\|\mathbf{x}\|}\bigr)\frac{\mathbf{x}}{\|\mathbf{x}\|} & \|\mathbf{x}\|>1
\end{cases}
$$

Mask BCE forces foreground opacity $O_{\mathrm{fg}}$ to match SAM masks so the
background field owns sky/trees.


In [ ]:
# 4.2 — Contraction, Hash-SDF, background NeRF, color head
import torch
import torch.nn as nn
import torch.nn.functional as F


def contract(x, ord_p: str = "inf"):
    """Mip-NeRF 360-style contraction. x: (N, 3)."""
    if ord_p == "inf":
        mag = x.abs().amax(dim=-1, keepdim=True)
    else:
        mag = torch.linalg.norm(x, dim=-1, keepdim=True)
    inside = mag <= 1.0
    # (2 - 1/||x||) * x/||x||
    x_out = (2.0 - 1.0 / mag.clamp_min(1e-6)) * (x / mag.clamp_min(1e-6))
    return torch.where(inside, x, x_out)


def sdf_to_density(sdf, alpha, beta):
    """σ(x) = α · sigmoid(-β · f(x))  — NeuS / VolSDF-style bridge."""
    return alpha * torch.sigmoid(-beta * sdf)


class HashSDF(nn.Module):
    def __init__(self, encoding: HashGridEncoding, hidden=64, n_layers=2, geo_init_bias=0.5):
        super().__init__()
        self.encoding = encoding
        layers = []
        in_d = encoding.out_dim
        for i in range(n_layers):
            layers += [nn.Linear(in_d if i == 0 else hidden, hidden), nn.ReLU(inplace=True)]
        self.mlp = nn.Sequential(*layers)
        self.sdf_out = nn.Linear(hidden, 1)
        self.feat_out = nn.Linear(hidden, hidden)
        # geometric init: sphere-ish bias
        nn.init.normal_(self.sdf_out.weight, 0, 1e-4)
        nn.init.constant_(self.sdf_out.bias, geo_init_bias)
        # learnable Laplace/sigmoid sharpness for density conversion
        self.log_beta = nn.Parameter(torch.tensor(1.6094))   # β≈5
        self.log_alpha = nn.Parameter(torch.tensor(0.0))     # α≈1
        self.beta_floor = None

    @property
    def beta(self):
        b = self.log_beta.exp()
        if self.beta_floor is not None:
            b = torch.clamp(b, min=float(self.beta_floor))   # floor set per-iteration by §6.1
        return b.clamp(0.1, 200.0)

    @property
    def alpha(self):
        return self.log_alpha.exp().clamp(0.1, 50.0)

    def forward(self, x, gamma=None):
        h = self.encoding(x, gamma=gamma)
        h = self.mlp(h)
        sdf = self.sdf_out(h)
        feat = self.feat_out(h)
        return sdf, feat

    def density(self, x, gamma=None):
        sdf, feat = self.forward(x, gamma=gamma)
        return sdf_to_density(sdf, self.alpha, self.beta), sdf, feat


class BackgroundNeRF(nn.Module):
    """Tiny MLP on contracted coords for sky / distant clutter."""

    def __init__(self, hidden=64, n_layers=4, n_freq=6):
        super().__init__()
        self.n_freq = n_freq
        in_d = 3 + 3 * 2 * n_freq
        layers = [nn.Linear(in_d, hidden), nn.ReLU(inplace=True)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        self.mlp = nn.Sequential(*layers)
        self.sigma = nn.Linear(hidden, 1)
        self.rgb = nn.Linear(hidden, 3)

    def pe(self, x):
        out = [x]
        for i in range(self.n_freq):
            out += [torch.sin((2 ** i) * math.pi * x), torch.cos((2 ** i) * math.pi * x)]
        return torch.cat(out, dim=-1)

    def forward(self, x):
        xc = contract(x)
        h = self.mlp(self.pe(xc))
        return F.relu(self.sigma(h)), torch.sigmoid(self.rgb(h))


class ColorNet(nn.Module):
    def __init__(self, feat_dim=64, hidden=64, n_freq=4):
        super().__init__()
        in_d = feat_dim + 3 + 3 + 3 + 3 * 2 * n_freq  # feat + x + normal + dir + dir PE
        self.n_freq = n_freq
        self.net = nn.Sequential(
            nn.Linear(in_d, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 3),
        )

    def pe(self, d):
        out = [d]
        for i in range(self.n_freq):
            out += [torch.sin((2 ** i) * math.pi * d), torch.cos((2 ** i) * math.pi * d)]
        return torch.cat(out, dim=-1)

    def forward(self, x, n, d, feat):
        inp = torch.cat([feat, x, n, self.pe(d)], dim=-1)
        return torch.sigmoid(self.net(inp))


print("✓ HashSDF / BackgroundNeRF / ColorNet defined")

### 4.3 — Generative prior (component 1)

The teacher is a dense occupancy grid on NeuS $[-1,1]^3$ (built once in §6.0a from the decoded MV-SAM3D Stage-2 geometry). On random points $\mathbf{x}$ in the unit ball:

$$
\mathcal{L}_{\mathrm{prior}}=\frac{1}{N}\sum_{\mathbf{x}}\bigl(1-\mathrm{sg}\,V(\mathbf{x})\bigr)\,\mathrm{BCE}\bigl(\sigma(-f_\theta(\mathbf{x})/\tau),\;o_{\mathrm{teacher}}(\mathbf{x})\bigr)
$$

BCE on logits keeps gradients for wrong-sign points (MSE on a sigmoid vanishes there). Normalising by $N$ makes the prior's total pull scale with how much of the scene is unobserved.

Modes: `"stage2"` (default), `"occupancy"` (Stage-1 grid), `"stub"` (smoke tests), `"off"`.

In [ ]:
# 4.3 — Generative prior: dense teacher occupancy + visibility-gated distillation
import torch
import torch.nn as nn
import torch.nn.functional as F


class DensePriorTarget(nn.Module):
    """Teacher occupancy on NeuS [-1,1]^3 (layout [z,y,x], trilinear lookup)."""

    def __init__(self, occ_grid: torch.Tensor, source: str = "stage1", meta: dict | None = None):
        super().__init__()
        assert occ_grid.ndim == 3, occ_grid.shape
        self.register_buffer("occ", occ_grid.float().clamp(0, 1))
        self.source = str(source)
        self.meta = dict(meta or {})

    @torch.no_grad()
    def sample(self, pts):
        return grid_sample_n11(self.occ, pts)

    def summary(self):
        return {
            "source": self.source,
            "res": int(self.occ.shape[0]),
            "occ_frac": float((self.occ > 0.5).float().mean()),
            **self.meta,
        }


def stub_prior_grid(res=64, device="cpu"):
    """DEV ONLY — ellipsoid teacher for smoke tests."""
    p = n11_grid_points(res, device)
    r2 = (p[:, 0] / 0.5) ** 2 + (p[:, 1] / 0.6) ** 2 + (p[:, 2] / 0.5) ** 2
    return (r2 < 1.0).float().reshape(res, res, res)


def sample_ball(n, device, radius=1.0):
    d = torch.randn(n, 3, device=device)
    d = d / d.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return d * (torch.rand(n, 1, device=device) ** (1.0 / 3.0)) * radius


def prior_distillation_loss(sdf_net, prior, pts, gamma=None, vis=None, tau=0.02, on_visible=False):
    """BCE between the Hash-SDF occupancy σ(−f/τ) and the teacher occupancy, weighted by 1 − sg(V).

    Normalised by the number of points, so the prior's total pull scales with how much of the
    scene is unobserved. BCE-with-logits keeps gradients alive for wrong-sign points.
    """
    with torch.no_grad():
        tgt = prior.sample(pts).clamp(0, 1)
        if vis is None or on_visible:
            w = torch.ones_like(tgt)
        else:
            w = 1.0 - vis.detach().clamp(0, 1)
    sdf, _ = sdf_net(pts, gamma=gamma)
    logits = -sdf.squeeze(-1) / float(tau)
    bce = F.binary_cross_entropy_with_logits(logits, tgt, reduction="none")
    loss = (w * bce).sum() / max(1, pts.shape[0])
    with torch.no_grad():
        inside_t = tgt > 0.5
        stats = {
            "w_mean": float(w.mean()),
            "unseen_in_target": float(w[inside_t].mean()) if bool(inside_t.any()) else float("nan"),
            "agree": float(((logits > 0) == inside_t).float().mean()),
        }
    return loss, stats


def build_flow_prior(mode: str = "stage2", occ_solid=None, stage2_prior=None):
    """'stage2' → §6.0a decoded Stage-2 target | 'occupancy'/'stage1' → solidified Stage-1 | 'stub' | 'off'."""
    mode = (mode or "off").lower()
    if mode in ("off", "none", "false"):
        return None
    if mode == "stub":
        print("⚠ stub ellipsoid prior — smoke tests only")
        return DensePriorTarget(stub_prior_grid(64), source="stub")
    if mode in ("occupancy", "stage1"):
        if occ_solid is None:
            raise RuntimeError("occupancy prior needs the Stage-1 grid — run §3.5 first")
        return DensePriorTarget(occ_solid, source="stage1")
    if mode in ("stage2", "slat", "mvsam3d"):
        if stage2_prior is None:
            raise RuntimeError("stage2 prior missing — run §6.0a (build_stage2_flow_prior) first")
        return stage2_prior
    raise ValueError(f"Unknown FLOW_PRIOR_MODE={mode}")


print("✓ dense prior + visibility-gated distillation ready")

### 4.4 — Occlusion-aware multi-view visibility $V(\mathbf{x})$

$$
V(\mathbf{x})=1-\exp\Bigl(-\sum_{i=1}^{N}\omega_i(\mathbf{x})\Bigr)
$$

$\omega_i=1$ only if the point projects inside image $i$'s **SAM mask** and passes a **z-buffer test**:
$z_{\mathrm{cam}} \le z_{\mathrm{surf}}(u,v)\,(1+\varepsilon_{\mathrm{rel}})+\varepsilon_{\mathrm{abs}}$.

$z_{\mathrm{surf}}$ is the first zero crossing of the **current Hash-SDF** along each pixel ray (low-res, `VIS_MAX_VIEWS` views, refreshed every `VIS_REFRESH_EVERY` iterations). Silhouette-only projection is *x-ray*: back faces and the interior land inside the masks, which is what kept the prior off the back of the building in the 8/18 run.

In [ ]:
# 4.4 — Occlusion-aware multi-view visibility V(x)
# A point counts as seen in view i only if it is inside mask i AND not behind z-buffer i, where the
# z-buffer is the first zero crossing of the current Hash-SDF (mask-only projection is x-ray).
import numpy as np
import torch
import torch.nn.functional as F


def rendering_weights_along_ray(density, deltas):
    """density, deltas: (N_rays, N_samp) → weights ω, transmittance T."""
    alpha = 1.0 - torch.exp(-density * deltas)
    T = torch.cumprod(
        torch.cat([torch.ones_like(alpha[:, :1]), 1.0 - alpha[:, :-1] + 1e-7], dim=-1),
        dim=-1,
    )
    return alpha * T, T


def near_far_from_sphere(rays_o, rays_d, radius=1.0):
    a = (rays_d * rays_d).sum(-1)
    b = 2 * (rays_o * rays_d).sum(-1)
    mid = -b / (2 * a + 1e-8)
    return (mid - radius).clamp_min(0.01), mid + radius


def project_points(pts, c2w, K):
    """pts (N,3) in world → u,v (pixel-centre convention), z_cam, in_front. c2w maps cam→world."""
    R = c2w[:3, :3]
    t = c2w[:3, 3]
    Xc = (pts - t[None, :]) @ R  # == R.T @ (pts - t) for row vectors
    z = Xc[:, 2]
    in_front = z > 1e-4
    zc = z.clamp_min(1e-4)
    u = K[0, 0] * (Xc[:, 0] / zc) + K[0, 2]
    v = K[1, 1] * (Xc[:, 1] / zc) + K[1, 2]
    return u, v, z, in_front


@torch.no_grad()
def render_sdf_zbuffer(sdf_net, c2w, K, H, W, ds=6, n_samples=128, gamma=None):
    """Camera-space depth of the first SDF zero crossing inside the unit sphere, at (H/ds, W/ds).
    Pixels whose ray never crosses zero get +inf (nothing occludes behind them)."""
    dev = next(sdf_net.parameters()).device
    c2w = torch.as_tensor(c2w, dtype=torch.float32, device=dev)
    K = torch.as_tensor(K, dtype=torch.float32, device=dev)
    h, w = max(1, int(H) // ds), max(1, int(W) // ds)
    ys = (torch.arange(h, device=dev, dtype=torch.float32) + 0.5) * (H / h) - 0.5
    xs = (torch.arange(w, device=dev, dtype=torch.float32) + 0.5) * (W / w) - 0.5
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    dirs_c = torch.stack([(xx - K[0, 2]) / K[0, 0], (yy - K[1, 2]) / K[1, 1], torch.ones_like(xx)], -1)
    dirs_c = (dirs_c / dirs_c.norm(dim=-1, keepdim=True)).reshape(-1, 3)
    rays_d = dirs_c @ c2w[:3, :3].T
    rays_o = c2w[:3, 3].expand_as(rays_d)
    near, far = near_far_from_sphere(rays_o, rays_d)
    t_lin = torch.linspace(0, 1, n_samples, device=dev)
    out = torch.full((rays_d.shape[0],), float("inf"), device=dev)
    chunk = max(256, 262144 // n_samples)
    for s in range(0, rays_d.shape[0], chunk):
        o, d = rays_o[s:s + chunk], rays_d[s:s + chunk]
        tt = near[s:s + chunk, None] + (far - near)[s:s + chunk, None] * t_lin[None, :]
        p = o[:, None, :] + d[:, None, :] * tt[..., None]
        sdf, _ = sdf_net(p.reshape(-1, 3), gamma=gamma)
        sdf = sdf.reshape(tt.shape).float()
        sdf = torch.where(p.norm(dim=-1) < 1.0, sdf, torch.ones_like(sdf))
        neg = sdf < 0
        hit = neg.any(dim=1)
        if not bool(hit.any()):
            continue
        i1 = neg.float().argmax(dim=1).clamp_min(1)
        i0 = i1 - 1
        s0 = sdf.gather(1, i0[:, None])[:, 0]
        s1 = sdf.gather(1, i1[:, None])[:, 0]
        t0 = tt.gather(1, i0[:, None])[:, 0]
        t1 = tt.gather(1, i1[:, None])[:, 0]
        frac = (s0 / (s0 - s1).clamp_min(1e-8)).clamp(0, 1)
        z_hit = (t0 + frac * (t1 - t0)) * dirs_c[s:s + chunk, 2]
        out[s:s + chunk] = torch.where(hit, z_hit, out[s:s + chunk])
    return out.reshape(h, w)


class VisibilityModel:
    """V(x) = 1 − exp(−Σ_i ω_i(x)); ω_i = 1 iff x projects into mask i and is not behind z-buffer i."""

    def __init__(self, Ks, masks, max_views=12, ds=6, n_samples=128,
                 eps_abs=0.05, eps_rel=0.01, depth_source="model"):
        self.masks_all = masks                     # (V,H,W) on device
        n = int(masks.shape[0])
        dev = masks.device
        self.view_idx = torch.linspace(0, n - 1, min(n, int(max_views)), device=dev).round().long().unique()
        self.Ks = torch.as_tensor(Ks, dtype=torch.float32, device=dev)[self.view_idx]
        self.masks = masks[self.view_idx]
        self.H, self.W = int(masks.shape[1]), int(masks.shape[2])
        self.ds, self.n_samples = int(ds), int(n_samples)
        self.eps_abs, self.eps_rel = float(eps_abs), float(eps_rel)
        self.depth_source = str(depth_source)
        self.c2ws = None
        self.zbuf = None

    @torch.no_grad()
    def refresh(self, sdf_net, c2ws_all, gamma=None):
        self.c2ws = c2ws_all.detach()[self.view_idx].float()
        if self.depth_source == "model":
            self.zbuf = torch.stack([
                render_sdf_zbuffer(sdf_net, self.c2ws[i], self.Ks[i], self.H, self.W,
                                   ds=self.ds, n_samples=self.n_samples, gamma=gamma)
                for i in range(self.c2ws.shape[0])
            ])
        else:
            self.zbuf = None

    @torch.no_grad()
    def visibility(self, pts):
        assert self.c2ws is not None, "call refresh() first"
        omega = torch.zeros(pts.shape[0], device=pts.device)
        for i in range(self.c2ws.shape[0]):
            u, v, z, front = project_points(pts, self.c2ws[i], self.Ks[i])
            inside = front & (u > -0.5) & (u < self.W - 0.5) & (v > -0.5) & (v < self.H - 0.5)
            ui = (u + 0.5).floor().long().clamp(0, self.W - 1)
            vi = (v + 0.5).floor().long().clamp(0, self.H - 1)
            seen = inside & (self.masks[i, vi, ui] > 0.5)
            if self.zbuf is not None:
                hb, wb = self.zbuf.shape[1:]
                ub = ((u + 0.5) * (wb / self.W)).floor().long().clamp(0, wb - 1)
                vb = ((v + 0.5) * (hb / self.H)).floor().long().clamp(0, hb - 1)
                zs = self.zbuf[i, vb, ub]
                seen = seen & ~(z > zs * (1.0 + self.eps_rel) + self.eps_abs)
            omega = omega + seen.float()
        return (1.0 - torch.exp(-omega)).clamp(0, 1)


@torch.no_grad()
def bake_vis_grid(vis_model, res=48, bs=65536):
    """Dense V on NeuS [-1,1]^3, layout [z,y,x] (sample with grid_sample_n11)."""
    pts = n11_grid_points(res, vis_model.masks.device)
    V = torch.cat([vis_model.visibility(pts[i:i + bs]) for i in range(0, pts.shape[0], bs)])
    return V.reshape(res, res, res)


@torch.no_grad()
def first_crossing_points(rays_o, rays_d, sdf, z):
    """First +→− SDF crossing along each sampled ray → (points, hit mask)."""
    neg = sdf < 0
    hit = neg.any(dim=1) & (~neg[:, 0])
    i1 = neg.float().argmax(dim=1).clamp_min(1)
    i0 = i1 - 1
    s0, s1 = sdf.gather(1, i0[:, None])[:, 0], sdf.gather(1, i1[:, None])[:, 0]
    z0, z1 = z.gather(1, i0[:, None])[:, 0], z.gather(1, i1[:, None])[:, 0]
    t = z0 + (s0 / (s0 - s1).clamp_min(1e-8)).clamp(0, 1) * (z1 - z0)
    return rays_o + rays_d * t[:, None], hit


def ray_expected_points(rays_o, rays_d, weights, z):
    depth = (weights * z).sum(dim=-1, keepdim=True) / (weights.sum(dim=-1, keepdim=True) + 1e-6)
    return rays_o + rays_d * depth


print("✓ occlusion-aware visibility V(x) ready (z-buffers from the current Hash-SDF)")

### 4.5 — Dual-field ray marcher + learnable poses

Rays accumulate **foreground Hash-SDF density** inside the unit sphere and **background NeRF**
outside (contracted). With `HARD_MASK_ROUTE=True`, unmasked rays take their *colour* from the
background field only — but the mask BCE always sees their **unrouted** foreground opacity, which is
what carves the Hash-SDF outside the SAM silhouettes. Images/masks are cached on the GPU once.

In [ ]:
# 4.5 — Cameras (BA) + GPU ray sampler + dual-field volume renderer (mask-routed colour, unrouted mask loss)
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LearnablePoses(nn.Module):
    """Axis-angle delta + translation delta on top of init c2w."""

    def __init__(self, c2ws_init: torch.Tensor):
        super().__init__()
        self.register_buffer("c2w0", c2ws_init.float().clone())
        n = c2ws_init.shape[0]
        self.r = nn.Parameter(torch.zeros(n, 3))
        self.t = nn.Parameter(torch.zeros(n, 3))

    def forward(self, idx=None):
        if idx is None:
            idx = torch.arange(self.c2w0.shape[0], device=self.c2w0.device)
        r = self.r[idx]
        angle = r.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        k = r / angle
        Kx = torch.zeros(len(idx), 3, 3, device=r.device, dtype=r.dtype)
        Kx[:, 0, 1], Kx[:, 0, 2] = -k[:, 2], k[:, 1]
        Kx[:, 1, 0], Kx[:, 1, 2] = k[:, 2], -k[:, 0]
        Kx[:, 2, 0], Kx[:, 2, 1] = -k[:, 1], k[:, 0]
        eye = torch.eye(3, device=r.device, dtype=r.dtype).expand(len(idx), 3, 3)
        R_delta = eye + torch.sin(angle)[..., None] * Kx + (1 - torch.cos(angle))[..., None] * (Kx @ Kx)
        R0 = self.c2w0[idx, :3, :3]
        t0 = self.c2w0[idx, :3, 3]
        out = torch.eye(4, device=r.device, dtype=r.dtype).expand(len(idx), 4, 4).clone()
        out[:, :3, :3] = R_delta @ R0
        out[:, :3, 3] = t0 + self.t[idx]
        return out


def pixels_to_rays(xs, ys, K, c2w):
    """Integer pixel centres → unit world rays. K, c2w are per-ray (N,3,3)/(N,4,4)."""
    dirs_c = torch.stack([
        (xs.float() - K[:, 0, 2]) / K[:, 0, 0],
        (ys.float() - K[:, 1, 2]) / K[:, 1, 1],
        torch.ones_like(K[:, 0, 0]),
    ], dim=-1)
    dirs_c = dirs_c / dirs_c.norm(dim=-1, keepdim=True)
    rays_d = (c2w[:, :3, :3] @ dirs_c.unsqueeze(-1)).squeeze(-1)
    return c2w[:, :3, 3], rays_d


class RaySampler:
    """Images / masks / intrinsics cached on the GPU once; prefer_mask of the rays come from mask pixels."""

    def __init__(self, images, masks, Ks, device, prefer_mask=0.7):
        self.device = torch.device(device)
        self.img = torch.from_numpy(np.stack(images).astype(np.float32)).to(self.device)   # (V,H,W,3)
        self.msk = torch.from_numpy(np.stack(masks).astype(np.float32)).to(self.device)    # (V,H,W)
        self.K = torch.as_tensor(np.asarray(Ks, np.float32), device=self.device)           # (V,3,3)
        self.V, self.H, self.W = self.msk.shape
        self.prefer_mask = float(prefer_mask)
        self.mask_flat = torch.nonzero(self.msk.reshape(-1) > 0.5).squeeze(-1)
        print(f"RaySampler: {self.V} views @ {self.W}x{self.H} on {self.device} | "
              f"mask pixels {self.mask_flat.numel():,} ({self.mask_flat.numel() / self.msk.numel():.1%})")

    def sample(self, c2ws, n_rays):
        HW = self.H * self.W
        n_mask = int(n_rays * self.prefer_mask) if self.mask_flat.numel() > 0 else 0
        parts = []
        if n_mask > 0:
            parts.append(self.mask_flat[torch.randint(0, self.mask_flat.numel(), (n_mask,), device=self.device)])
        if n_rays - n_mask > 0:
            parts.append(torch.randint(0, self.V * HW, (n_rays - n_mask,), device=self.device))
        flat = torch.cat(parts)
        vi = flat // HW
        rem = flat - vi * HW
        ys = rem // self.W
        xs = rem - ys * self.W
        rays_o, rays_d = pixels_to_rays(xs, ys, self.K[vi], c2ws[vi])
        return rays_o, rays_d, self.img[vi, ys, xs], self.msk[vi, ys, xs], vi


def render_dual_field(
    rays_o, rays_d,
    sdf_net, bg_net, color_net,
    gamma=None, n_samples=64, n_bg=32, mask_rays=None, hard_mask_route=True,
    need_eikonal=True, perturb=True,
):
    """Dual-field render.

    O_raw   — FG opacity WITHOUT mask routing → mask BCE on every ray (carves outside the masks).
    O_fg    — routed FG opacity used for the colour composite (unmasked rays → background colour).
    """
    device = rays_o.device
    n_rays = rays_o.shape[0]
    near, far = near_far_from_sphere(rays_o, rays_d)

    t_vals = torch.linspace(0, 1, n_samples, device=device)
    z = near[:, None] + (far - near)[:, None] * t_vals[None, :]
    if perturb:
        mids = 0.5 * (z[:, 1:] + z[:, :-1])
        upper = torch.cat([mids, z[:, -1:]], -1)
        lower = torch.cat([z[:, :1], mids], -1)
        z = lower + (upper - lower) * torch.rand_like(z)

    pts = rays_o[:, None, :] + rays_d[:, None, :] * z[..., None]
    pts_flat = pts.reshape(-1, 3)
    dirs_flat = rays_d[:, None, :].expand_as(pts).reshape(-1, 3)
    inside = pts_flat.detach().norm(dim=-1) < 1.0

    with torch.enable_grad():
        if not pts_flat.requires_grad:
            pts_flat.requires_grad_(True)
        dens, sdf, feat = sdf_net.density(pts_flat, gamma=gamma)
        grad = torch.autograd.grad(
            sdf.sum(), pts_flat, create_graph=need_eikonal, retain_graph=True
        )[0]
    if need_eikonal and bool(inside.any()):
        eik = ((grad[inside].norm(dim=-1) - 1.0) ** 2).mean()
    else:
        eik = torch.zeros((), device=device)
    normal = F.normalize(grad.detach(), dim=-1)

    rgb_fg = color_net(pts_flat, normal, dirs_flat, feat).reshape(n_rays, n_samples, 3)
    dens = dens.reshape(n_rays, n_samples) * inside.reshape(n_rays, n_samples).float()
    sdf = sdf.reshape(n_rays, n_samples)

    deltas = z[:, 1:] - z[:, :-1]
    deltas = torch.cat([deltas, torch.full_like(deltas[:, :1], 1e-2)], dim=-1)
    w_raw, _ = rendering_weights_along_ray(dens, deltas)
    O_raw = w_raw.sum(dim=-1).clamp(0, 1)
    if hard_mask_route and mask_rays is not None:
        w_fg = w_raw * (mask_rays > 0.5).float()[:, None]
    else:
        w_fg = w_raw
    O_fg = w_fg.sum(dim=-1).clamp(0, 1)
    rgb_fg_acc = (w_fg[..., None] * rgb_fg).sum(dim=1)

    z_bg = far[:, None] + torch.linspace(0, 1, n_bg, device=device)[None, :] * 2.0
    pts_bg = rays_o[:, None, :] + rays_d[:, None, :] * z_bg[..., None]
    dens_bg, rgb_bg = bg_net(pts_bg.reshape(-1, 3))
    dens_bg = dens_bg.reshape(n_rays, n_bg)
    rgb_bg = rgb_bg.reshape(n_rays, n_bg, 3)
    dlt_bg = z_bg[:, 1:] - z_bg[:, :-1]
    dlt_bg = torch.cat([dlt_bg, torch.full_like(dlt_bg[:, :1], 1e-1)], dim=-1)
    w_bg, _ = rendering_weights_along_ray(dens_bg, dlt_bg)
    rgb_bg_acc = (w_bg[..., None] * rgb_bg).sum(dim=1)
    rgb = rgb_fg_acc + (1.0 - O_fg)[:, None] * rgb_bg_acc

    return {
        "rgb": rgb, "O_fg": O_fg, "O_raw": O_raw, "weights_fg": w_fg, "weights_raw": w_raw,
        "eikonal": eik, "sdf": sdf, "z": z, "depth": (w_raw * z).sum(dim=1) / O_raw.clamp_min(1e-4),
    }


@torch.no_grad()
def render_image(view_idx, c2ws, Ks, H, W, sdf_net, bg_net, color_net, gamma=None, ds=2, chunk=4096,
                 n_samples=64, n_bg=32):
    """Full-view render at (H/ds, W/ds) → dict(rgb (h,w,3), opacity, depth, ys, xs pixel indices)."""
    dev = next(sdf_net.parameters()).device
    h, w = int(H) // ds, int(W) // ds
    ys = ((torch.arange(h, device=dev) + 0.5) * (H / h) - 0.5).round().long().clamp(0, H - 1)
    xs = ((torch.arange(w, device=dev) + 0.5) * (W / w) - 0.5).round().long().clamp(0, W - 1)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    yy, xx = yy.reshape(-1), xx.reshape(-1)
    K = torch.as_tensor(Ks, dtype=torch.float32, device=dev)[view_idx].expand(yy.shape[0], 3, 3)
    c2w = c2ws[view_idx].detach().float().expand(yy.shape[0], 4, 4)
    rgb, opa, dep = [], [], []
    for s in range(0, yy.shape[0], chunk):
        ro, rd = pixels_to_rays(xx[s:s + chunk], yy[s:s + chunk], K[s:s + chunk], c2w[s:s + chunk])
        out = render_dual_field(ro, rd, sdf_net, bg_net, color_net, gamma=gamma, n_samples=n_samples,
                                n_bg=n_bg, mask_rays=None, hard_mask_route=False,
                                need_eikonal=False, perturb=False)
        rgb.append(out["rgb"].detach()); opa.append(out["O_raw"].detach()); dep.append(out["depth"].detach())
    return {
        "rgb": torch.cat(rgb).reshape(h, w, 3).clamp(0, 1).cpu().numpy(),
        "opacity": torch.cat(opa).reshape(h, w).cpu().numpy(),
        "depth": torch.cat(dep).reshape(h, w).cpu().numpy(),
        "ys": ys.cpu().numpy(), "xs": xs.cpu().numpy(),
    }


print("✓ poses + GPU ray sampler + renderer ready (mask BCE on unrouted opacity)")

## 5 — Smoke tests (no training)

Verifies hash annealing, density conversion, z-buffer visibility, the prior loss backward into the
hash table, and a single render step with pose gradients.

In [ ]:
# 5.0 — Module smoke tests (tiny models; no training)
import numpy as np
import torch
import torch.nn.functional as F

device = torch.device(DEVICE)
_enc = HashGridEncoding(
    n_levels=HASH_N_LEVELS, n_features_per_level=HASH_N_FEATURES,
    log2_hashmap_size=min(HASH_LOG2_SIZE, 16),
    base_resolution=HASH_BASE_RES, max_resolution=512,
).to(device)
_sdf = HashSDF(_enc, hidden=SDF_HIDDEN, n_layers=SDF_N_LAYERS).to(device)
_bg = BackgroundNeRF(hidden=BG_HIDDEN).to(device)
_col = ColorNet(feat_dim=SDF_HIDDEN, hidden=SDF_HIDDEN).to(device)

g = anneal_gamma(1000, HASH_N_LEVELS, ANNEAL_ITERS)
W = hash_anneal_weights(HASH_N_LEVELS, g)
print(f"[anneal] γ(1000)={g:.2f} | active levels≈{(W > 0.01).sum().item()} | W[:6]={W[:6].cpu().numpy().round(3)}")

pts = torch.rand(128, 3, device=device) * 2 - 1
dens, sdf, feat = _sdf.density(pts, gamma=g)
print(f"[density] sdf∈[{sdf.min():.3f},{sdf.max():.3f}] dens∈[{dens.min():.3f},{dens.max():.3f}] β={_sdf.beta.item():.2f}")

_c2w0 = torch.from_numpy(scene["c2ws"]).float().to(device)
_masks = torch.from_numpy(np.stack(scene["masks"]).astype(np.float32)).to(device)
_vis = VisibilityModel(scene["Ks"], _masks, max_views=2, ds=16, n_samples=32, depth_source=VIS_DEPTH_SOURCE)
_vis.refresh(_sdf, _c2w0, gamma=g)
V = _vis.visibility(pts)
_zb = _vis.zbuf
print(f"[vis] V∈[{float(V.min()):.3f},{float(V.max()):.3f}] mean={float(V.mean()):.3f} | "
      f"zbuf {None if _zb is None else tuple(_zb.shape)} finite={0 if _zb is None else float(torch.isfinite(_zb).float().mean()):.2f}")

_prior = build_flow_prior("stub").to(device)
_vg = bake_vis_grid(_vis, 16)
_pp = sample_ball(1024, device)
L_prior, _st = prior_distillation_loss(_sdf, _prior, _pp, g, vis=grid_sample_n11(_vg, _pp), tau=PRIOR_TAU)
L_prior.backward()
_grad_ok = _enc.backend == "tcnn" or (_enc.table.grad is not None and float(_enc.table.grad.abs().sum()) > 0)
print(f"[prior] L={float(L_prior):.4f} grad_reaches_hash={_grad_ok} stats={_st}")
_sdf.zero_grad(set_to_none=True)

_poses = LearnablePoses(_c2w0.cpu()).to(device)
_sampler = RaySampler(scene["images"], scene["masks"], scene["Ks"], device)
rays_o, rays_d, rgb_gt, msk, vi = _sampler.sample(_poses(), 64)
out = render_dual_field(rays_o, rays_d, _sdf, _bg, _col, gamma=g, n_samples=32, n_bg=16,
                        mask_rays=msk, hard_mask_route=HARD_MASK_ROUTE)
L_photo = F.mse_loss(out["rgb"], rgb_gt)
L_mask = F.binary_cross_entropy(out["O_raw"].clamp(1e-4, 1 - 1e-4), msk.clamp(0, 1))
(L_photo + 0.1 * out["eikonal"] + 0.5 * L_mask).backward()
_pose_grad = _poses.t.grad is not None and float(_poses.t.grad.abs().sum()) > 0
print(f"[render] photo={float(L_photo):.4f} mask={float(L_mask):.4f} eik={float(out['eikonal']):.4f} "
      f"O_raw={float(out['O_raw'].mean()):.3f} O_fg={float(out['O_fg'].mean()):.3f} pose_grad={_pose_grad}")
del _enc, _sdf, _bg, _col, _vis, _prior, _poses, _sampler
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✓ all smoke tests passed")

## 6 — Train GF-HashBA

Each iteration:

1. Sample rays (GPU-cached pixels, 70 % from masks); render the dual field with the current $\gamma(k)$
2. $\mathcal{L}_{\mathrm{photo}}$ (routed colour), $\mathcal{L}_{\mathrm{mask}}$ (unrouted opacity → carving), $\mathcal{L}_{\mathrm{eik}}$ (inside the sphere)
3. $\mathcal{L}_{\mathrm{prior}}$ on `PRIOR_N_PTS` random points, weighted by $1-\mathrm{sg}(V)$ (unless `FLOW_PRIOR_ON_VISIBLE`)
4. Every `VIS_REFRESH_EVERY`: re-render z-buffers and re-bake $V$
5. Every `SNAPSHOT_EVERY`: checkpoint + `progress_it*.png` mesh preview → Drive

### 6.0b — Mesh utilities + prefit Hash-SDF to Stage-1 occupancy

Same idea as BA-NeuS §13.2: before photometric training, fit the SDF to
`pseudo_sdf = (0.5 - p) * 0.2` from Stage-1. The Stage-1 occupancy is a surface shell
(TRELLIS sparse structure), so with `PREFIT_SOLIDIFY=True` the enclosed interior is filled
first — one closed surface instead of a double-walled shell. §6.0 calls this automatically.

`mc_mesh` reorders marching-cubes vertices from array order (z,y,x) to xyz.

In [ ]:
# 6.0b — Mesh utilities + Stage-1 SDF prefit helper (called from §6.0)
import time
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path

try:
    from skimage.measure import marching_cubes
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])
    from skimage.measure import marching_cubes
import trimesh
import matplotlib.pyplot as plt


@torch.no_grad()
def sdf_volume(sdf_net, res, gamma, bs=65536):
    """SDF on a res^3 grid over [-1,1]^3 → numpy, layout [z,y,x].

    Intersected with the unit ball (max(f, |x|−1)): the renderer, prefit and prior only constrain
    the field inside the sphere; outside it the hash values are noise."""
    dev = next(sdf_net.parameters()).device
    pts = n11_grid_points(res, dev)
    vals = []
    for i in range(0, pts.shape[0], bs):
        p = pts[i:i + bs]
        f = sdf_net(p, gamma=gamma)[0].squeeze(-1)
        vals.append(torch.maximum(f, p.norm(dim=-1) - 1.0).float().cpu())
    return torch.cat(vals).reshape(res, res, res).numpy()


def mc_mesh(vol, level=0.0):
    """Marching cubes on a [z,y,x] grid over [-1,1]^3 → (verts xyz, faces) in NeuS units.
    skimage returns vertices in array-axis order (z,y,x); reordering to xyz is a reflection,
    so the winding is flipped to keep normals outward."""
    res = vol.shape[0]
    verts, faces, _, _ = marching_cubes(vol, level=level)
    verts = (verts[:, ::-1] / (res - 1) * 2.0 - 1.0).astype(np.float32)
    faces = faces[:, ::-1].astype(np.int64)
    return np.ascontiguousarray(verts), np.ascontiguousarray(faces)


def keep_large_components(verts, faces, min_frac=0.01):
    """Drop floating debris: keep connected components with ≥ min_frac of the faces."""
    m = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    parts = m.split(only_watertight=False)
    if len(parts) <= 1:
        return verts, faces, len(parts)
    keep = [p for p in parts if len(p.faces) >= min_frac * len(faces)] or [max(parts, key=lambda p: len(p.faces))]
    out = trimesh.util.concatenate(keep)
    return np.asarray(out.vertices, np.float32), np.asarray(out.faces, np.int64), len(parts)


def plot_meshes(items, path=None, views=((-60, 20), (40, 25), (90, 75)), max_faces=40000, lim=1.0, show=True):
    """items: list of (title, verts, faces). One row per mesh, one column per view (same axes for all)."""
    rows = len(items)
    fig = plt.figure(figsize=(4.0 * len(views), 3.8 * rows))
    rng = np.random.default_rng(0)
    for r, (title, v, f) in enumerate(items):
        if f is not None and len(f) > max_faces:
            f = f[rng.choice(len(f), max_faces, replace=False)]
        for c, (az, el) in enumerate(views):
            ax = fig.add_subplot(rows, len(views), r * len(views) + c + 1, projection="3d")
            if v is not None and f is not None and len(f) > 0:
                ax.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f, cmap="viridis",
                                linewidth=0, antialiased=False)
            ax.view_init(elev=el, azim=az)
            ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
            ax.set_box_aspect((1, 1, 1))
            ax.set_title(f"{title} ({az},{el})", fontsize=9)
    plt.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=120, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return path


PREFIT_MESH = None   # (verts, faces) in NeuS frame — reused by the §7 comparison figure


def prefit_hash_sdf(
    sdf_net,
    n_iters=SDF_PREFIT_ITERS,
    batch=SDF_PREFIT_BATCH,
    lr=SDF_PREFIT_LR,
    cache_path=None,
):
    """Fit the Hash-SDF to the (solidified) Stage-1 pseudo-SDF inside the unit ball.

    Uses hash levels up to PREFIT_GAMMA only, so main training (which starts its anneal at
    ANNEAL_START_LVL = PREFIT_GAMMA) sees exactly the fitted field at iteration 0.
    With cache_path (Drive), a matching earlier prefit is loaded instead of re-fitting.
    """
    global PREFIT_MESH
    if not HAS_STAGE1:
        print("skip prefit — no Stage-1")
        return None
    device = next(sdf_net.parameters()).device
    path = OUTPUT_DIR / "gfhashba_sdf_prefit.pth"

    if cache_path is not None and USE_CACHE and not REFRESH_CACHE and Path(cache_path).exists():
        ck = torch.load(cache_path, map_location=device, weights_only=False)
        sdf_net.load_state_dict(ck["sdf"])
        print(f"✓ prefit loaded from Drive cache {Path(cache_path).name} (mse={ck.get('mse')}) — skipped fitting")
        shutil.copy2(cache_path, path)
    else:
        # Diagnose teacher BEFORE fitting — mse→0 with an empty/constant target is a false win
        with torch.no_grad():
            _probe = sample_ball(8192, device)
            _p = sample_prefit_occ(_probe)
            _tgt = pseudo_sdf_from_prob(_p)
            print(
                f"Stage-1 teacher probe: occ mean={float(_p.mean()):.3f}  "
                f"frac>0.5={float((_p > 0.5).float().mean()):.3f}  "
                f"pseudo-SDF∈[{float(_tgt.min()):.3f},{float(_tgt.max()):.3f}] mean={float(_tgt.mean()):.3f}"
            )
            if float((_p > 0.5).float().mean()) < 0.01:
                print("⚠ Almost no occupied voxels in NeuS ball — Stage-1↔frame mapping is wrong.")
                print("  Prefit will learn ~empty SDF. Fix sam_baneus §12.1b/§12.2 before trusting this run.")

        opt = torch.optim.Adam(sdf_net.parameters(), lr=lr)
        sdf_net.train()
        t0_ = time.time()
        last = 0.0
        for it in range(n_iters):
            pts = sample_ball(batch, device)
            with torch.no_grad():
                tgt = pseudo_sdf_from_prob(sample_prefit_occ(pts)).unsqueeze(-1)
            g = anneal_gamma(it, PREFIT_GAMMA, max(1, n_iters // 2), 1.0)   # 1 → PREFIT_GAMMA, then hold
            pred, _ = sdf_net(pts, gamma=g)
            loss = F.mse_loss(pred, tgt)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            last = float(loss)
            if it % 500 == 0 or it == n_iters - 1:
                print(f"  prefit {it:5d}/{n_iters}  mse={last:.5f}  γ={g:.2f}  [{time.time()-t0_:.0f}s]")
        torch.save({"sdf": sdf_net.state_dict(), "n_iters": n_iters, "mse": last, "gamma": PREFIT_GAMMA}, path)
        print("saved", path)
        if cache_path is not None:
            try:
                Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, cache_path)
                print("  cached on Drive →", cache_path)
            except Exception as e:
                print("⚠ prefit cache write failed:", e)

    sdf_net.eval()
    vol = sdf_volume(sdf_net, PREVIEW_RES, float(PREFIT_GAMMA))
    sdf_net.train()
    print(f"prefit SDF range [{vol.min():.3f},{vol.max():.3f}]  frac_inside={float((vol < 0).mean()):.4f}")
    out_glb = out_png = None
    try:
        verts, faces = mc_mesh(vol, 0.0)
        PREFIT_MESH = (verts, faces)
        out_glb = OUTPUT_DIR / "gfhashba_prefit_mesh.glb"
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(out_glb)
        print(f"prefit mesh: {len(verts)} verts → {out_glb}")
        out_png = plot_meshes([("prefit (Stage-1)", verts, faces)], OUTPUT_DIR / "gfhashba_prefit_preview.png")
    except Exception as e:
        print("prefit MC failed:", e)
    to_drive(path, out_glb, out_png)
    return path


print("mesh utils + prefit helper ready (exports → Drive)")

### 6.0a — Build the MV-SAM3D Stage-2 prior

Install path **mirrors** `mvsam3d_stage2LoRA` / `sam_baneus_ultrra` / `mvsam3d_example_current`:
`PIP_CONSTRAINT` (numpy 1.26.4 + scipy 1.13.1) → same pip dep list → Drive
`colab_compiled_pkgs.tar.gz` → companion wheels (`pygltflib`, `pccm`) → hydra patch → `Inference`.

Then: teacher SLAT on the **exported Stage-1 coords** (multi-view MultiDiffusion via
`inject_generator_multi_view`, optional LoRA) → un-normalise → **`decode_slat(..., ["mesh"])`** →
canonical → Sim3 → NeuS → surface-sample → solidify → `STAGE2_PRIOR`. Check
`stage2_align_check.png` (IoU vs the Stage-1 solid should be well above 0.3).

The SAM3D pipeline is freed afterwards — nothing is queried during training.
If this cell fails and `STAGE2_FALLBACK_TO_STAGE1=True`, training still runs with the Stage-1 prior
and every figure/summary is labelled `prior_source='stage1 (stage2 failed)'`.

Requires **torch 2.5.1+cu121** (§1.0b). Missing `pygltflib`/`pccm` after the tarball is normal.

In [ ]:
# 6.0a — Stage-2 prior: MV-SAM3D teacher SLAT → mesh decoder → Sim3 → NeuS occupancy
import os, sys, importlib.util, subprocess
import numpy as np
import torch
from pathlib import Path
from PIL import Image
import trimesh
import matplotlib.pyplot as plt

STAGE2_PRIOR = None
STAGE2_MESH_NEUS = None


def _ensure_mvsam3d_stack():
    """Clone / link / resolve paths exactly like mvsam3d_stage2LoRA §§1–2 + load cell."""
    root = Path("/content")
    sam3d = root / "sam-3d-objects"
    mvsam = root / "MV-SAM3D"

    repos = {
        "sam-3d-objects": "https://github.com/facebookresearch/sam-3d-objects.git",
        "MV-SAM3D": "https://github.com/devinli123/MV-SAM3D.git",
    }
    cwd = os.getcwd()
    try:
        os.chdir(str(root))
        for name, url in repos.items():
            if (root / name).is_dir():
                print(f"OK   {name}/")
            else:
                print(f"cloning {name}…")
                subprocess.check_call(["git", "clone", "--depth", "1", url, name])
    finally:
        os.chdir(cwd)

    inf = mvsam / "notebook" / "inference.py"
    if not inf.exists():
        (mvsam / "notebook").mkdir(parents=True, exist_ok=True)
        for fn in ("inference.py", "load_images_and_masks.py", "mesh_alignment.py"):
            src, dst = sam3d / "notebook" / fn, mvsam / "notebook" / fn
            if src.exists() and not dst.exists():
                os.symlink(src, dst)
                print("linked", dst)

    candidates = [
        mvsam,
        Path("/content/drive/MyDrive/SatelliteFuse/code/MV-SAM3D"),
        sam3d,
    ]
    mvsam_root, inf_py = None, None
    for r in candidates:
        cand = r / "notebook" / "inference.py"
        if cand.exists():
            mvsam_root, inf_py = r, cand
            break
    if inf_py is None:
        raise FileNotFoundError(
            "notebook/inference.py not found after clone/link.\n"
            f"Checked: {[str(c / 'notebook' / 'inference.py') for c in candidates]}"
        )

    try:
        from huggingface_hub import login, snapshot_download
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
        if hf_token:
            login(token=hf_token)
        ckpt_hf = sam3d / "checkpoints" / "hf"
        need_dl = not (
            (ckpt_hf / "pipeline.yaml").exists()
            or (ckpt_hf / "checkpoints" / "pipeline.yaml").exists()
        )
        if need_dl:
            if not hf_token:
                raise RuntimeError("Need Colab secret HF_TOKEN for facebook/sam-3d-objects")
            print("Downloading facebook/sam-3d-objects checkpoints…")
            snapshot_download(repo_id="facebook/sam-3d-objects", token=hf_token, local_dir=str(ckpt_hf))
        dst = mvsam / "checkpoints" / "hf"
        src_nested = ckpt_hf / "checkpoints"
        src = src_nested if (src_nested / "pipeline.yaml").exists() else ckpt_hf
        if not dst.exists() and src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(src, dst)
            print("symlinked", dst, "→", src)
    except Exception as e:
        print(f"⚠ checkpoint ensure: {type(e).__name__}: {e}")

    pipe_yaml = mvsam / "checkpoints" / "hf" / "pipeline.yaml"
    if not pipe_yaml.exists():
        pipe_yaml = mvsam_root / "checkpoints" / "hf" / "pipeline.yaml"
    if not pipe_yaml.exists():
        for alt in (
            sam3d / "checkpoints" / "hf" / "pipeline.yaml",
            sam3d / "checkpoints" / "hf" / "checkpoints" / "pipeline.yaml",
        ):
            if alt.exists():
                pipe_yaml = alt
                break
    if not pipe_yaml.exists():
        raise FileNotFoundError(f"pipeline.yaml missing under {mvsam}/checkpoints/hf — set HF_TOKEN")

    nb_dir = str(inf_py.parent)
    if nb_dir not in sys.path:
        sys.path.insert(0, nb_dir)
    if str(mvsam_root) not in sys.path:
        sys.path.insert(0, str(mvsam_root))

    print("MVSAM3D_ROOT =", mvsam_root)
    print("inference.py =", inf_py)
    print("PIPELINE_YAML =", pipe_yaml)
    return mvsam_root, inf_py, pipe_yaml


def _ensure_sam3d_python_deps():
    """Exact env path from sam_baneus_ultrra + mvsam3d_stage2LoRA + mvsam3d_example_current.

    Critical order (do not reorder):
      1) PIP_CONSTRAINT + numpy/scipy pin
      2) pip core deps (LoRA list) + warp-lang companions
      3) CUMM_DISABLE_JIT=1 / SPCONV_DISABLE_JIT=1  BEFORE any cumm/spconv import
      4) restore colab_compiled_pkgs.tar.gz
      5) re-pin numpy/scipy
      6) import pytorch3d, flash_attn, kaolin, gsplat, spconv.pytorch
    """
    import site
    import importlib

    def _pip(*args):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

    import torch
    TARGET = "2.5.1"
    print(f"torch={torch.__version__} (Stage-2 needs {TARGET}+cu121) py={sys.version.split()[0]}")
    if sys.version_info >= (3, 13):
        raise RuntimeError(
            f"Python {sys.version.split()[0]} has no torch 2.5.1 wheels. "
            "Pin Colab Runtime version to 2026.07/2026.04 (Python 3.12)."
        )
    if not torch.__version__.startswith(TARGET):
        # Install one-by-one so torchvision cannot fall back to bare PyPI (0.1.6/0.2.0 only).
        cu = "https://download.pytorch.org/whl/cu121"
        for pkg in ("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1"):
            _pip(pkg, "--index-url", cu)
        try:
            _pip("xformers==0.0.28.post3", "--index-url", cu)
        except Exception:
            pass
        raise RuntimeError(
            f"Torch pinned to {TARGET}. Runtime → Restart session, then re-run from §1.1 / §6.0a."
        )

    pin = Path("/content/_numpy_pin.txt")
    pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
    os.environ["PIP_CONSTRAINT"] = str(pin)
    print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"])
    _pip("numpy==1.26.4", "scipy==1.13.1")
    import numpy, scipy
    print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

    try:
        import sam3d_objects  # noqa: F401
    except Exception:
        _pip("--no-deps", "-e", "/content/sam-3d-objects")

    print("pip: LoRA/sam_baneus core deps…")
    _pip(
        "hydra-core==1.3.2",
        "omegaconf",
        "loguru",
        "opencv-python-headless",
        "trimesh",
        "roma",
        "seaborn",
        "safetensors",
        "tqdm",
        "einops",
        "astor",
        "lightning",
        "xatlas",
        "pyvista",
        "pymeshfix",
        "igraph",
        "numpy<2",
        "huggingface_hub[cli]",
        "gradio",
        "peft==0.13.2",
        "accelerate",
        "matplotlib",
        "pillow",
        "pygltflib",
        "pccm",
        "addict",
        "yapf",
        "warp-lang",
        "ipycanvas",
        "ipyevents",
        "plyfile",
        "easydict",
        "jaxtyping",
        "timm",
        "imageio",
        "scikit-image",
    )
    try:
        _pip("open3d")
    except Exception as e:
        print("⚠ open3d:", e)

    try:
        import utils3d  # noqa: F401
    except Exception:
        _pip(
            "--no-deps",
            "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900",
        )
    try:
        import MoGe  # noqa: F401
    except Exception:
        try:
            import moge  # noqa: F401
        except Exception:
            _pip(
                "--no-deps",
                "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b",
            )

    # ── sam_baneus: MUST set before ANY cumm/spconv import (reads env once) ──
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    print("CUMM_DISABLE_JIT=1 SPCONV_DISABLE_JIT=1 (sam_baneus — skip bogus ninja rebuild)")

    def _purge_compiled():
        for name in list(sys.modules):
            root = name.split(".")[0]
            if root in ("pytorch3d", "flash_attn", "kaolin", "gsplat", "spconv", "cumm", "warp"):
                del sys.modules[name]

    def _compiled_ok():
        """Same spirit as LoRA _compiled_ok, but probe spconv.pytorch (sam_baneus)."""
        try:
            import pytorch3d  # noqa: F401
            import flash_attn  # noqa: F401
            import kaolin  # noqa: F401
            import gsplat  # noqa: F401
            import spconv.pytorch  # noqa: F401
            return None
        except Exception as e:
            return e

    # example_current / LoRA: restore tarball FIRST — never import spconv before that
    # (importing a broken/partial cumm triggers ninja rebuild → tensorview/*.h errors).
    sp = site.getsitepackages()[0]
    backup = next(
        (
            b
            for b in (
                Path("/content/drive/MyDrive/colab_compiled_pkgs.tar.gz"),
                Path("/content/drive/MyDrive/SatelliteFuse/colab_compiled_pkgs.tar.gz"),
            )
            if b.exists()
        ),
        None,
    )
    if backup is None:
        raise RuntimeError(
            "No colab_compiled_pkgs.tar.gz. Run mvsam3d_stage2LoRA §§0–4c or "
            "mvsam3d_example_current restore once, then retry."
        )

    print("Restoring compiled pkgs from", backup)
    subprocess.check_call(["tar", "xzf", str(backup), "-C", sp])
    _pip("numpy==1.26.4", "scipy==1.13.1")
    # re-assert JIT flags after any import side effects
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    _purge_compiled()

    # Pure-python companions ONLY — never pip cumm/spconv/pytorch3d (tarball owns those).
    # Bug: auto `pip cumm` on ModuleNotFoundError destroyed prebuilt cumm.core_cc.
    _SAFE_PIP = {
        "warp": "warp-lang",
        "ipycanvas": "ipycanvas",
        "ipyevents": "ipyevents",
        "pygltflib": "pygltflib",
        "pccm": "pccm",
        "addict": "addict",
        "plyfile": "plyfile",
        "easydict": "easydict",
        "jaxtyping": "jaxtyping",
    }
    _TAR_ONLY = {"cumm", "spconv", "pytorch3d", "flash_attn", "kaolin", "gsplat", "core_cc"}

    def _restore_tarball():
        print("Restoring compiled pkgs from", backup)
        subprocess.check_call(["tar", "xzf", str(backup), "-C", sp])
        _pip("numpy==1.26.4", "scipy==1.13.1")
        os.environ["CUMM_DISABLE_JIT"] = "1"
        os.environ["SPCONV_DISABLE_JIT"] = "1"
        _purge_compiled()

    err = _compiled_ok()
    if err is not None and isinstance(err, ModuleNotFoundError):
        miss = getattr(err, "name", None) or ""
        root = miss.split(".")[0] if miss else ""
        if not root and "No module named" in str(err):
            import re
            m = re.search(r"No module named ['\"]([^'\"]+)['\"]", str(err))
            root = m.group(1).split(".")[0] if m else ""
        if root in _TAR_ONLY or root.startswith("cumm"):
            print(f"compiled missing '{miss}' — re-extract tarball (do NOT pip {root})")
            _restore_tarball()
            err = _compiled_ok()
        elif root in _SAFE_PIP:
            print(f"missing {root} → pip {_SAFE_PIP[root]}")
            _pip(_SAFE_PIP[root])
            _purge_compiled()
            err = _compiled_ok()
        else:
            print(f"⚠ unknown missing module '{root}': {err}")

    if err is not None and ("cumm" in str(err).lower() or "spconv" in str(err).lower()):
        print("cumm/spconv still broken — clean tarball re-extract (no pip cumm)")
        _restore_tarball()
        err = _compiled_ok()

    if err is not None:
        raise RuntimeError(
            "compiled stack still broken after tarball + CUMM_DISABLE_JIT.\n"
            f"{type(err).__name__}: {err}\n"
            "If you previously saw 'missing cumm → pip cumm', the kernel is poisoned:\n"
            "Runtime → Restart session, then §1.0b → … → §6.0a (fixed notebook)."
        )
    print("✓ compiled stack OK (pytorch3d/flash_attn/kaolin/gsplat/spconv.pytorch)")


def ensure_peft_for_lora():
    try:
        import peft
        print(f"✓ peft {getattr(peft, '__version__', '?')}")
        return peft
    except Exception as e:
        print(f"peft import failed ({e}); pip peft==0.13.2 …")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "peft==0.13.2"]
        )
        for name in list(sys.modules):
            if name.split(".")[0] in ("peft", "transformers"):
                del sys.modules[name]
        import peft
        return peft


def _load_inference(inf_py, pipe_yaml):
    # Colab has no conda — inference.py does: CUDA_HOME = CONDA_PREFIX
    cuda_home = "/usr/local/cuda"
    if not Path(cuda_home).exists():
        for cand in ("/usr/local/cuda-12.1", "/usr/local/cuda-12", "/usr/local/cuda-11.8"):
            if Path(cand).exists():
                cuda_home = cand
                break
    os.environ["CUDA_HOME"] = cuda_home
    os.environ["CONDA_PREFIX"] = cuda_home
    os.environ["LIDRA_SKIP_INIT"] = "true"
    # sam_baneus: must precede any accidental cumm import inside deps
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"

    _ensure_sam3d_python_deps()

    hydra_patch = Path("/content/sam-3d-objects/patching/hydra")
    if hydra_patch.exists():
        try:
            subprocess.check_call([sys.executable, str(hydra_patch)])
            print("✓ Hydra patched")
        except Exception as e:
            print(f"⚠ hydra patch: {e}")

    # Final gate + Inference import with auto-pip for remaining pure ModuleNotFoundErrors
    _ALIAS = {
        "plyfile": "plyfile",
        "easydict": "easydict",
        "jaxtyping": "jaxtyping",
        "timm": "timm",
        "warp": "warp-lang",
        "ipycanvas": "ipycanvas",
        "imageio": "imageio",
        "skimage": "scikit-image",
        "cv2": "opencv-python-headless",
    }

    def _pip_miss(name):
        pkg = _ALIAS.get(name, name)
        print(f"  Inference missing '{name}' → pip {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    def _load_inference_cls():
        try:
            from inference import Inference as _Inf
            return _Inf
        except ModuleNotFoundError:
            spec = importlib.util.spec_from_file_location("mvsam3d_notebook_inference", inf_py)
            mod = importlib.util.module_from_spec(spec)
            sys.modules["mvsam3d_notebook_inference"] = mod
            spec.loader.exec_module(mod)
            return mod.Inference

    Inference = None
    last_err = None
    for _round in range(15):
        try:
            import hydra, utils3d, kaolin  # noqa: F401
            Inference = _load_inference_cls()
            last_err = None
            break
        except ModuleNotFoundError as e:
            last_err = e
            miss = getattr(e, "name", None) or ""
            miss = miss.split(".")[0] if miss else ""
            if not miss:
                import re
                m = re.search(r"No module named ['\"]([^'\"]+)['\"]", str(e))
                miss = m.group(1).split(".")[0] if m else ""
            if not miss:
                raise
            _pip_miss(miss)
            # drop half-imported sam3d / inference modules so retry is clean
            for name in list(sys.modules):
                if name.split(".")[0] in ("inference", "sam3d_objects", "mvsam3d_notebook_inference", miss):
                    del sys.modules[name]
        except Exception as e:
            raise RuntimeError(
                f"Pre-Inference import failed: {type(e).__name__}: {e}\n"
                "Deps should match mvsam3d_stage2LoRA / sam_baneus_ultrra."
            ) from e
    if Inference is None:
        raise RuntimeError(f"Could not import Inference after retries: {last_err}")

    print("Loading Inference pipeline (Stage-2 / FlowMatching)…")
    inference = Inference(str(pipe_yaml), compile=False)
    return inference


def _as_alpha(mask) -> np.ndarray:
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m[..., 3] if m.shape[-1] == 4 else m[..., 0]
    if m.dtype == bool:
        m = m.astype(np.uint8) * 255
    elif m.dtype != np.uint8:
        m = (m * 255).astype(np.uint8) if float(m.max()) <= 1.0 else m.astype(np.uint8)
    return m[..., None]


def _preprocess_slat_views(pipeline, view_images, view_masks):
    """RGBA + slat preprocess (same as mvsam3d_stage2LoRA). The Stage-1/pointmap branch is skipped:
    with exported Stage-1 coords it is never used, and it runs MoGe once per view."""
    slat_dicts = []
    for image, mask in zip(view_images, view_masks):
        image = np.asarray(image)[..., :3].astype(np.uint8)
        if mask is not None:
            alpha = _as_alpha(mask)
            if alpha.shape[:2] != image.shape[:2]:
                alpha = np.asarray(Image.fromarray(alpha[..., 0]).resize(image.shape[1::-1], Image.NEAREST))[..., None]
        else:
            alpha = np.full((*image.shape[:2], 1), 255, np.uint8)
        rgba = Image.fromarray(np.concatenate([image, alpha], axis=-1))
        slat_dicts.append(pipeline.preprocess_image(rgba, pipeline.slat_preprocessor))
    return slat_dicts


@torch.no_grad()
def mesh_to_occupancy_n11(verts_n11, faces, res=128, n_samples=2_000_000, close_iters=2):
    """Area-weighted surface samples of a NeuS-frame mesh → voxel shell → closing + fill → solid [z,y,x]."""
    dev = torch.device(DEVICE)
    v = torch.as_tensor(np.asarray(verts_n11), dtype=torch.float32, device=dev)
    f = torch.as_tensor(np.asarray(faces), dtype=torch.long, device=dev)
    a, b, c = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    area = torch.linalg.cross(b - a, c - a, dim=-1).norm(dim=-1) * 0.5
    idx = torch.multinomial(area / area.sum().clamp_min(1e-12), int(n_samples), replacement=True)
    r1 = torch.rand(idx.shape[0], 1, device=dev).sqrt()
    r2 = torch.rand(idx.shape[0], 1, device=dev)
    p = (1 - r1) * a[idx] + r1 * (1 - r2) * b[idx] + r1 * r2 * c[idx]
    gi = ((p + 1.0) * 0.5 * (res - 1)).round().long()
    gi = gi[((gi >= 0) & (gi < res)).all(dim=-1)]
    vox = torch.zeros(res, res, res, device=dev)
    vox[gi[:, 2], gi[:, 1], gi[:, 0]] = 1.0
    return solidify_occupancy(vox, thr=0.5, close_iters=close_iters)


def _occ_iou(a, b, thr=0.5):
    a, b = (a > thr), (b > thr)
    return float((a & b).sum()) / max(1.0, float((a | b).sum()))


STAGE2_INFO = {}


def stage2_cache_path():
    lora = str(STAGE2_LORA_DIR) if (STAGE2_LORA_DIR is not None and Path(str(STAGE2_LORA_DIR)).exists()) else None
    key = cache_key("stage2_v2", SAM_BANEU_RUN, CASE_NAME, DATA_ROOT, lora, STAGE2_TEACHER_STEPS,
                    STAGE2_MICRO_N, PRIOR_GRID_RES, USE_EXPORTED_STAGE1_COORDS)
    return CACHE_DIR / f"stage2_prior_{key}.npz"


def _finish_stage2(occ2, v_neus, f_np, info):
    """Shared by fresh builds and Drive-cache loads: checks, figures, snapshot, STAGE2_PRIOR."""
    global STAGE2_PRIOR, STAGE2_MESH_NEUS, STAGE2_INFO
    import json as _json
    occ2 = torch.as_tensor(occ2).float()
    iou = _occ_iou(occ2, OCC_SOLID_N11)
    in_ball = float((np.linalg.norm(v_neus, axis=1) < 1.0).mean())
    print(f"  Stage-2 occupancy @ {occ2.shape[0]}³: solid={float((occ2 > 0.5).float().mean()):.4f} | "
          f"IoU vs Stage-1 solid={iou:.3f} | verts in unit ball={in_ball:.3f}")
    STAGE2_MESH_NEUS = (v_neus, f_np)
    _glb = OUTPUT_DIR / "stage2_prior_mesh_neus.glb"
    trimesh.Trimesh(vertices=v_neus, faces=f_np, process=False).export(_glb)
    _mid = occ2.shape[0] // 2
    fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
    for ax, img, title in (
        (axs[0], OCC_SOLID_N11[_mid].numpy(), "Stage-1 solid (z mid)"),
        (axs[1], occ2[_mid].numpy(), "Stage-2 decoded solid (z mid)"),
        (axs[2], OCC_SOLID_N11[:, _mid].numpy() + 2 * occ2[:, _mid].numpy(), "overlay y mid (1=S1, 2=S2, 3=both)"),
    ):
        ax.imshow(img, origin="lower", cmap="magma"); ax.set_title(title, fontsize=9); ax.axis("off")
    fig.suptitle(f"Stage-2 prior vs Stage-1 | IoU={iou:.3f}")
    fig.tight_layout()
    _png = OUTPUT_DIR / "stage2_align_check.png"
    fig.savefig(_png, dpi=120, bbox_inches="tight"); plt.show()
    _png2 = plot_meshes([("MV-SAM3D Stage-2 (NeuS frame)", v_neus, f_np)], OUTPUT_DIR / "stage2_prior_preview.png")
    STAGE2_INFO = {**info, "iou_vs_stage1_solid": iou, "verts_in_unit_ball": in_ball, "RUN_TS": str(RUN_TS)}
    _snap = OUTPUT_DIR / "debug_snapshot_6_0a.json"
    _snap.write_text(_json.dumps(STAGE2_INFO, indent=2, default=str))
    print("=== DEBUG SNAPSHOT §6.0a (paste this block) ===")
    print(_json.dumps(STAGE2_INFO, indent=2, default=str))
    print("=== END SNAPSHOT ===")
    to_drive(_glb, _png, _png2, _snap)
    if iou < 0.1:
        raise RuntimeError(
            f"Stage-2 decoded geometry barely overlaps Stage-1 (IoU={iou:.3f}) — frame mismatch; "
            "not using it as a prior."
        )
    if iou < 0.3:
        print("  ⚠ low overlap with Stage-1 — inspect stage2_align_check.png before trusting this run")
    STAGE2_PRIOR = DensePriorTarget(occ2, source="stage2_mesh", meta={"iou_vs_stage1": iou, "lora": info.get("lora_used")})
    print(f"✓ STAGE2_PRIOR ready: {STAGE2_PRIOR.summary()}")
    return STAGE2_PRIOR


def load_stage2_cache(path):
    import json as _json
    z = np.load(path, allow_pickle=False)
    info = _json.loads(str(z["info"]))
    info["loaded_from_cache"] = str(path)
    print(f"✓ Stage-2 prior loaded from Drive cache {Path(path).name} — SAM3D stack not needed this run")
    return _finish_stage2(z["occ"].astype(np.float32), z["verts"].astype(np.float32), z["faces"].astype(np.int64), info)


def build_stage2_flow_prior(micro_n=None, teacher_steps=None, lora_dir=None, seed=0, cache_path=None):
    """Teacher SLAT (multi-view MultiDiffusion, Stage-1 coords) → SLAT mesh decoder → Sim3 → NeuS →
    solid occupancy → DensePriorTarget. Saves everything needed to skip this next time to cache_path."""
    import gc, json as _json
    micro_n = int(micro_n if micro_n is not None else STAGE2_MICRO_N)
    teacher_steps = int(teacher_steps if teacher_steps is not None else STAGE2_TEACHER_STEPS)
    lora_dir = lora_dir if lora_dir is not None else STAGE2_LORA_DIR
    if STAGE1_ALIGN is None or OCC_SOLID_N11 is None:
        raise RuntimeError("STAGE1_ALIGN missing — run §3.5 before §6.0a (needs baneus_scene_meta Sim3)")

    _mvsam_root, inf_py, pipe_yaml = _ensure_mvsam3d_stack()
    inference = _load_inference(inf_py, pipe_yaml)
    pipeline = inference._pipeline
    device_s2 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    slat_gen = pipeline.models["slat_generator"]
    cfg_mod = slat_gen.reverse_fn
    print("slat_generator:", type(slat_gen).__name__, "| reverse_fn:", type(cfg_mod).__name__)

    lora_used = False
    if lora_dir is not None and Path(str(lora_dir)).exists():
        try:
            ensure_peft_for_lora()
            from peft import PeftModel
            print("Loading Stage-2 LoRA from", lora_dir)
            cfg_mod.backbone = PeftModel.from_pretrained(cfg_mod.backbone, str(lora_dir))
            lora_used = True
        except Exception as e:
            print(f"⚠ LoRA load failed ({type(e).__name__}: {e}) — using base Stage-2 weights")
    elif lora_dir is not None:
        print(f"⚠ STAGE2_LORA_DIR missing ({lora_dir}) — using base Stage-2 weights")
    for p in slat_gen.parameters():
        p.requires_grad_(False)

    img_dir, mask_dir = AERIAL_DIR / IMG_SUBDIR, AERIAL_DIR / MASK_SUBDIR

    def _nat(p):
        s = Path(p).stem
        return (0, int(s), s) if s.isdigit() else (1, 0, s)

    paths = sorted(
        [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}], key=_nat
    )[:micro_n]
    assert paths, f"No images in {img_dir}"
    view_images, view_masks = [], []
    for ip in paths:
        view_images.append(Image.open(ip).convert("RGB"))
        mp = mask_dir / f"{ip.stem}.png"
        if not mp.exists():
            cands = list(mask_dir.glob(ip.stem + "*"))
            mp = cands[0] if cands else None
        view_masks.append(Image.open(mp) if mp is not None else None)
    print("Stage-2 teacher views:", [p.stem for p in paths])

    torch.manual_seed(seed)
    slat_dicts = _preprocess_slat_views(pipeline, view_images, view_masks)

    from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view
    from sam3d_objects.model.backbone.tdfy_dit.modules import sparse as sp

    if not (bool(USE_EXPORTED_STAGE1_COORDS) and STAGE1_COORDS is not None):
        raise RuntimeError("Stage-2 prior needs the exported Stage-1 coords (USE_EXPORTED_STAGE1_COORDS=True).")
    coords = STAGE1_COORDS.to(device=device_s2, dtype=torch.int32)
    if coords.shape[-1] == 3:
        coords = torch.cat([torch.zeros_like(coords[:, :1]), coords], dim=-1)
    print(f"Stage-1 coords from export (same voxels as the prefit): {tuple(coords.shape)}")

    print(f"Teacher Stage-2 ({len(slat_dicts)} views, MultiDiffusion, {teacher_steps} steps) → SLAT…")
    slat_gen.no_shortcut = True
    cfg_mod.strength = float(getattr(pipeline, "slat_cfg_strength", 0.0) or 0.0)
    prev_steps = slat_gen.inference_steps
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=pipeline.dtype, enabled=(device_s2.type == "cuda")):
            cond_args, cond_kwargs = pipeline.get_multi_view_condition_input(
                pipeline.condition_embedders["slat_condition_embedder"],
                slat_dicts,
                pipeline.slat_condition_input_mapping,
            )
            cond_args = tuple(cond_args) + (coords.detach().cpu().numpy().astype(np.int32, copy=False),)
            latent_shape = (slat_dicts[0]["image"].shape[0], coords.shape[0], 8)
            slat_gen.inference_steps = teacher_steps
            try:
                with inject_generator_multi_view(
                    slat_gen, num_views=len(slat_dicts), num_steps=teacher_steps, mode="multidiffusion",
                ):
                    teacher = slat_gen(latent_shape, device_s2, *cond_args, **cond_kwargs)
            finally:
                slat_gen.inference_steps = prev_steps
    x1 = (teacher[0] if isinstance(teacher, (tuple, list)) else teacher).detach().float()
    if x1.dim() == 3:
        x1 = x1[0]
    print(f"  teacher SLAT feats {tuple(x1.shape)}  |x| mean={float(x1.norm(dim=-1).mean()):.3f}")

    # Generator latents are normalised; the mesh decoder (FlexiCubes, 256³) expects slat·std + mean.
    slat = sp.SparseTensor(coords=coords, feats=x1).to(device_s2)
    slat = slat * pipeline.slat_std.to(device_s2) + pipeline.slat_mean.to(device_s2)
    mesh_out = pipeline.decode_slat(slat, formats=["mesh"])["mesh"][0]
    v_can = mesh_out.vertices.detach().float()
    f_can = mesh_out.faces.detach().long()
    if v_can.shape[0] == 0 or f_can.shape[0] == 0:
        raise RuntimeError("Stage-2 mesh decode returned an empty mesh")
    print(f"  decoded Stage-2 mesh: {v_can.shape[0]:,} verts, {f_can.shape[0]:,} faces | canonical bbox "
          f"{v_can.min(0).values.cpu().numpy().round(3)} … {v_can.max(0).values.cpu().numpy().round(3)}")

    # Same Sim3 chain as the Stage-1 prefit: canonical → M → world_align → NeuS.
    v_neus = canonical_to_neus_norm(v_can.to(device_s2)).cpu().numpy().astype(np.float32)
    f_np = f_can.cpu().numpy().astype(np.int64)
    occ2, shell2, solid2 = mesh_to_occupancy_n11(v_neus, f_np, res=PRIOR_GRID_RES)
    info = {
        "section": "6.0a", "prior_source": "stage2_mesh", "lora_used": lora_used,
        "lora_dir": str(lora_dir) if lora_used else None, "teacher_views": [p.stem for p in paths],
        "teacher_steps": teacher_steps, "n_coords": int(coords.shape[0]), "mesh_faces": int(f_np.shape[0]),
        "stage2_solid_frac": solid2, "stage2_shell_frac": shell2,
    }
    if cache_path is not None:
        try:
            Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(
                cache_path, occ=occ2.numpy().astype(np.float16), verts=v_neus, faces=f_np.astype(np.int32),
                slat_coords=coords.cpu().numpy().astype(np.int32), slat_feats=x1.cpu().numpy().astype(np.float32),
                verts_canonical=v_can.cpu().numpy().astype(np.float32), info=_json.dumps(info),
            )
            print("  cached on Drive →", cache_path, "(occupancy, mesh, raw teacher SLAT)")
        except Exception as e:
            print("⚠ Stage-2 cache write failed:", e)

    # Free the SAM3D stack: the target is cached, nothing is queried during training.
    del inference, pipeline, slat_gen, cfg_mod, slat, mesh_out, teacher, cond_args, cond_kwargs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return _finish_stage2(occ2, v_neus, f_np, info)


if USE_FLOW_PRIOR and str(FLOW_PRIOR_MODE).lower() in ("stage2", "slat", "mvsam3d"):
    _s2_cache = stage2_cache_path()
    try:
        if USE_CACHE and not REFRESH_CACHE and _s2_cache.exists():
            load_stage2_cache(_s2_cache)
        else:
            if USE_CACHE and not _s2_cache.exists():
                print(f"(no Stage-2 cache yet at {_s2_cache.name} — building; next run will reuse it)")
            build_stage2_flow_prior(cache_path=_s2_cache)
    except Exception as e:
        import traceback
        traceback.print_exc()
        STAGE2_PRIOR = None
        STAGE2_INFO = {"section": "6.0a", "prior_source": "FAILED", "error": f"{type(e).__name__}: {e}"}
        if STAGE2_FALLBACK_TO_STAGE1 and OCC_SOLID_N11 is not None:
            print("\n" + "!" * 88)
            print(f"✗ Stage-2 prior failed: {type(e).__name__}: {e}")
            print("  STAGE2_FALLBACK_TO_STAGE1=True → training continues with the Stage-1 occupancy prior.")
            print("  Your figures will say prior_source='stage1 (stage2 failed)'. Fix the stack and rerun for Stage-2.")
            print("!" * 88)
        else:
            raise
else:
    print(f"Skipping Stage-2 load (USE_FLOW_PRIOR={USE_FLOW_PRIOR}, MODE={FLOW_PRIOR_MODE})")

In [ ]:
# 6.0 — Build full model + optimizers + prior + visibility (+ Stage-1 prefit)
device = torch.device(DEVICE)

enc = HashGridEncoding(
    n_levels=HASH_N_LEVELS,
    n_features_per_level=HASH_N_FEATURES,
    log2_hashmap_size=HASH_LOG2_SIZE if HAS_TCNN else min(HASH_LOG2_SIZE, 18),
    base_resolution=HASH_BASE_RES,
    max_resolution=HASH_MAX_RES,
).to(device)
sdf_net = HashSDF(enc, hidden=SDF_HIDDEN, n_layers=SDF_N_LAYERS, geo_init_bias=0.5).to(device)
bg_net = BackgroundNeRF(hidden=BG_HIDDEN).to(device)
color_net = ColorNet(feat_dim=SDF_HIDDEN, hidden=SDF_HIDDEN).to(device)
poses = LearnablePoses(torch.from_numpy(scene["c2ws"]).float()).to(device)

MASKS_T = torch.from_numpy(np.stack(scene["masks"]).astype(np.float32)).to(device)
Ks_T = torch.from_numpy(np.asarray(scene["Ks"], np.float32)).to(device)
SAMPLER = RaySampler(scene["images"], scene["masks"], scene["Ks"], device)
VIS = VisibilityModel(
    Ks_T, MASKS_T, max_views=VIS_MAX_VIEWS, ds=VIS_ZBUF_DS, n_samples=VIS_ZBUF_SAMPLES,
    eps_abs=VIS_EPS_ABS, eps_rel=VIS_EPS_REL, depth_source=VIS_DEPTH_SOURCE,
)
if any(d is not None for d in scene.get("depths", [])):
    print("note: depth maps were loaded but are not used (their units are not the NeuS frame); "
          "occlusion comes from the Hash-SDF z-buffers.")

# ---- generative prior ----
_mode = (FLOW_PRIOR_MODE if USE_FLOW_PRIOR else "off").lower()
_s2 = globals().get("STAGE2_PRIOR")
if _mode in ("stage2", "slat", "mvsam3d") and _s2 is None:
    if STAGE2_FALLBACK_TO_STAGE1 and OCC_SOLID_N11 is not None:
        print("⚠⚠ Stage-2 prior unavailable → using the Stage-1 occupancy prior (label the report accordingly)")
        prior = DensePriorTarget(OCC_SOLID_N11, source="stage1 (stage2 failed)")
    else:
        raise RuntimeError("FLOW_PRIOR_MODE='stage2' but §6.0a did not produce STAGE2_PRIOR")
else:
    prior = build_flow_prior(_mode, occ_solid=OCC_SOLID_N11, stage2_prior=_s2)
prior = prior.to(device) if prior is not None else None
PRIOR_SOURCE = prior.source if prior is not None else "off"
USE_FLOW_PRIOR_EFF = prior is not None

opt_sdf = torch.optim.Adam([
    {"params": list(sdf_net.parameters()), "lr": LR_SDF, "base_lr": LR_SDF},
    {"params": list(color_net.parameters()) + list(bg_net.parameters()), "lr": LR_COLOR, "base_lr": LR_COLOR},
])
opt_pose = torch.optim.Adam(poses.parameters(), lr=LR_POSE)

print(
    f"params SDF+color+bg: "
    f"{sum(p.numel() for p in sdf_net.parameters())+sum(p.numel() for p in color_net.parameters())+sum(p.numel() for p in bg_net.parameters()):,}"
)
print(f"params poses: {sum(p.numel() for p in poses.parameters()):,}")
print(f"hash backend: {enc.backend} | prior: {PRIOR_SOURCE} | on_visible={FLOW_PRIOR_ON_VISIBLE}")
print(f"Stage-1: {HAS_STAGE1} | hard_mask_route={HARD_MASK_ROUTE} | visibility views={VIS.view_idx.tolist()}")

# ---- resume (Drive checkpoint) ----
def _find_ckpt(spec):
    if spec in (None, "", False):
        return None
    if str(spec) == "latest":
        cands = sorted(GFHASHBA_DRIVE.glob(f"*/{CASE_NAME}/gfhashba_ckpt.pth"), key=lambda q: q.stat().st_mtime)
        return cands[-1] if cands else None
    q = Path(str(spec))
    return q if q.exists() else None


RESUME_STATE = None
if RESUME_FROM:
    _ck_path = _find_ckpt(RESUME_FROM)
    if _ck_path is None:
        raise FileNotFoundError(f"RESUME_FROM={RESUME_FROM!r}: no checkpoint found under {GFHASHBA_DRIVE}")
    _st = torch.load(_ck_path, map_location=device, weights_only=False)
    sdf_net.load_state_dict(_st["sdf"]); bg_net.load_state_dict(_st["bg"])
    color_net.load_state_dict(_st["color"]); poses.load_state_dict(_st["poses"])
    if "opt_sdf" in _st:
        opt_sdf.load_state_dict(_st["opt_sdf"]); opt_pose.load_state_dict(_st["opt_pose"])
    RESUME_STATE = {"iter": int(_st["iter"]), "history": _st.get("history", {}), "path": str(_ck_path)}
    print(f"✓ resumed from {_ck_path} at iter {RESUME_STATE['iter']} (prior then: {_st['config'].get('PRIOR_SOURCE')})")

# ---- Stage-1 prefit (cached on Drive; skipped when resuming) ----
PREFIT_CACHE = CACHE_DIR / ("prefit_" + cache_key(
    "prefit_v2", SAM_BANEU_RUN, CASE_NAME, PREFIT_SOLIDIFY, PREFIT_GAMMA, SDF_PREFIT_ITERS, SDF_PREFIT_BATCH,
    SDF_PREFIT_LR, PRIOR_GRID_RES, HASH_N_LEVELS, HASH_N_FEATURES, enc.log2_hashmap_size, HASH_BASE_RES,
    HASH_MAX_RES, SDF_HIDDEN, SDF_N_LAYERS, enc.backend,
) + ".pth")
if RESUME_STATE is not None:
    print("skipping prefit (resumed)")
    if PREFIT_CACHE.exists():   # still rebuild PREFIT_MESH for the §7 comparison figure
        _tmp = HashSDF(HashGridEncoding(HASH_N_LEVELS, HASH_N_FEATURES, enc.log2_hashmap_size, HASH_BASE_RES,
                                        HASH_MAX_RES), hidden=SDF_HIDDEN, n_layers=SDF_N_LAYERS).to(device)
        _tmp.load_state_dict(torch.load(PREFIT_CACHE, map_location=device, weights_only=False)["sdf"])
        try:
            PREFIT_MESH = mc_mesh(sdf_volume(_tmp, PREVIEW_RES, float(PREFIT_GAMMA)), 0.0)
        except Exception as e:
            print("  prefit mesh for figures unavailable:", e)
        del _tmp
elif USE_STAGE1_PREFIT and HAS_STAGE1:
    print("=== Stage-1 Hash-SDF prefit ===")
    prefit_hash_sdf(sdf_net, n_iters=SDF_PREFIT_ITERS, batch=SDF_PREFIT_BATCH, lr=SDF_PREFIT_LR,
                    cache_path=PREFIT_CACHE)
else:
    print("skipping Stage-1 prefit")

### 6.0c — Prior schedule (after the Stage-1 prefit)

**Prefit:** Stage-1 occupancy only (no Stage-2).

**Idea:** photo alone is an ill-behaved surface early — let the Stage-2 prior lead on blind spots
first, then hand off so photo owns fidelity near the minimum.

**Weight (hold → cosine):** hold `W_FLOW_START` for the first `FLOW_HOLD_FRAC` of training, then cosine
to `W_FLOW_END`. Applied **every iteration**, spatially gated by $1-\mathrm{sg}(V)$ inside the loss — so
visible geometry is never pulled by the prior unless `FLOW_PRIOR_ON_VISIBLE=True`.

In [ ]:
_sched_png = plot_flow_prior_schedule(OUTPUT_DIR / "flow_prior_schedule.png")
to_drive(_sched_png)
for _it in (0, int(FLOW_HOLD_FRAC * FLOW_ANNEAL_ITERS), FLOW_ANNEAL_ITERS // 2, END_ITER - 1):
    print(f"  it={_it:5d}  W_prior={flow_prior_schedule(_it)[0]:.3f}  (× (1 − V) per point)")

In [ ]:
# 6.1 — Training loop: photo + mask (carving) + eikonal on rays; Stage-2 prior on blind spots
from collections import defaultdict
import time
import math
import json
import torch.nn.functional as F

history = defaultdict(list, {k: list(v) for k, v in (RESUME_STATE or {}).get("history", {}).items()})
START_IT = (RESUME_STATE["iter"] + 1) if RESUME_STATE else 0
LOG_EVERY = 100
HIST_EVERY = 10
TRAIN_LOG = OUTPUT_DIR / "train_log.txt"
t0 = time.time()
sdf_net.train(); bg_net.train(); color_net.train(); poses.train()
vis_grid = None
_vis_info = {}


def _vis_refresh(it, gamma):
    """Re-render the visibility z-buffers from the current SDF and re-bake V on a grid."""
    global vis_grid, _vis_info
    VIS.refresh(sdf_net, poses().detach(), gamma)
    vis_grid = bake_vis_grid(VIS, VIS_GRID_RES)
    with torch.no_grad():
        _pp = sample_ball(32768, device)
        _inside = sdf_net(_pp, gamma=gamma)[0].squeeze(-1) < 0
        _v = grid_sample_n11(vis_grid, _pp)
        _vis_info = {
            "it": it,
            "unseen_frac_ball": float((_v < 0.5).float().mean()),
            "unseen_frac_inside": float((_v[_inside] < 0.5).float().mean()) if bool(_inside.any()) else float("nan"),
            "zbuf_hit_frac": float(torch.isfinite(VIS.zbuf).float().mean()) if VIS.zbuf is not None else float("nan"),
        }
    history["vis_it"].append(it)
    history["unseen_inside"].append(_vis_info["unseen_frac_inside"])
    print(f"  [vis @ {it}] z-buffer hit={_vis_info['zbuf_hit_frac']:.3f} | unseen: ball={_vis_info['unseen_frac_ball']:.3f} "
          f"inside-surface={_vis_info['unseen_frac_inside']:.3f}")


def _log(line):
    print(line)
    with open(TRAIN_LOG, "a") as fh:
        fh.write(line + "\n")


def _save_ckpt(it):
    """Resumable checkpoint (model + poses + optimizers + history) → Drive. Set RESUME_FROM to continue."""
    ck = {
        "sdf": sdf_net.state_dict(), "bg": bg_net.state_dict(), "color": color_net.state_dict(),
        "poses": poses.state_dict(), "opt_sdf": opt_sdf.state_dict(), "opt_pose": opt_pose.state_dict(),
        "iter": it, "history": dict(history), "run_ts": str(RUN_TS),
        "config": {
            "CASE_NAME": CASE_NAME, "END_ITER": END_ITER, "RUN_PRESET": RUN_PRESET,
            "HASH_N_LEVELS": HASH_N_LEVELS, "ANNEAL_ITERS": ANNEAL_ITERS,
            "W_FLOW_START": W_FLOW_START, "W_FLOW_END": W_FLOW_END, "FLOW_HOLD_FRAC": FLOW_HOLD_FRAC,
            "PRIOR_SOURCE": PRIOR_SOURCE, "FLOW_PRIOR_ON_VISIBLE": FLOW_PRIOR_ON_VISIBLE,
            "VIS_DEPTH_SOURCE": VIS_DEPTH_SOURCE, "USE_STAGE1_PREFIT": USE_STAGE1_PREFIT,
        },
    }
    path = OUTPUT_DIR / "gfhashba_ckpt.pth"
    torch.save(ck, path)
    to_drive(path, TRAIN_LOG)
    return path


def _snapshot(it, final=False):
    """Progress mesh preview (report figure) → Drive."""
    png = None
    try:
        sdf_net.eval()
        vol = sdf_volume(sdf_net, PREVIEW_RES, anneal_gamma(it, HASH_N_LEVELS, ANNEAL_ITERS, ANNEAL_START_LVL))
        sdf_net.train()
        if vol.min() < 0 < vol.max():
            v, f = mc_mesh(vol, 0.0)
            png = plot_meshes([(f"iter {it}", v, f)], OUTPUT_DIR / f"progress_it{it:06d}.png", show=final)
            history["progress_png"].append(str(png))
    except Exception as e:
        sdf_net.train()
        print("  snapshot mesh skipped:", e)
    to_drive(png)


_log(f"MAIN TRAIN: iters {START_IT}→{END_ITER} | prior={PRIOR_SOURCE} (on_visible={FLOW_PRIOR_ON_VISIBLE}) | "
     f"visibility={VIS_DEPTH_SOURCE} | γ {ANNEAL_START_LVL}→{HASH_N_LEVELS} | Stage-1 is used ONLY in the prefit")
if START_IT >= END_ITER:
    print("nothing to train — checkpoint already at END_ITER; continue with §6.2 / §7")

for it in range(START_IT, END_ITER):
    gamma = anneal_gamma(it, HASH_N_LEVELS, ANNEAL_ITERS, ANNEAL_START_LVL)
    w_prior, _ = flow_prior_schedule(it)
    sdf_net.beta_floor = beta_floor_schedule(it)

    if vis_grid is None or it % VIS_REFRESH_EVERY == 0:
        _vis_refresh(it, gamma)
        if it == START_IT:
            to_drive(TRAIN_LOG)

    c2ws = poses()
    rays_o, rays_d, rgb_gt, msk, vi = SAMPLER.sample(c2ws, BATCH_RAYS)
    out = render_dual_field(
        rays_o, rays_d, sdf_net, bg_net, color_net,
        gamma=gamma, n_samples=N_SAMPLES, n_bg=N_BG_SAMPLES,
        mask_rays=msk, hard_mask_route=HARD_MASK_ROUTE,
    )

    err = (out["rgb"] - rgb_gt) ** 2
    w_pix = 0.05 + 0.95 * msk
    L_photo = (err * w_pix[:, None]).sum() / (w_pix.sum() * 3 + 1e-6)
    L_mask = F.binary_cross_entropy(out["O_raw"].clamp(1e-4, 1 - 1e-4), msk.clamp(0, 1))
    L_eik = out["eikonal"]

    L_prior = torch.zeros((), device=device)
    pstats = {}
    if prior is not None and w_prior > 0:
        _pts = sample_ball(PRIOR_N_PTS, device)
        L_prior, pstats = prior_distillation_loss(
            sdf_net, prior, _pts, gamma=gamma, vis=grid_sample_n11(vis_grid, _pts),
            tau=PRIOR_TAU, on_visible=FLOW_PRIOR_ON_VISIBLE,
        )

    loss = W_PHOTO * L_photo + W_MASK * L_mask + W_EIKONAL * L_eik + w_prior * L_prior

    opt_sdf.zero_grad(set_to_none=True)
    opt_pose.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(sdf_net.parameters()) + list(color_net.parameters()) + list(bg_net.parameters()), 5.0
    )
    opt_sdf.step()
    if it >= POSE_START_ITER:
        opt_pose.step()

    if it < WARMUP_END:
        sc = 0.1 + 0.9 * it / WARMUP_END
    else:
        sc = 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (it - WARMUP_END) / max(1, END_ITER - WARMUP_END)))
    for pg in opt_sdf.param_groups:
        pg["lr"] = pg["base_lr"] * sc
    for pg in opt_pose.param_groups:
        pg["lr"] = LR_POSE * sc

    if it % HIST_EVERY == 0 or it == END_ITER - 1:
        history["it"].append(it)
        history["loss"].append(float(loss))
        history["photo"].append(float(L_photo))
        history["mask"].append(float(L_mask))
        history["eik"].append(float(L_eik))
        history["prior"].append(float(L_prior))
        history["prior_agree"].append(pstats.get("agree", float("nan")))
        history["gamma"].append(gamma)
        history["w_prior"].append(float(w_prior))
        history["beta"].append(float(sdf_net.beta))
        history["pose_dt"].append(float(poses.t.detach().norm(dim=-1).mean()))
        history["pose_dr_deg"].append(float(poses.r.detach().norm(dim=-1).mean() * 180.0 / math.pi))

    if it % LOG_EVERY == 0 or it == END_ITER - 1:
        with torch.no_grad():
            _surf, _hit = first_crossing_points(rays_o.detach(), rays_d.detach(), out["sdf"].detach(), out["z"].detach())
            v_surf = float(VIS.visibility(_surf[_hit]).mean()) if bool(_hit.any()) else float("nan")
        history["log_it"].append(it)
        history["V_surf"].append(v_surf)
        dt = time.time() - t0
        eta = dt / max(1, it - START_IT + 1) * (END_ITER - it - 1)
        _log(
            f"iter {it:6d}/{END_ITER}  loss={float(loss):.4f}  photo={float(L_photo):.4f}  "
            f"mask={float(L_mask):.4f}  eik={float(L_eik):.4f}  prior={float(L_prior):.4f} "
            f"(agree={pstats.get('agree', float('nan')):.3f} unseen∩tgt={pstats.get('unseen_in_target', float('nan')):.2f})  "
            f"Wp={w_prior:.3f}  V_surf={v_surf:.3f}  β={float(sdf_net.beta):.1f}  γ={gamma:.2f}  "
            f"[{dt:.0f}s, eta {eta / 60:.0f} min]"
        )

    if (it > START_IT and it % CKPT_EVERY == 0) or it == END_ITER - 1:
        _save_ckpt(it)
    if (it > START_IT and it % SNAPSHOT_EVERY == 0) or it == END_ITER - 1:
        _snapshot(it, final=(it == END_ITER - 1))

_log(f"✓ training done in {(time.time() - t0) / 60:.1f} min (checkpoint + log on Drive: {DRIVE_OUT})")
to_drive(TRAIN_LOG)

In [ ]:
# 6.2 — Loss curves (report figure)
import matplotlib.pyplot as plt
import numpy as np

_its = np.asarray(history.get("it", []))
panels = [
    ("loss", "total loss", True), ("photo", "photometric (MSE)", True), ("mask", "mask BCE (unrouted)", True),
    ("eik", "eikonal", True), ("prior", "Stage-2 prior BCE × (1−V)", True), ("prior_agree", "prior sign agreement", False),
    ("w_prior", "prior weight W", False), ("gamma", "hash levels γ", False), ("beta", "SDF sharpness β", False),
    ("pose_dr_deg", "mean |Δrotation| (deg)", False),
]
fig, axes = plt.subplots(2, 5, figsize=(19, 7))
for ax, (k, title, logy) in zip(axes.ravel(), panels):
    y = np.asarray(history.get(k, []), dtype=np.float64)
    if y.size:
        ax.plot(_its[: y.size], y, lw=1)
        if logy and np.all(y[np.isfinite(y)] > 0):
            ax.set_yscale("log")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
plt.suptitle(f"GF-HashBA training | prior={PRIOR_SOURCE} | visibility={VIS_DEPTH_SOURCE}")
plt.tight_layout()
_curves = OUTPUT_DIR / "train_curves.png"
fig.savefig(_curves, dpi=120, bbox_inches="tight")
plt.show()

if history.get("V_surf"):
    fig2, ax2 = plt.subplots(1, 2, figsize=(11, 3.5))
    ax2[0].plot(history["log_it"], history["V_surf"], lw=1.5)
    ax2[0].set_title("V at rendered surface points"); ax2[0].set_ylim(0, 1); ax2[0].grid(alpha=0.3)
    ax2[1].plot(history["vis_it"], history["unseen_inside"], "o-", lw=1.5)
    ax2[1].set_title("fraction of the solid that is unobserved (prior-owned)"); ax2[1].set_ylim(0, 1)
    ax2[1].grid(alpha=0.3)
    plt.tight_layout()
    fig2.savefig(OUTPUT_DIR / "visibility_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
to_drive(_curves, OUTPUT_DIR / "visibility_curves.png")
print("γ final:", history["gamma"][-1] if history.get("gamma") else None,
      "| W_prior final:", history["w_prior"][-1] if history.get("w_prior") else None)

## 7 — Extract mesh (marching cubes on Hash-SDF) + comparison figure

Evaluate the SDF on a 256³ grid → marching cubes → health gates → cleaned mesh (components < 1 % of faces dropped).

**Health gates:** if the SDF range is almost all-positive (e.g. `[-0.06, 0.33]`) or the mesh
has dozens of connected components you get floating slabs — **stop and inspect**, do not trust the GLB.

`mesh_comparison.png` puts the Stage-1 prefit, the MV-SAM3D Stage-2 prior and the GF-HashBA result side by side in the same NeuS frame and viewpoints.

In [ ]:
# 7.0 — Marching cubes export + mesh health gates + comparison figure
import sys, json
from pathlib import Path
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt

for _r in (Path("/content"), Path.cwd(), Path("/Users/taiganishida/Documents/GSV-MSV")):
    if (_r / "gfhashba_align.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))
try:
    from gfhashba_align import mesh_health, occ_bbox_from_grid, sdf_volume_health
    _HAS_ALIGN_LIB = True
except Exception as _e:
    print("⚠ gfhashba_align missing — inline health fallbacks:", _e)
    _HAS_ALIGN_LIB = False

RES = int(MESH_RES)
sdf_net.eval()
vol = sdf_volume(sdf_net, RES, float(HASH_N_LEVELS))

# ---- SDF health (catches the floating-slab mode) ----
if _HAS_ALIGN_LIB:
    _sh = sdf_volume_health(vol, 0.0)
else:
    _fi = float((vol < 0).mean())
    _sh = {
        "vmin": float(vol.min()), "vmax": float(vol.max()), "crosses_zero": bool(vol.min() < 0 < vol.max()),
        "frac_inside": _fi, "ok_sign_balance": (0.01 <= _fi <= 0.85) and (vol.min() < 0 < vol.max()),
    }
print(
    f"SDF health: range=[{_sh['vmin']:.3f},{_sh['vmax']:.3f}]  "
    f"frac_inside={_sh['frac_inside']:.4f}  crosses0={_sh['crosses_zero']}  ok_sign={_sh['ok_sign_balance']}"
)

_mid = RES // 2
fig, axs = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, sl, t in (
    (axs[0], vol[_mid], "SDF z mid"),
    (axs[1], vol[:, _mid, :], "SDF y mid"),
    (axs[2], (vol < 0).astype(np.float32)[_mid], "inside (sdf<0) z mid"),
):
    ax.imshow(sl, origin="lower", cmap="coolwarm" if "SDF" in t else "gray")
    ax.set_title(t); ax.axis("off")
fig.tight_layout()
_sdf_png = OUTPUT_DIR / "sdf_slices_pre_mc.png"
fig.savefig(_sdf_png, dpi=120, bbox_inches="tight")
plt.show()

level = 0.0
if not _sh["ok_sign_balance"]:
    print("⚠ SDF sign balance BAD — level-0 marching cubes will produce floating debris. "
          "Inspect sdf_slices_pre_mc.png; trying an adaptive level as a last resort.")
    level = float(np.median(vol[vol < 0])) if (vol < 0).any() else float(np.percentile(vol, 20))
    print(f"  adaptive MC level={level:.4f}")
try:
    verts, faces = mc_mesh(vol, level)
except Exception as e:
    print("MC failed:", e, "— retry at the median")
    level = float(np.median(vol))
    verts, faces = mc_mesh(vol, level)
print(f"mesh: {len(verts):,} verts, {len(faces):,} faces | MC level={level:.4f}")

# ---- mesh health (verts are xyz now, like the occupancy bbox) ----
_occ_bb = None
if OCC_GRID_N11 is not None and _HAS_ALIGN_LIB:
    _occ_bb = occ_bbox_from_grid(OCC_GRID_N11.detach().float().cpu().numpy(), 0.5)
if _HAS_ALIGN_LIB:
    _mh = mesh_health(verts, faces, occ_bbox=_occ_bb)
else:
    _mh = {"n_verts": len(verts), "n_faces": len(faces), "n_components": -1,
           "largest_frac": float("nan"), "ok": len(faces) >= 500, "reason": "no align lib"}
print(
    f"Mesh health: comps={_mh.get('n_components')}  largest_frac={_mh.get('largest_frac', float('nan')):.3f}  "
    f"bbox_iou_vs_occ={_mh.get('bbox_iou_vs_occ', float('nan'))}  ok={_mh.get('ok')}  ({_mh.get('reason')})"
)
if not _mh.get("ok", False):
    print("⚠ MESH HEALTH FAIL — floating debris / empty / misaligned. Do not trust this GLB for A/B.")

verts_c, faces_c, _n_comp = keep_large_components(verts, faces, min_frac=0.01)
print(f"cleaned mesh: {len(faces_c):,} faces (dropped components < 1% of faces; {_n_comp} before)")

# NeuS normalized → world via the cameras_sphere scale_mat (x_world = S @ x_norm)
scale_mat = np.asarray(scene.get("scale_mat", np.eye(4)), np.float64)
verts_w = (scale_mat @ np.concatenate([verts_c, np.ones((len(verts_c), 1))], 1).T).T[:, :3]

out_norm = OUTPUT_DIR / "gfhashba_mesh_norm.glb"
out_clean = OUTPUT_DIR / "gfhashba_mesh_clean_norm.glb"
out_world = OUTPUT_DIR / "gfhashba_mesh_world.glb"
out_ply = OUTPUT_DIR / "gfhashba_mesh_norm.ply"
trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(out_norm)
trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(out_ply)
trimesh.Trimesh(vertices=verts_c, faces=faces_c, process=False).export(out_clean)
trimesh.Trimesh(vertices=verts_w, faces=faces_c, process=False).export(out_world)
print("saved", out_norm.name, out_clean.name, out_world.name)

# ---- report figure: Stage-1 prefit vs MV-SAM3D Stage-2 prior vs GF-HashBA (same NeuS frame) ----
_rows = []
if globals().get("PREFIT_MESH") is not None:
    _rows.append(("Stage-1 prefit", *PREFIT_MESH))
if globals().get("STAGE2_MESH_NEUS") is not None:
    _rows.append(("MV-SAM3D Stage-2 prior", *STAGE2_MESH_NEUS))
_rows.append((f"GF-HashBA final ({PRIOR_SOURCE})", verts_c, faces_c))
_cmp_png = plot_meshes(_rows, OUTPUT_DIR / "mesh_comparison.png")
plot_meshes([("GF-HashBA", verts_c, faces_c)], OUTPUT_DIR / "mesh_preview.png", show=False)

_snap7 = {
    "section": "7",
    "prior_source": PRIOR_SOURCE,
    "sdf_vmin": float(_sh["vmin"]), "sdf_vmax": float(_sh["vmax"]),
    "frac_inside": float(_sh["frac_inside"]), "ok_sign_balance": bool(_sh["ok_sign_balance"]),
    "mc_level": float(level),
    "n_verts": int(len(verts)), "n_faces": int(len(faces)), "n_faces_clean": int(len(faces_c)),
    "n_components": _mh.get("n_components"), "largest_frac": _mh.get("largest_frac"),
    "bbox_iou_vs_occ": _mh.get("bbox_iou_vs_occ"),
    "mesh_ok": bool(_mh.get("ok", False)), "mesh_reason": str(_mh.get("reason")),
    "FLOW_PRIOR_ON_VISIBLE": bool(FLOW_PRIOR_ON_VISIBLE), "VIS_DEPTH_SOURCE": VIS_DEPTH_SOURCE,
    "unseen_inside_last": (history.get("unseen_inside") or [None])[-1],
    "RUN_TS": str(RUN_TS), "DRIVE_OUT": str(DRIVE_OUT),
}
(OUTPUT_DIR / "debug_snapshot_7.json").write_text(json.dumps(_snap7, indent=2, default=float))
print("=== DEBUG SNAPSHOT §7 (paste this block) ===")
print(json.dumps(_snap7, indent=2, default=float))
print("=== END SNAPSHOT ===")

to_drive(out_norm, out_clean, out_world, out_ply, _sdf_png, _cmp_png, OUTPUT_DIR / "mesh_preview.png",
         OUTPUT_DIR / "train_curves.png", OUTPUT_DIR / "flow_prior_schedule.png",
         OUTPUT_DIR / "align_diag_s1.png", OUTPUT_DIR / "debug_snapshot_7.json",
         OUTPUT_DIR / "debug_snapshot_3_5b.json", OUTPUT_DIR / "debug_snapshot_6_0a.json")
print("→ Drive", DRIVE_OUT)

### 7.1 — Rendered views vs inputs (report figure)

Renders a few training views with the learned poses (½ resolution): input | render (PSNR, and PSNR inside the mask) | foreground opacity | foreground depth. Writes `run_summary.json` (prior source, Stage-2 diagnostics, mesh health, PSNR) next to the figures on Drive.

The same `render_image(...)` helper renders held-out cameras for DreamSim once those are available.

In [ ]:
# 7.1 — Rendered views vs input images (report figure) + run summary
import json
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RENDER_DS = 2
sdf_net.beta_floor = beta_floor_schedule(END_ITER)   # same sharpness as the end of training
_c2ws_final = poses().detach()
_n_show = min(3, scene["n_views"])
_views = sorted(set(np.linspace(0, scene["n_views"] - 1, _n_show).round().astype(int).tolist()))
_psnr = {}
fig, axs = plt.subplots(len(_views), 4, figsize=(15, 3.4 * len(_views)), squeeze=False)
sdf_net.eval(); bg_net.eval(); color_net.eval()
for r, v in enumerate(_views):
    rend = render_image(v, _c2ws_final, scene["Ks"], scene["H"], scene["W"], sdf_net, bg_net, color_net,
                        gamma=float(HASH_N_LEVELS), ds=RENDER_DS, n_samples=N_SAMPLES, n_bg=N_BG_SAMPLES)
    gt = np.asarray(scene["images"][v])[rend["ys"]][:, rend["xs"]]
    m = np.asarray(scene["masks"][v])[rend["ys"]][:, rend["xs"]] > 0.5
    mse = float(((rend["rgb"] - gt) ** 2).mean())
    mse_m = float(((rend["rgb"] - gt) ** 2)[m].mean()) if m.any() else float("nan")
    _psnr[int(v)] = {"psnr": -10 * np.log10(max(mse, 1e-10)), "psnr_mask": -10 * np.log10(max(mse_m, 1e-10))}
    d = rend["depth"].copy(); d[rend["opacity"] < 0.5] = np.nan
    for c, (img, title) in enumerate((
        (gt, f"input view {v}"),
        (rend["rgb"], f"GF-HashBA render | PSNR {_psnr[v]['psnr']:.1f} (mask {_psnr[v]['psnr_mask']:.1f})"),
        (rend["opacity"], "foreground opacity"),
        (d, "foreground depth"),
    )):
        axs[r, c].imshow(img, cmap=None if c < 2 else ("gray" if c == 2 else "viridis"))
        axs[r, c].set_title(title, fontsize=9)
        axs[r, c].axis("off")
plt.tight_layout()
_rv_png = OUTPUT_DIR / "render_vs_input.png"
fig.savefig(_rv_png, dpi=120, bbox_inches="tight")
plt.show()

RUN_SUMMARY = {
    "run_ts": str(RUN_TS), "case": CASE_NAME, "preset": RUN_PRESET, "end_iter": END_ITER,
    "prior_source": PRIOR_SOURCE, "flow_prior_on_visible": bool(FLOW_PRIOR_ON_VISIBLE),
    "visibility": VIS_DEPTH_SOURCE, "sam_baneus_run": str(SAM_BANEU_RUN),
    "stage2": globals().get("STAGE2_INFO", {}),
    "psnr_train_views": _psnr,
    "mesh": globals().get("_snap7", {}),
    "train_minutes": round((time.time() - t0) / 60, 1) if "t0" in globals() else None,
    "figures": [
        "gfhashba_prefit_preview.png", "stage2_prior_preview.png", "stage2_align_check.png",
        "mesh_comparison.png", "render_vs_input.png", "train_curves.png", "visibility_curves.png",
        "flow_prior_schedule.png", "align_diag_s1.png",
    ] + [Path(p).name for p in history.get("progress_png", [])],
}
_sum_path = OUTPUT_DIR / "run_summary.json"
_sum_path.write_text(json.dumps(RUN_SUMMARY, indent=2, default=float))
print(json.dumps({k: RUN_SUMMARY[k] for k in ("prior_source", "visibility", "psnr_train_views", "train_minutes")},
                 indent=2, default=float))
to_drive(_rv_png, _sum_path)
print("→ Drive", DRIVE_OUT)

## 8 — Stage-2 prior notes

**Default:** `FLOW_PRIOR_MODE="stage2"` — §6.0a decodes the MV-SAM3D Stage-2 teacher SLAT into a mesh and maps it with the same Sim3 as the Stage-1 prefit. Check `stage2_align_check.png` and `stage2_prior_preview.png`.

Optional LoRA: set `STAGE2_LORA_DIR` to a saved `lora_adapter/` from `mvsam3d_stage2LoRA.ipynb` (load failures fall back to base weights).

**Visible toggle:** `FLOW_PRIOR_ON_VISIBLE=False` → prior only on blind spots `(1−V)`; `True` → also on visible regions (A/B ablation).

Other modes:
- `"occupancy"` — solidified Stage-1 teacher (ablation: Stage-1 vs Stage-2 prior)
- `"off"` — no generative prior (pure photometric + mask baseline)

## 9 — Relation to the BA-NeuS notebooks

| | `sam_baneus_ultrra.ipynb` | **GF-HashBA (this)** |
|--|--|--|
| Geometry | MLP NeuS SDF | **Hash-SDF** + bg NeRF |
| Topology prior | Stage-1 occupancy MSE | Stage-1 **prefit only**; **Stage-2 decoded occupancy** on unseen regions |
| Visibility | (none / prior everywhere) | **Occlusion-aware V(x)** (z-buffers from the current SDF) gates photo ↔ prior |
| Early BA stability | MLP spectral bias | **Hash annealing** $W_l(\gamma)$ |

**Need from sam_baneus:** `cameras_sphere.npz`, `baneus_scene_meta.npz`, `stage1/stage1_occupancy.npz`.